In [1]:
print("A_infinity algebra")

A_infinity algebra


My first A-infinity computation
We define a basis $x,y,z$ and then compute $m_2(x,y)$.

In [2]:
from IPython.display import display, Latex

display(Latex(r"$m_2(m_2(a,b),c) - m_2(a,m_2(b,c)) = 0$"))

<IPython.core.display.Latex object>

In [3]:
import sympy as sp

In [4]:
from sympy import Symbol, simplify

In [5]:
import keyword
import builtins

def safe_make_global(name, value):
    if not name.isidentifier():
        print(f"Did not create variable {name}: not a valid Python variable name.")
        return

    if keyword.iskeyword(name):
        print(f"Did not create variable {name}: it is a Python keyword.")
        return

    if name in dir(builtins):
        print(f"Did not create variable {name}: it is a built-in Python name.")
        return

    globals()[name] = value
    print(f"Created global variable {name}.")

In [6]:
class CellFamily:
    def __init__(self, quiver):
        self.quiver = quiver
        self.cells = {}

    def key(self, f):
        if hasattr(f, "key"):
            return f.key()

        return repr(f)

    def __getitem__(self, f):
        k = self.key(f)

        if k not in self.cells and repr(f) in self.cells:
            return self.cells[repr(f)]

        if k not in self.cells:
            print("No cell has been attached to this differential yet.")
            print("Differential:", f)
            return None

        return self.cells[k]

    def __setitem__(self, f, cell):
        if not isinstance(cell, Arrow):
            raise TypeError("Only Arrow objects can be stored as cells.")

        k = self.key(f)
        self.cells[k] = cell

    def __contains__(self, f):
        return self.key(f) in self.cells or repr(f) in self.cells

    def __repr__(self):
        return "CellFamily"

In [7]:
class Quiver:
    def __init__(self):
        self.vertices = set()
        self.arrows = {}
        self.idempotents = {}
        self.differential = {}

        self.homotopy_registry = {}
        self.pending_homotopies = {}

        self.virtual_homotopies = {}
        self.virtual_values = {}
        
        self.u = CellFamily(self)
        globals()["u"] = self.u

        self.v = CellFamily(self)
        globals()["v"] = self.v

        self.cells = {}


        self.resolved_monomials = {}
        self.massey_products = {}
        self.resolved_massey_products = {}
        self.resolved_mp_expressions = {}
        self.bridge_history = []
        self.bridge_expressions = {}
        self.bridge_cells = {}
        self.attachment_history = []
        self.mp_primitive_candidates = {}
        self.bridge_interval_records = []
        self.bridge_interval_equalities = []

    def zero(self):
        """
        The zero element.
        """
        return Element({})

    def add_vertex(self, v):
        """
        Add a vertex v and automatically create its idempotent e_v.
        Also creates the global variable e_v.
        """
        if v in self.vertices:
            print(f"Vertex {v} already exists.")
            return

        self.vertices.add(v)

        # Create idempotent e_v
        e = Path(self, [], v, v)
        self.idempotents[v] = e

        # Idempotents have zero differential
        self.differential[f"e_{v}"] = self.zero()

        # Create global variable e_v
        safe_make_global(f"e_{v}", e)
        
    def add_vertices(self):
        while True:
            v = input("Enter a vertex name, or type ok to finish: ")

            if v == "ok":
                break

            self.add_vertex(v)

    def delete_vertices(self):
        while True:
            v = input("Enter a vertex name to delete, or type ok to finish: ")

            if v == "ok":
                break

            if v not in self.vertices:
                print(f"Vertex {v} does not exist.")
                continue

            # Find all arrows touching this vertex
            arrows_to_delete = []

            for name, arrow in self.arrows.items():
                if arrow.source == v or arrow.target == v:
                    arrows_to_delete.append(name)

            # Delete those arrows
            for name in arrows_to_delete:
                arrow = self.arrows[name]

                # Delete its differential entry if it exists
                if arrow in self.differential:
                    del self.differential[arrow]

                # Delete from the arrow dictionary
                del self.arrows[name]

                # Delete global arrow variable if it exists
                if name in globals():
                    del globals()[name]

            # If some deleted arrows were attached cells u[f],
            # remove them from the CellFamily as well
            keys_to_delete = []

            for key, cell in self.u.cells.items():
                if cell.name in arrows_to_delete:
                    keys_to_delete.append(key)

            for key in keys_to_delete:
                del self.u.cells[key]

            # Delete the idempotent e_v from the differential dictionary if needed
            if v in self.idempotents:
                e_v = self.idempotents[v]

                if e_v in self.differential:
                    del self.differential[e_v]

                del self.idempotents[v]

            # Delete global idempotent variable e_v if it exists
            idempotent_name = f"e_{v}"

            if idempotent_name in globals():
                del globals()[idempotent_name]

            # Delete the vertex itself
            self.vertices.remove(v)

            print(f"Deleted vertex {v}.")
            print(f"Also deleted arrows: {arrows_to_delete}")

    def make_idempotents(self):
        """
        Create idempotents e_v for every vertex v.
        Also creates global variables e_v.
        """
        for v in self.vertices:
            if v not in self.idempotents:
                self.idempotents[v] = Path(self, [], v, v)

            e = self.idempotents[v]

            # Automatically set d(e_v) = 0
            self.differential[e] = self.zero()

            safe_make_global(f"e_{v}", e)

        return self.idempotents

    def idempotent(self, v):
        """
        Return the idempotent e_v.
        """
        if v not in self.vertices:
            raise ValueError(f"Vertex {v} is not in the quiver.")

        if v not in self.idempotents:
            e = Path(self, [], v, v)
            self.idempotents[v] = e

            # Automatically set d(e_v) = 0
            self.differential[e] = self.zero()

            safe_make_global(f"e_{v}", e)

        return self.idempotents[v]
    
    def add_arrow(self, name, source, target, grading=None):
        """
        Add a generating arrow.
        Also creates the global variable with this arrow's name.
        """
        if source not in self.vertices:
            raise ValueError(f"Source vertex {source} is not in the quiver.")

        if target not in self.vertices:
            raise ValueError(f"Target vertex {target} is not in the quiver.")

        if name in self.arrows:
            raise ValueError(f"Arrow {name} already exists.")

        # Important: use Arrow, not Path
        arrow = Arrow(self, name, source, target, grading=grading)
        self.arrows[name] = arrow

        # For now, every generating arrow has zero differential
        self.differential[name] = self.zero()

        safe_make_global(name, arrow)

        return arrow
        
    def add_arrows(self):
        while True:
            name = input("Arrow name, or type ok to finish: ")

            if name == "ok":
                break

            if name in self.arrows:
                print(f"{name} is already an arrow.")
                continue

            if len(self.vertices) == 1:
                source = next(iter(self.vertices))
                target = source
                print(
                    f"Quiver has only one vertex {source}, "
                    "source and target are automatically assigned."
                )
            else:
                source = input(f"Source of {name}: ")
                target = input(f"Target of {name}: ")

            if source not in self.vertices:
                print(f"Warning: {source} is not in the vertex set.")
                add_source = input(f"Add {source} as a new vertex? yes/no: ")

                if add_source == "yes":
                    self.vertices.add(source)
                    self.idempotents[source] = Path(self, [], source, source)
                    safe_make_global(f"e_{source}", self.idempotents[source])
                    print(f"Added vertex {source} and idempotent e_{source}.")
                else:
                    print(f"Arrow {name} was not added.")
                    continue

            if target not in self.vertices:
                print(f"Warning: {target} is not in the vertex set.")
                add_target = input(f"Add {target} as a new vertex? yes/no: ")

                if add_target == "yes":
                    self.vertices.add(target)
                    self.idempotents[target] = Path(self, [], target, target)
                    safe_make_global(f"e_{target}", self.idempotents[target])
                    print(f"Added vertex {target} and idempotent e_{target}.")
                else:
                    print(f"Arrow {name} was not added.")
                    continue

            # Create an Arrow
            new_arrow = Arrow(self, name, source, target)

            # Store it in the quiver
            self.arrows[name] = new_arrow

            # Ordinary arrows have differential zero
            self.differential[new_arrow] = self.zero()

            # Add the arrow as a global variable
            safe_make_global(name, new_arrow)

            print(f"Added arrow {name}: {source} -> {target}, grading {new_arrow.g()}.")

    def delete_arrows(self):
        while True:
            name = input("Enter an arrow name to delete, or type ok to finish: ")

            if name == "ok":
                break

            if name not in self.arrows:
                print(f"Arrow {name} does not exist.")
                continue

            # Save the Arrow object before deleting it from the dictionary
            arrow = self.arrows[name]

            # Delete its differential entry, if it exists
            if arrow in self.differential:
                del self.differential[arrow]

            # Delete from the quiver's generating arrow dictionary
            del self.arrows[name]

            # Also delete the global variable, if it exists
            if name in globals():
                del globals()[name]

            print(f"Deleted arrow {name}.")

    def multiply_paths(self, p, q):
        """
        Multiply two paths p and q.

        Convention:
            p*q is allowed iff target(p) == source(q).
        """
        if p.Q is not self or q.Q is not self:
            raise ValueError("Both paths must belong to this quiver.")

        if p.target != q.source:
            return Element({})

        new_arrows = list(p.arrows) + list(q.arrows)

        return Path(self, new_arrows, p.source, q.target)

    def l2(self, x, y):
        """
        Multiply two elements by distributing path multiplication.
        """
        x = to_element(x)
        y = to_element(y)

        result = Element({})

        for p, cp in x.terms.items():
            for q, cq in y.terms.items():
                product = self.multiply_paths(p, q)
                result = result + (cp * cq) * product

        return result

    def sg(self, x):
        """
        Return the Koszul sign (-1)^g(x).

        If g(x) is a known integer, return Sign.scalar(1) or Sign.scalar(-1).

        If g(x) is not numerically known, return a formal Sign object.
        """

        # If x is a VirtualElement, it must be homogeneous in sign.
        if isinstance(x, VirtualElement):
            if not x.terms:
                return Sign.one()

            signs = [
                self.sg(monomial)
                for monomial in x.terms
            ]

            first = signs[0]

            if all(sign == first for sign in signs):
                return first

            return Sign.eps(x)

        # If x is a VirtualMonomial, multiply signs of its factors.
        if isinstance(x, VirtualMonomial):
            result = Sign.one()

            for factor in x.factors:
                result = result * self.sg(factor)

            return result

        # If x is a VirtualAtom.
        if isinstance(x, VirtualAtom):
            if x.kind == "h":
                return Sign.scalar(-1) * self.sg(x.data)

            if x.kind == "d":
                return Sign.scalar(-1) * self.sg(x.data)

            return Sign.eps(x)

        # If x is an AinfMonomial.
        # g(M_n(x1,...,xn)) = sum g(xi) + 2 - n,
        # so sg(M_n) = (-1)^n * product sg(xi).
        if isinstance(x, AinfMonomial):
            result = Sign.scalar((-1) ** len(x.inputs))

            for input_x in x.inputs:
                result = result * self.sg(input_x)

            return result

        # Attached cells have |cell| = |d(cell)| - 1, so sg(cell) = -sg(d(cell)).
        if isinstance(x, Arrow) and (hasattr(x, "cell_prefix") or hasattr(x, "index")):
            differential = self.differential.get(x, None)

            if differential is not None:
                try:
                    return -self.sg(differential)
                except Exception:
                    pass

        # If x is an Element with one term, extract the path.
        if isinstance(x, Element):
            if len(x.terms) != 1:
                try:
                    signs = [self.sg(path) for path in x.terms]

                    if signs and all(sign == signs[0] for sign in signs):
                        return signs[0]
                except Exception:
                    pass

                return Sign.eps(x)

            x = next(iter(x.terms.keys()))

        # Try numerical degree first.
        try:
            degree = x.g()

            if isinstance(degree, int):
                return Sign.scalar((-1) ** degree)

        except Exception:
            pass

        # If x is a Path, build the product of signs of its arrows.
        if isinstance(x, Path):
            result = Sign.one()

            for a in x.arrows:
                name = a.name if hasattr(a, "name") else a

                try:
                    arrow = self.arrows[name]

                    if hasattr(arrow, "cell_prefix") or hasattr(arrow, "index"):
                        result = result * self.sg(arrow)
                        continue

                    degree = arrow.g()

                    if isinstance(degree, int):
                        result = result * Sign.scalar((-1) ** degree)
                    else:
                        result = result * Sign.eps(arrow)

                except Exception:
                    result = result * Sign.eps(a)

            return result

        # If x is an Arrow.
        if isinstance(x, Arrow):
            try:
                degree = x.g()

                if isinstance(degree, int):
                    return Sign.scalar((-1) ** degree)

            except Exception:
                pass

            return Sign.eps(x)

        # Scalars have sign 1.
        if is_scalar_coeff(x):
            return Sign.one()

        # Fallback.
        return Sign.eps(x)

    def g(self, f):
        if isinstance(f, VirtualElement):
            return f.g()

        if isinstance(f, VirtualMonomial):
            return f.g()

        if isinstance(f, VirtualAtom):
            return f.g()

        if isinstance(f, AinfElement):
            return f.g()

        if isinstance(f, AinfMonomial):
            return f.g()

        if isinstance(f, Element):
            return f.g()

        if isinstance(f, Path):
            return f.g()

        if isinstance(f, Arrow):
            return f.g()

        if is_scalar_coeff(f):
            return 0

        raise TypeError(f"Cannot compute grading of {f}.")

    def d_generator(self, name):
        """
        Differential of one generating arrow or idempotent.

        If the differential has not been assigned, return zero.
        """
        return self.differential.get(name, self.zero())

    def d_arrow(self, a):
        """
        Differential of a single Arrow.

        If the arrow has an explicitly assigned differential, return it.
        Otherwise return 0.

        This applies both to ordinary arrows and attached cells.
        """

        if a in self.differential:
            return self.differential[a]

        return self.zero()

    def d_path(self, p):
        """
        Differential of a single path using the signed Leibniz rule.

        If p = a1*a2*...*an, then

            d(p) =
                sum_i (-1)^g(a1...a_{i-1})
                a1...a_{i-1} d(ai) a_{i+1}...an
        """

        # Idempotents have zero differential.
        if len(p.arrows) == 0:
            return self.zero()

        # If p is a length-one path, use d_arrow.
        if len(p.arrows) == 1:
            arrow = self.arrows[p.arrows[0]]
            return self.d_arrow(arrow)

        result = self.zero()

        for i, arrow_name in enumerate(p.arrows):
            arrow = self.arrows[arrow_name]

            left_names = p.arrows[:i]
            right_names = p.arrows[i + 1:]

            # Left part a1...a_{i-1}
            if len(left_names) == 0:
                left = self.idempotent(p.source)
            else:
                left = Path(self, left_names, p.source, arrow.source)

            # Right part a_{i+1}...an
            if len(right_names) == 0:
                right = self.idempotent(p.target)
            else:
                right = Path(self, right_names, arrow.target, p.target)

            # Koszul sign from the degree of the left part
            sign = self.sg(left)

            # Important: sign is a coefficient, not a path.
            term = Element({left: sign}) * self.d_arrow(arrow) * Element({right: 1})

            result = result + term

        return result

    def d(self, x):
        """
        Linear differential.

        If x = c1*p1 + c2*p2 + ...,
        then d(x) = c1*d(p1) + c2*d(p2) + ...

        This is where linearity is implemented.
        """
        x = to_element(x)

        result = self.zero()

        for p, c in x.terms.items():
            result = result + c * self.d_path(p)

        return result

    def __repr__(self):
        output = "Quiver\n"
        output += f"Vertices: {self.vertices}\n"
        output += "Arrows:\n"

        for name, arrow in self.arrows.items():
            output += f"  {name}: {arrow.source} -> {arrow.target}, degree {arrow.g()}\n"

        output += "Idempotents:\n"
        for v in self.vertices:
            output += f"  e_{v}\n"

        return output
    def set_differential(self, generator, value):
        self.differential[generator] = to_element(value)

        
    def generate_cycles(self):
        """
        Generate more cycles in the DG sense.

        At this stage, since differentials of arrows are zero,
        every composable path is a cycle.

        By default, generates 10 more.
        If the user enters a number, generate that many instead.
        If the user enters 'ok', use the default 10.

        Already generated cycles are skipped, so the next call continues
        from the next new one.
        """

        answer = input(
            "Will generate 10 more cycles by default, press ok to continue "
            "or enter a number of new cycles to be generated: "
        )

        if answer == "" or answer.lower() == "ok":
            max_new = 10
        else:
            max_new = int(answer)

        if not hasattr(self, "generated_cycles"):
            self.generated_cycles = {}

        if not hasattr(self, "_generated_cycle_names"):
            self._generated_cycle_names = set()

        arrow_names = list(self.arrows.keys())

        newly_generated = []

        length = 1

        while len(newly_generated) < max_new:

            # Generate all composable paths of this length
            path_tuples = [(name,) for name in arrow_names]

            for _ in range(length - 1):
                next_path_tuples = []

                for path_tuple in path_tuples:
                    last_arrow = self.arrows[path_tuple[-1]]

                    for next_name in arrow_names:
                        next_arrow = self.arrows[next_name]

                        # Left-to-right composition:
                        # path_tuple followed by next_name
                        if last_arrow.target == next_arrow.source:
                            next_path_tuples.append(path_tuple + (next_name,))

                path_tuples = next_path_tuples

            if not path_tuples:
                break

            for path_tuple in path_tuples:
                path_name = "".join(path_tuple)

                # Skip paths already generated in previous calls
                if path_name in self._generated_cycle_names:
                    continue

                path_obj = Path(path_name, list(path_tuple))

                self.generated_cycles[path_name] = path_obj
                self._generated_cycle_names.add(path_name)

                newly_generated.append(path_obj)

                print(f"Generated cycle {path_name}: {path_obj}")

                if len(newly_generated) >= max_new:
                    break

            length += 1

        if len(newly_generated) == 0:
            print("No new cycles were generated.")

        return newly_generated
        
    def source_target_of_element(self, element):
        """
        Return the common source and target of all paths appearing in an Element.

        If the element is zero, or if its terms do not all have the same
        source and target, return (None, None).
        """

        if element.is_zero():
            return None, None

        sources = set()
        targets = set()

        for path in element.terms:
            sources.add(path.source)
            targets.add(path.target)

        if len(sources) != 1 or len(targets) != 1:
            return None, None

        return list(sources)[0], list(targets)[0]

    def mp_factor_key(self, x):
        """
        Stable key for a factor in an MPProduct.
        """

        if hasattr(x, "key"):
            return x.key()

        if hasattr(x, "name"):
            return ("Arrow", x.name)

        return ("Object", repr(x))

    def mp_product_key(self, P):
        """
        Stable key for a formal product of Massey products.
        """

        if isinstance(P, MasseyProduct):
            P = P.as_product()

        if not isinstance(P, MPProduct):
            raise TypeError(f"Expected MPProduct, got {P}.")

        return (
            "MPProduct",
            tuple(self.mp_factor_key(factor) for factor in P.factors)
        )

    def mp_expression_key(self, expr):
        """
        Stable key for a Massey-product expression.

        This includes single MasseyProduct objects, MPProduct objects,
        and MPElement linear combinations such as P - R.
        """

        if isinstance(expr, MasseyProduct):
            return expr.key()

        if isinstance(expr, MPProduct):
            return self.mp_product_key(expr)

        if isinstance(expr, MPElement):
            return expr.key()

        raise TypeError(f"Cannot make an MP-expression key for {expr}.")

    def to_mp_element(self, expr):
        """
        Convert an MP expression into a formal linear combination.
        """

        if isinstance(expr, MPElement):
            return expr

        if isinstance(expr, MPProduct):
            return MPElement(self, {expr: 1})

        if isinstance(expr, MasseyProduct):
            return MPElement(self, {expr.as_product(): 1})

        if expr == 0:
            return MPElement(self, {})

        raise TypeError(f"Cannot convert {expr} to an MPElement.")

    def expand_mp_product(self, P):
        """
        Expand a formal product of Massey products into an Element.
        """

        if isinstance(P, MasseyProduct):
            return P.expand()

        if not isinstance(P, MPProduct):
            print("Warning: expand_mp_product expects an MPProduct.")
            print("Got:", P)
            return None

        if len(P.factors) == 0:
            print("Warning: cannot expand the empty MPProduct.")
            return None

        result = None

        for factor in P.factors:
            if isinstance(factor, MasseyProduct):
                expanded = factor.expand()
            else:
                expanded = self.expand_mp_input(factor)

            if expanded is None:
                print(f"Warning: cannot expand factor {factor} in {P}.")
                return None

            if result is None:
                result = expanded
            else:
                result = result * expanded

        return result

    def expand_mp_expression(self, expr):
        """
        Expand an MP expression, including coefficient-weighted sums.
        """

        if isinstance(expr, MasseyProduct):
            return expr.expand()

        if isinstance(expr, MPProduct):
            return self.expand_mp_product(expr)

        if isinstance(expr, MPElement):
            result = self.zero()

            for product, coeff in expr.terms.items():
                expanded = self.expand_mp_product(product)

                if expanded is None:
                    return None

                result = result + coeff * expanded

            return result

        if isinstance(expr, Arrow) or isinstance(expr, Path) or isinstance(expr, Element):
            return to_element(expr)

        print("Warning: cannot expand this MP expression:", expr)
        return None

    def _record_resolved_mp_expression(self, expr, primitive):
        """
        Record a primitive for a general MP expression.
        """

        key = self.mp_expression_key(expr)
        self.resolved_mp_expressions[key] = primitive

        if isinstance(expr, MPElement) and len(expr.terms) >= 2:
            self.ensure_bridge_history()
            self.bridge_expressions[key] = expr
            self.bridge_cells[key] = primitive

            if key not in self.bridge_history:
                self.bridge_history.append(key)

            if hasattr(primitive, "__dict__"):
                primitive.cell_kind = "bridge"
                primitive.bridge_key = key

        print(f"Recorded primitive of {expr}:")
        print(f"    h[{expr}] = {primitive}")

        return primitive

    def ensure_attachment_history(self):
        """
        Make sure universal attachment history exists.
        """

        if not hasattr(self, "attachment_history"):
            self.attachment_history = []

    def record_cell_attachment(self, kind, key, cell, expression=None, differential=None, family=None):
        """
        Record one attached cell in universal chronological history.
        """

        self.ensure_attachment_history()

        entry = {
            "kind": kind,
            "key": key,
            "cell": cell,
            "expression": expression,
            "differential": differential,
            "family": family,
        }

        self.attachment_history.append(entry)
        return entry

    def forget_attachment_history_entry(self, kind=None, key=None, cell=None):
        """
        Remove the most recent matching universal attachment-history entry.
        """

        self.ensure_attachment_history()

        for i in range(len(self.attachment_history) - 1, -1, -1):
            entry = self.attachment_history[i]

            if kind is not None and entry.get("kind") != kind:
                continue

            if key is not None and entry.get("key") != key:
                continue

            if cell is not None and entry.get("cell") is not cell:
                continue

            return self.attachment_history.pop(i)

        return None

    def attachment_record_for_cell(self, original_input, differential, cell_family, cell_index, new_cell):
        """
        Record a new cell attachment with the right universal undo kind.
        """

        if isinstance(original_input, MasseyProduct):
            key = self.mp_key(*original_input.inputs)
            return self.record_cell_attachment(
                "mp_cell",
                key,
                new_cell,
                expression=original_input,
                differential=differential,
                family="v"
            )

        if isinstance(original_input, MPElement) and len(original_input.terms) >= 2:
            key = self.mp_expression_key(original_input)
            return self.record_cell_attachment(
                "bridge",
                key,
                new_cell,
                expression=original_input,
                differential=differential,
                family="v"
            )

        if isinstance(original_input, (MPProduct, MPElement)):
            key = self.mp_expression_key(original_input)
            return self.record_cell_attachment(
                "mp_expression_cell",
                key,
                new_cell,
                expression=original_input,
                differential=differential,
                family="v"
            )

        key = cell_family.key(cell_index)
        return self.record_cell_attachment(
            "ordinary_cell",
            key,
            new_cell,
            expression=cell_index,
            differential=differential,
            family="u"
        )

    def attach_one_cell(self, f):
        """
        Attach one cell.

        If f is an ordinary cycle, attach u[f].

        If f is a Massey-product expression, attach v[f], where f is the
        actual formal expression, not merely a display string.
        """
        original_input = f

        # Case 1: f is a Massey-product expression.
        if isinstance(f, (MasseyProduct, MPProduct, MPElement)):
            differential = self.expand_mp_expression(f)

            if differential is None:
                print(f"Warning: cannot attach cell to {f}, because it cannot be expanded.")
                return None

            cell_family = self.v
            cell_prefix = "v"
            cell_index = f

        # Case 2: f is an ordinary cycle.
        else:
            differential = f

            if isinstance(differential, Arrow) or isinstance(differential, Path):
                differential = Element({differential: 1})

            if not isinstance(differential, Element):
                print("The differential must be an Element, Arrow, Path, MasseyProduct, MPProduct, or MPElement.")
                print("No cell was attached.")
                return None

            cell_family = self.u
            cell_prefix = "u"
            cell_index = differential

        # Check source and target.
        source, target = self.source_target_of_element(differential)

        if source is None or target is None:
            print("Warning: the chosen differential does not have a unique source and target.")
            print(f"The expression {original_input} is not homogeneous as a quiver element.")
            print("No cell was attached.")
            return None

        # Check cycle condition.
        df = self.d(differential)

        if not df.is_zero():
            print("Warning: the chosen differential is not a cycle.")
            print(f"d({original_input}) = {df}")
            print("No cell was attached.")
            return None

        # Check whether this cell already exists.
        if cell_index in cell_family:
            print("A cell with this differential/index has already been attached.")
            print(f"Existing cell: {cell_family[cell_index]}")
            return cell_family[cell_index]

        # Choose an internal Python name.
        internal_name = f"cell_{len(self.arrows) + 1}"

        while internal_name in self.arrows:
            internal_name = f"cell_{len(self.arrows) + 1}"

        # Degree convention: |cell| = |differential| - 1.
        try:
            cell_degree = differential.g() - 1
        except Exception:
            cell_degree = None

        # Height convention: h(cell) = h(differential) + 1.
        try:
            cell_height = differential.h() + 1
        except Exception:
            cell_height = 1

        # Create the new cell.
        new_cell = Arrow(
            self,
            internal_name,
            source,
            target,
            grading=cell_degree,
            height=cell_height
        )

        # Store the actual indexing object.
        new_cell.index = cell_index
        new_cell.cell_prefix = cell_prefix

        # Display name uses the real index object.
        new_cell.display_name = f"{cell_prefix}[{cell_index}]"

        # Register in the quiver.
        self.arrows[internal_name] = new_cell
        self.differential[new_cell] = differential

        # Store in the correct cell family.
        cell_family[cell_index] = new_cell

        # If this is an ordinary monomial, also store in self.cells.
        if not isinstance(original_input, (MasseyProduct, MPProduct, MPElement)):
            if len(differential.terms) == 1:
                monomial_path = next(iter(differential.terms.keys()))
                key = tuple(
                    a.name if hasattr(a, "name") else a
                    for a in monomial_path.arrows
                )
                self.cells[key] = new_cell

        # If this is a MasseyProduct, record its primitive.
        if isinstance(original_input, MasseyProduct):
            self._record_resolved_mp(original_input, new_cell)

        # If this is a general MP expression, record its primitive too.
        if isinstance(original_input, (MPProduct, MPElement)):
            self._record_resolved_mp_expression(original_input, new_cell)

        self.attachment_record_for_cell(
            original_input,
            differential,
            cell_family,
            cell_index,
            new_cell
        )

        print(f"Attached new cell {new_cell}")
        print(f"Internal arrow name: {internal_name}")
        print(f"Source: {source}")
        print(f"Target: {target}")
        print(f"d({new_cell}) = {differential}")

        return new_cell

    def attach_mp_cell(self, f=None):
        """
        Attach an MP cell v[f] for a Massey-product expression f.

        The expression may be a MasseyProduct, an MPProduct, or an
        MPElement linear combination with coefficients.
        """

        if f is None:
            while True:
                expr = input("Choose MP expression for v[...], or type ok to finish: ")

                if expr == "ok":
                    break

                self.attach_mp_cell(expr)

            return None

        if isinstance(f, str):
            try:
                f = eval(f, self.mp_eval_environment())
            except Exception as e:
                print("Could not understand this MP expression.")
                print("Python error:", e)
                return None

        if not isinstance(f, (MasseyProduct, MPProduct, MPElement)):
            print("The MP cell index must be a MasseyProduct, MPProduct, or MPElement.")
            print("No cell was attached.")
            return None

        return self.attach_one_cell(f)

    def attach_mp_difference_cell(self, g1, g2=None):
        """
        Attach v[g] where g = g1 - g2 is a difference of MPProducts.

        If g2 is omitted, g1 may already be an MPElement such as P - R.
        Coefficients on MPProducts are preserved.
        """

        if isinstance(g1, str):
            try:
                g1 = eval(g1, self.mp_eval_environment())
            except Exception as e:
                print("Could not understand the first MP expression.")
                print("Python error:", e)
                return None

        if isinstance(g2, str):
            try:
                g2 = eval(g2, self.mp_eval_environment())
            except Exception as e:
                print("Could not understand the second MP expression.")
                print("Python error:", e)
                return None

        try:
            if g2 is None:
                g = self.to_mp_element(g1)
            else:
                g = self.to_mp_element(g1) - self.to_mp_element(g2)
        except Exception as e:
            print("Could not form the MP difference.")
            print("Python error:", e)
            return None

        return self.attach_one_cell(g)

    def attach_mp_diff_cell(self, g1, g2=None):
        """
        Short alias for attach_mp_difference_cell.
        """

        return self.attach_mp_difference_cell(g1, g2)

    def attach_bridge(self, g1, g2=None):
        """
        Attach a bridge cell v[g1 - g2].

        Coefficients on the two MPProducts are allowed, e.g.
        Q.attach_bridge(2*P, 3*R).
        """

        return self.attach_mp_difference_cell(g1, g2)

    def attach_bridge_cell(self, g1, g2=None):
        """
        Alias for attach_bridge.
        """

        return self.attach_bridge(g1, g2)

    def attach_cells(self):
        """
        Attach new cells to cycles.

        The user enters an element f.
        If all terms of f have the same source and target, and d(f) = 0,
        then we create a new Arrow cell satisfying

            d(u[f]) = f.

        The cell is stored in the CellFamily self.u, so later one can call

            u[f]

        to retrieve it.
        """

        while True:
            diff_input = input("Choose the differential for the new cell, or type ok to finish: ")

            if diff_input == "ok":
                break

            # Step 1: interpret the user's input as a Python expression
            try:
                f = eval(diff_input, globals())
            except Exception as e:
                print("Could not understand this differential.")
                print("Python error:", e)
                continue

            # Step 2: if f is an Arrow or Path, convert it into an Element
            if isinstance(f, Arrow) or isinstance(f, Path):
                f = Element({f: 1})

            if not isinstance(f, Element):
                print("The differential must be an Element, Arrow, or Path.")
                print("No cell was attached.")
                continue

            # Step 3: check whether all terms of f have the same source and target
            source, target = self.source_target_of_element(f)

            if source is None or target is None:
                print("Warning: the chosen differential does not have a unique source and target.")
                print(f"The expression {diff_input} is not homogeneous as a quiver element.")
                print("For example, a sum x + y is invalid if x and y have different sources or targets.")
                print("No cell was attached.")
                continue

            # Step 4: check whether f is a cycle
            df = self.d(f)

            if not df.is_zero():
                print("Warning: the chosen differential is not a cycle.")
                print(f"d({diff_input}) = {df}")
                print("No cell was attached.")
                continue

            # Step 5: check whether a cell with this differential already exists
            if f in self.u:
                print("A cell with this differential has already been attached.")
                print(f"Existing cell: u[{f}] = {self.u[f]}")
                continue

            # Step 6: choose an internal Python name for the new Arrow
            internal_name = f"cell_{len(self.u.cells) + 1}"

            while internal_name in self.arrows:
                internal_name = f"cell_{len(self.u.cells) + 2}"

            # Step 7: determine degree of the new cell
            #
            # Convention:
            # If d raises degree by 1, then |u[f]| = |f| - 1.
            try:
                cell_degree = f.g() - 1
            except Exception:
                cell_degree = None

            # Step 7.5: determine height of the new cell
            #
            # Convention:
            # If d(u[f]) = f, then h(u[f]) = h(f) + 1.
            try:
                cell_height = f.h() + 1
            except Exception:
                cell_height = 1

            # Step 8: create the new cell as an ordinary Arrow
            new_cell = Arrow(
                self,
                internal_name,
                source,
                target,
                grading=cell_degree,
                height=cell_height
            )
            new_cell.display_name = f"u[{f}]"


            # Step 9: register it in the quiver
            self.arrows[internal_name] = new_cell
            self.differential[new_cell] = f

            # Step 10: store it in the cell family
            self.u[f] = new_cell

            # Step 10.5: if f is a single monomial, also store it in self.cells
            # using the tuple of arrow names as the key.
            if len(f.terms) == 1:
                monomial_path = next(iter(f.terms.keys()))
                key = tuple(a.name if hasattr(a, "name") else a for a in monomial_path.arrows)
                self.cells[key] = new_cell

            self.record_cell_attachment(
                "ordinary_cell",
                self.u.key(f),
                new_cell,
                expression=f,
                differential=f,
                family="u"
            )

            print(f"Attached new cell u[{f}]")
            print(f"Internal arrow name: {internal_name}")
            print(f"Source: {source}")
            print(f"Target: {target}")
            print(f"d(u[{f}]) = {f}")

    def path_from_names(self, names):
        """
        Build a Path from a list of arrow names, using the quiver to determine
        source and target.

        This avoids accidentally creating paths with source=None or target=None.
        """

        names = [
            a.name if hasattr(a, "name") else a
            for a in names
        ]

        if len(names) == 0:
            raise ValueError("path_from_names needs at least one arrow name.")

        source = self.arrows[names[0]].source
        target = self.arrows[names[-1]].target

        # Check composability.
        for i in range(len(names) - 1):
            a = self.arrows[names[i]]
            b = self.arrows[names[i + 1]]

            if a.target != b.source:
                raise ValueError(
                    f"Arrows {a.name} and {b.name} are not composable: "
                    f"{a.target} != {b.source}."
                )

        return Path(self, names, source, target)

    def find_primitives_monomial(self, path):
        """
        Given a monomial/path f, search for substrings M of f
        such that u[M] exists.

        If f = L M R, then the candidate primitive is

            (-1)^g(L) L u[M] R.

        Sign issues are handled using self.sg(L).
        """

        # If input is an Element with one term, extract its unique path.
        if isinstance(path, Element):
            if len(path.terms) != 1:
                raise ValueError("find_primitives_monomial expects a single monomial, not a sum.")
            path = next(iter(path.terms.keys()))

        # If input is an Arrow, convert it to a length-one path.
        if isinstance(path, Arrow):
            path = Path(self, [path.name], path.source, path.target)

        # Normalize the arrow names.
        arrow_names = [
            a.name if hasattr(a, "name") else a
            for a in path.arrows
        ]

        # Rebuild the path with reliable source/target data.
        path = self.path_from_names(arrow_names)

        # Check whether the input monomial is a cycle.
        path_element = Element({path: 1})
        d_path_element = self.d(path_element)

        if not d_path_element.is_zero():
            print("Warning: the input is not a cycle.")
            print(f"d({path_element}) = {d_path_element}")
            print("Continuing anyway and searching for primitive candidates.")

        n = len(arrow_names)
        candidates = []

        for i in range(n):
            for j in range(i + 1, n + 1):
                left_names = arrow_names[:i]
                middle_names = arrow_names[i:j]
                right_names = arrow_names[j:]

                middle_key = tuple(middle_names)

                if middle_key in self.cells:
                    cell = self.cells[middle_key]
                    cell_name = cell.name if hasattr(cell, "name") else cell

                    # Build the full primitive path directly:
                    #
                    #     L u[M] R
                    #
                    primitive_names = left_names + [cell_name] + right_names
                    primitive_path = self.path_from_names(primitive_names)

                    # Koszul sign: (-1)^g(L)
                    if left_names:
                        left_path = self.path_from_names(left_names)
                        sign = self.sg(left_path)
                    else:
                        sign = 1

                    term = Element({primitive_path: sign})
                    candidates.append(term)

        return candidates


    def homotopy_monomial(self, f):
        """
        Given a monomial element f = c*m, return c*p where d(p)=m.

        This assumes f has exactly one term.
        """

        f = to_element(f)

        if f.is_zero():
            return Element({})

        if len(f.terms) != 1:
            raise ValueError(
                f"homotopy_monomial expects a monomial, but got {f}"
            )

        monomial, coeff = next(iter(f.terms.items()))

        bare_monomial = Element({monomial: 1})

        primitives = self.find_primitives_monomial(bare_monomial)

        if len(primitives) == 0:
            raise ValueError(
                f"No primitive found for monomial {bare_monomial}"
            )

        primitive = primitives[0]
        answer = coeff * primitive

        error = self.d(answer) - f

        if not error.is_zero():
            raise ValueError(
                f"homotopy_monomial failed: d({answer}) = {self.d(answer)}, "
                f"but expected {f}"
            )

        return answer


    def homotopy(self, f):
        """
        Apply homotopy_monomial term-by-term.

        If f = sum c_i*m_i, then

            homotopy(f) = sum c_i*homotopy_monomial(m_i).
        """

        f = to_element(f)

        if f.is_zero():
            return Element({})

        result = Element({})

        for monomial, coeff in f.terms.items():
            term = Element({monomial: coeff})
            result = result + self.homotopy_monomial(term)

        return result


    def path_key(self, p):
        """
        Canonical key for a path.
        """

        if isinstance(p, Arrow):
            return (p.name,)

        if isinstance(p, Path):
            return tuple(
                a.name if hasattr(a, "name") else a
                for a in p.arrows
            )

        raise TypeError(f"Cannot make path key for {p}.")

    def element_key(self, f):
        """
        Canonical key for an Element.

        This should be used for dictionaries such as homotopy_registry.
        """

        f = to_element(f)

        pieces = []

        for path, coeff in f.terms.items():
            pieces.append((self.path_key(path), coeff_key(coeff)))

        pieces.sort()

        return tuple(pieces)

    def homotopy_virtual(self, f):
        """
        Formal virtual homotopy h[f].

        This is linear over coefficients:

            h[a*f] = a*h[f]

        and also over sums:

            h[f + g] = h[f] + h[g]

        The actual atoms are only created for coefficient-free monomials.
        """

        f = self.to_virtual_element(f)

        if not f.terms:
            return VirtualElement(self, {})

        # Linearity: h(sum c_i m_i) = sum c_i h(m_i)
        if len(f.terms) != 1:
            result = VirtualElement(self, {})

            for monomial, coeff in f.terms.items():
                bare_term = VirtualElement(self, {monomial: 1})
                result = result + coeff * self.homotopy_virtual(bare_term)

            return result

        # Scalar pull-out: h[c*m] = c*h[m]
        monomial, coeff = next(iter(f.terms.items()))

        if coeff != 1:
            bare_term = VirtualElement(self, {monomial: 1})
            return coeff * self.homotopy_virtual(bare_term)

        # Now f is coefficient-free, so create/reuse the virtual atom h[f].
        key = self.virtual_key(f)

        if key in self.virtual_homotopies:
            return self.virtual_homotopies[key]

        atom = VirtualAtom(self, "h", f)
        monomial = atom.as_monomial()
        element = VirtualElement(self, {monomial: 1})

        self.virtual_homotopies[key] = element

        return element

    def assign_homotopy(self, f, primitive):
        """
        Assign h(f) = primitive.

        This fixes the chosen homotopy value permanently, unless overwritten
        manually.
        """

        f = to_element(f)
        primitive = to_element(primitive)

        key = self.homotopy_key(f)

        # Optional safety check: verify d(primitive) = f.
        error = self.d(primitive) - f

        if not error.is_zero():
            raise ValueError(
                f"Cannot assign h({f}) = {primitive}, because "
                f"d({primitive}) = {self.d(primitive)}, not {f}."
            )

        self.homotopy_registry[key] = primitive

        if key in self.pending_homotopies:
            del self.pending_homotopies[key]

        print(f"Assigned h[{f}] = {primitive}")

    def homotopy_key(self, f):
        """
        Key used to store and retrieve chosen homotopies.
        """

        return self.element_key(f)

    def M(self, *inputs):
        return AinfMonomial(self, inputs)

    def to_virtual_monomial(self, f):
        if isinstance(f, VirtualMonomial):
            return f

        if isinstance(f, VirtualAtom):
            return f.as_monomial()

        if isinstance(f, AinfMonomial):
            return VirtualMonomial(self, (f,))

        if isinstance(f, Arrow):
            return VirtualMonomial(self, (f,))

        if isinstance(f, Path):
            return VirtualMonomial(self, (f,))

        raise TypeError(f"Cannot convert {f} to a VirtualMonomial.")

    def to_virtual_element(self, f):
        if isinstance(f, VirtualElement):
            return f

        if isinstance(f, VirtualMonomial):
            return VirtualElement(self, {f: 1})

        if isinstance(f, VirtualAtom):
            return VirtualElement(self, {f.as_monomial(): 1})

        if isinstance(f, Arrow):
            monomial = VirtualMonomial(self, (f,))
            return VirtualElement(self, {monomial: 1})

        if isinstance(f, Path):
            monomial = VirtualMonomial(self, (f,))
            return VirtualElement(self, {monomial: 1})

        if isinstance(f, AinfMonomial):
            monomial = VirtualMonomial(self, (f,))
            return VirtualElement(self, {monomial: 1})

        if isinstance(f, Element):
            terms = {}

            for path, coeff in f.terms.items():
                monomial = VirtualMonomial(self, (path,))
                terms[monomial] = terms.get(monomial, 0) + coeff

            return VirtualElement(self, terms)

        if is_scalar_coeff(f):
            if f == 0:
                return VirtualElement(self, {})

            raise TypeError("Only 0 can be converted to a VirtualElement.")

        raise TypeError(f"Cannot convert {f} to a VirtualElement.")

    def virtual_key(self, f):
        if isinstance(f, VirtualElement):
            return f.key()

        if isinstance(f, VirtualMonomial):
            return f.key()

        if isinstance(f, VirtualAtom):
            return f.key()

        if isinstance(f, Element):
            return f.key()

        if isinstance(f, Path):
            return f.key()

        if isinstance(f, Arrow):
            return f.name

        return str(f)

    def virtual_product(self, x, y):
        """
        Multiply two virtual elements by concatenating virtual monomials.

        This is the formal product used for expressions involving h[...].
        """

        x = self.to_virtual_element(x)
        y = self.to_virtual_element(y)

        result_terms = {}

        for mx, cx in x.terms.items():
            for my, cy in y.terms.items():
                product = VirtualMonomial(
                    self,
                    tuple(mx.factors) + tuple(my.factors)
                )

                result_terms[product] = result_terms.get(product, 0) + cx * cy

        return VirtualElement(self, result_terms)


    def G_L(self, *inputs):
        """
        Auxiliary operation G(L(...)).

        For one input:

            G_L(x) = -x

        For two or more inputs:

            G_L(x1,...,xn) = h[L(x1,...,xn)]
        """

        if len(inputs) == 1:
            return -self.to_virtual_element(inputs[0])

        return self.homotopy_virtual(self.L(*inputs))


    def virtual_d(self, x):
        """
        Virtual differential.

        If x is an ordinary Element/Path/Arrow, compute the actual DG
        differential and convert it to a VirtualElement.

        If x is already virtual/formal, return the formal symbol d[x].
        """

        if isinstance(x, Element) or isinstance(x, Path) or isinstance(x, Arrow):
            return self.to_virtual_element(self.d(x))

        if isinstance(x, VirtualElement):
            if not x.terms:
                return VirtualElement(self, {})

            atom = VirtualAtom(self, "d", x)
            monomial = atom.as_monomial()
            return VirtualElement(self, {monomial: 1})

        if isinstance(x, VirtualMonomial):
            atom = VirtualAtom(self, "d", x)
            monomial = atom.as_monomial()
            return VirtualElement(self, {monomial: 1})

        if isinstance(x, VirtualAtom):
            atom = VirtualAtom(self, "d", x)
            monomial = atom.as_monomial()
            return VirtualElement(self, {monomial: 1})

        if isinstance(x, AinfMonomial):
            atom = VirtualAtom(self, "d", x)
            monomial = atom.as_monomial()
            return VirtualElement(self, {monomial: 1})

        raise TypeError(f"Cannot compute virtual differential of {x}.")

    def H_L(self, *inputs):
        """
        The corrected block used in the recursive lambda formula.

        For one input:

            H_L(x) = x

        For two or more inputs:

            H_L(x1,...,xn) = h[L(x1,...,xn)]
        """

        if len(inputs) == 0:
            raise ValueError("H_L needs at least one input.")

        if len(inputs) == 1:
            return self.to_virtual_element(inputs[0])

        return self.homotopy_virtual(self.L(*inputs))


    def L_split_coefficient(self, left_factor, right_length):
        """
        Koszul coefficient for one recursive split.

        If the right block has length 1, no homotopy is being moved past
        the left factor, so the coefficient is 1.

        If the right block has length >= 2, the right factor is h[L(...)].
        The Leibniz rule contributes sg(left_factor), so we insert

            -sg(left_factor)

        This gives:

            L(a,b,c) = h[L(a,b)]*c - sg(a)*a*h[L(b,c)].
        """

        if right_length == 1:
            return Sign.one()

        return -self.sg(left_factor)


    def L(self, *inputs):
        """
        Recursive higher multiplication lambda_n.

        Unary case:

            L(f) = d(f)

        Binary case:

            L(a,b) = a*b

        Higher case:

            L(x1,...,xn)
              =
            sum over splits i:
                coeff_i * H_L(x1,...,xi) * H_L(x_{i+1},...,xn)

        where

            H_L(x) = x
            H_L(x1,...,xk) = h[L(x1,...,xk)] for k >= 2

        and coeff_i is determined by L_split_coefficient.
        """

        n = len(inputs)

        if n == 0:
            raise ValueError("L needs at least one input.")

        # L(f) = d(f)
        if n == 1:
            return self.virtual_d(inputs[0])

        # L(a,b) = a*b
        if n == 2:
            return self.virtual_product(inputs[0], inputs[1])

        result = VirtualElement(self, {})

        for i in range(1, n):
            left_inputs = inputs[:i]
            right_inputs = inputs[i:]

            left_factor = self.H_L(*left_inputs)
            right_factor = self.H_L(*right_inputs)

            coeff = self.L_split_coefficient(
                left_factor,
                len(right_inputs)
            )

            term = coeff * self.virtual_product(left_factor, right_factor)

            result = result + term

        return result


    def mp_key(self, *inputs):
        """
        Canonical key for a Massey product.
        """
        inputs = self.normalize_mp_inputs(inputs)
        return tuple(inputs)


    def is_zero_mp_input(self, x):
        """
        Return True if x should be treated as the zero input in an MP.
        """

        if isinstance(x, Element):
            return x.is_zero()

        try:
            return is_zero_coeff(x)
        except Exception:
            return False

    def has_zero_mp_input(self, inputs):
        """
        Return True if any MP input is zero.
        """

        return any(self.is_zero_mp_input(x) for x in tuple(inputs))


    def are_composable(self, a, b):
        """
        Return True if a and b are composable.
        """
        if self.is_zero_mp_input(a) or self.is_zero_mp_input(b):
            return True

        return a.target == b.source



    def _record_resolved_mp(self, M, primitive, check=False):
        """
        Internal helper.

        Record that the Massey product M has primitive `primitive`.

        This means:
            d(primitive) = M.expand()

        By default check=False, because higher expand_mp may not be fully
        implemented yet.
        """
        if M is None:
            print("Warning: cannot resolve None as a Massey product.")
            return None

        if not isinstance(M, MasseyProduct):
            print("Warning: expected a MasseyProduct.")
            print("Got:", M)
            return None

        if M.Q is not self:
            print("Warning: this MasseyProduct belongs to a different quiver.")
            return None

        key = self.normalize_mp_inputs(M.inputs)
        
        # Make sure this Massey product is recorded as existing.
        if key not in self.massey_products:
            self.massey_products[key] = M

        if check:
            expanded = M.expand()

            if expanded is None:
                print(f"Warning: cannot check primitive for {M}, because it does not expand.")
                return None

            primitive_element = to_element(primitive)
            d_primitive = self.d(primitive_element)

            if d_primitive != expanded:
                print(f"Warning: {primitive} is not a primitive of {M}.")
                print(f"d({primitive}) = {d_primitive}")
                print(f"{M}.expand() = {expanded}")
                return None

        self.resolved_massey_products[key] = primitive
        
        self.ensure_mp_cell_history()

        if key not in self.mp_cell_history:
            self.mp_cell_history.append(key)
        self.mp_primitive_candidates[key] = [primitive]

        print(f"Recorded primitive of {M}:")
        print(f"    h[{M}] = {primitive}")

        return primitive

    def mp_is_resolved(self, *inputs):
        """
        Return True if mp(inputs) has a recorded primitive.
        """
        key = tuple(inputs)
        return key in self.resolved_massey_products


    def mp_block_primitive(self, *inputs, allow_bridge_replacements=True):
        """
        Return a primitive for a product block, if one is recorded.

        This checks ordinary resolved Massey products, general MP-expression
        cells such as v[Q.mp(x1)*Q.mp(x2)*Q.mp(x3)], ordinary monomial
        cells such as u[x1*x2*x3], and optionally bridge-replacement
        primitives.
        """

        inputs = self.normalize_mp_inputs(inputs)

        if len(inputs) == 0:
            return None

        if self.has_zero_mp_input(inputs):
            return self.zero()

        if len(inputs) == 1:
            return self.expand_mp_input(inputs[0])

        key = tuple(inputs)

        try:
            if key in self.resolved_massey_products:
                return self.resolved_massey_products[key]
        except TypeError:
            pass

        try:
            factors = self.canonical_mp_factors(inputs)
            expr_key = self.mp_expression_key(MPProduct(self, factors))
        except Exception:
            factors = None
            expr_key = None

        if expr_key is not None:
            primitive = getattr(self, "resolved_mp_expressions", {}).get(expr_key, None)

            if primitive is not None:
                return primitive

            target_product = MPProduct(self, factors)
            target_key = self.mp_product_key(target_product)

            for expression_key, primitive in getattr(self, "resolved_mp_expressions", {}).items():
                expression = None

                for entry in getattr(self, "attachment_history", []):
                    if entry.get("key", None) == expression_key:
                        expression = entry.get("expression", None)
                        break

                if not isinstance(expression, MPElement) or len(expression.terms) != 1:
                    continue

                product, coeff = next(iter(expression.terms.items()))

                if is_zero_coeff(coeff):
                    continue

                if self.mp_product_key(product) != target_key:
                    continue

                try:
                    if coeff == 1:
                        inverse_coeff = simplify_coeff(1)
                    elif coeff == -1:
                        inverse_coeff = simplify_coeff(-1)
                    else:
                        inverse_coeff = simplify_coeff(1 / coeff)
                except Exception:
                    continue

                return inverse_coeff * primitive

        if factors is None:
            return None

        element = self.mp_factors_to_element(factors)

        if element is not None and len(element.terms) == 1:
            path, coeff = next(iter(element.terms.items()))

            if coeff == 1:
                path_key = tuple(path.arrows)
                primitive = getattr(self, "cells", {}).get(path_key, None)

                if primitive is not None:
                    return primitive

        if allow_bridge_replacements:
            return self.bridge_mp_block_primitive(*inputs)

        return None

    def bridge_mp_block_primitive(self, *inputs):
        """
        Return a primitive for a block produced by bridge replacements.

        This is deliberately separate from the direct checks above so that
        active bridge searches can disable it and avoid bridge loops.
        """

        inputs = self.normalize_mp_inputs(inputs)

        try:
            factors = self.canonical_mp_factors(inputs)
        except Exception:
            return None

        if len(factors) < 2:
            return None

        product = MPProduct(self, factors)
        target = self.mp_factors_to_element(factors)

        if target is None:
            return None

        candidates = self.bridge_replacement_primitive_candidates(
            product,
            record=False,
            minimal_only=True
        )

        for candidate in candidates:
            primitive = self.primitive_candidate_to_element(candidate)

            if primitive is None:
                continue

            if (self.d(primitive) - target).is_zero():
                return primitive

        return None

    def mp_block_has_primitive(self, inputs, allow_bridge_replacements=True):
        """
        Return True if a lower product block has a recorded primitive.
        """

        return self.mp_block_primitive(
            *tuple(inputs),
            allow_bridge_replacements=allow_bridge_replacements
        ) is not None


    def mp_primitive(self, *inputs):
        """
        Return the recorded primitive of Q.mp(inputs), if it exists.
        """
        inputs = self.normalize_mp_inputs(inputs)
        key = tuple(inputs)

        if key not in self.resolved_massey_products:
            print(f"Warning: Q.mp{key} has no recorded primitive.")
            return None

        return self.resolved_massey_products[key]





    def mp(self, *inputs):
        """
        Try to create/retrieve the Massey product mp(inputs).

        If it is not currently defined, print a warning and return None.
        """
        inputs = self.normalize_mp_inputs(inputs)
        key = tuple(inputs)

        if key in self.massey_products:
            return self.massey_products[key]

        if not self.can_define_mp(*inputs):
            missing = self.missing_requirements_for_mp(*inputs)
            print(
                f"Warning: mp{key} is not defined yet. "
                f"Missing resolved lower products: {missing}"
            )
            return None

        M = MasseyProduct(self, key)
        self.massey_products[key] = M
        return M


    def can_define_mp(self, *inputs):
        """
        Return True if mp(inputs) is currently defined.
        """
        inputs = self.normalize_mp_inputs(inputs)
        n = len(inputs)

        if n == 0:
            return False

        if self.has_zero_mp_input(inputs):
            return True

        if n == 1:
            return True

        if n == 2:
            return self.are_composable(inputs[0], inputs[1])

        required = self.required_lower_mps(*inputs)

        for subkey in required:
            if not self.mp_block_has_primitive(subkey):
                return False

        return True


    def required_lower_mps(self, *inputs):
        """
        Return all proper consecutive lower Massey products required
        to define mp(inputs).

        For mp(a,b,c), returns:
            (a,b), (b,c)

        For mp(a,b,c,d), returns:
            (a,b), (b,c), (c,d), (a,b,c), (b,c,d)
        """
        inputs = self.normalize_mp_inputs(inputs)
        n = len(inputs)
        required = []

        if self.has_zero_mp_input(inputs):
            return required

        for length in range(2, n):
            for start in range(0, n - length + 1):
                subkey = tuple(inputs[start:start + length])
                subkey = self.normalize_mp_inputs(subkey)
                required.append(subkey)

        return required


    def missing_requirements_for_mp(self, *inputs):
        """
        Return lower Massey products that are required but not yet resolved.
        """
        inputs = self.normalize_mp_inputs(inputs)
        missing = []

        for subkey in self.required_lower_mps(*inputs):
            if not self.mp_block_has_primitive(subkey):
                missing.append(subkey)

        return missing
        
    def show_resolved_mps(self):
        """
        Display all resolved Massey products.
        """
        if not self.resolved_massey_products:
            print("No resolved Massey products recorded.")
            return

        for key, primitive in self.resolved_massey_products.items():
            print(f"mp{key} -> {primitive}")

    def mp_eval_environment(self):
        """
        Create an evaluation environment for resolve_mp.

        In this environment, every generating arrow x is interpreted as
        the length-one Massey product Q.mp(x).

        Therefore, inside Q.resolve_mp(), typing:

            x*y

        means:

            Q.mp(x) * Q.mp(y) = Q.mp(x, y).
        """
        env = dict(globals())

        # Make Q available if the user types Q.mp(...)
        env["Q"] = self

        # Replace each arrow name by the length-one Massey product Q.mp(arrow).
        for name, arrow in self.arrows.items():
            env[name] = self.mp(arrow)

        return env

    def resolve_mp(self, M=None, primitive=None, check=False):
        """
        Resolve Massey products.

        Interactive usage:

            Q.resolve_mp()

        Then enter either:

            Q.mp(x, y)

        or, in the special MP input environment:

            x*y

        where x*y is interpreted as an MPProduct and then converted to
        Q.mp(x, y).

        Direct usage:

            Q.resolve_mp(Q.mp(x, y))

        or:

            Q.resolve_mp(Q.mp(x) * Q.mp(y))
        """

        # Direct storage mode:
        # Q.resolve_mp(Q.mp(...), primitive)
        if M is not None and primitive is not None:
            M = self.mp_from_product(M)

            if M is None:
                print("Warning: resolve_mp expects a MasseyProduct or MPProduct.")
                return None

            return self._record_resolved_mp(M, primitive, check=check)

        # Direct attach mode:
        # Q.resolve_mp(Q.mp(...))
        # Q.resolve_mp(Q.mp(x) * Q.mp(y))
        if M is not None and primitive is None:
            M = self.mp_from_product(M)

            if M is None:
                print("Warning: resolve_mp expects a MasseyProduct or MPProduct.")
                return None

            new_cell = self.attach_one_cell(M)

            if new_cell is None:
                print(f"Warning: could not resolve {M}.")
                return None

            return new_cell

        # Interactive mode.
        while True:
            mp_input = input("Start typing Massey product to be resolved, or type ok to finish: ")

            if mp_input == "ok":
                break

            try:
                M = eval(mp_input, self.mp_eval_environment())
            except Exception as e:
                print("Could not understand this Massey product.")
                print("Python error:", e)
                continue

            M = self.mp_from_product(M)

            if M is None:
                print("The input must be a MasseyProduct or MPProduct, for example Q.mp(x, y) or x*y.")
                continue

            new_cell = self.attach_one_cell(M)

            if new_cell is None:
                print(f"Could not resolve {M}.")
                continue

            print(f"Resolved {M} by {new_cell}.")

    def expand_mp(self, *inputs):
        """
        Expand Q.mp(inputs) into an actual Element of Q.

        This supports nested Massey products as inputs.

        Length 1:
            mp(a) = a

        Length 2:
            mp(a,b) = expand(a) * expand(b)

        Length >= 3:
            recursive split formula using recorded primitives.
        """
        inputs = tuple(self.normalize_mp_input(x) for x in inputs)
        n = len(inputs)

        if n == 0:
            print("Warning: empty Massey product is not allowed.")
            return None

        if self.has_zero_mp_input(inputs):
            return self.zero()

        if n == 1:
            return self.expand_mp_input(inputs[0])

        if n == 2:
            left = self.expand_mp_input(inputs[0])
            right = self.expand_mp_input(inputs[1])

            if left is None or right is None:
                print(f"Warning: cannot expand Q.mp{inputs}.")
                return None

            return left * right

        result = self.zero()

        for left_length in range(1, n):
            left_inputs = inputs[:left_length]
            right_inputs = inputs[left_length:]

            left_factor = self.H_mp_block(left_inputs)
            right_factor = self.H_mp_block(right_inputs)

            if left_factor is None or right_factor is None:
                print(f"Warning: cannot expand Q.mp{inputs}.")
                print(f"Missing primitive in split {left_inputs} | {right_inputs}.")
                return None

            coeff = self.mp_split_coefficient_for_inputs(inputs, left_length)

            term = coeff * (left_factor * right_factor)
            result = result + term

        return result

    def expand_mp_input(self, x):
        """
        Expand one input of a Massey product.

        If x is a MasseyProduct, return x.expand().
        Otherwise convert x to an Element.
        """
        if isinstance(x, MasseyProduct):
            return x.expand()

        if self.is_zero_mp_input(x):
            return self.zero()

        return to_element(x)

    def H_mp_block(self, block):
        """
        H-block for Massey expansion.

        If block has length 1:
            return expansion of that input.

        If block has length >= 2:
            return the recorded primitive of Q.mp(block).
        """
        block = tuple(self.normalize_mp_input(x) for x in block)

        if len(block) == 0:
            print("Warning: empty H_mp_block is not allowed.")
            return None

        if len(block) == 1:
            return self.expand_mp_input(block[0])

        primitive = self.mp_block_primitive(*block)

        if primitive is None:
            print(f"Warning: no primitive recorded for block {block}.")
            return None

        return to_element(primitive)

    def mp_split_coefficient_for_inputs(self, inputs, left_length):
        """
        Koszul coefficient for the split inputs[:i] | inputs[i:].

        The sign depends on the whole input word, not only on the left
        factor. It is chosen recursively so every three-block refinement
        cancels in the differential of the higher Massey expression.
        """
        inputs = tuple(self.normalize_mp_input(x) for x in tuple(inputs))
        n = len(inputs)

        if n < 2 or left_length <= 0 or left_length >= n:
            return Sign.one()

        if n == 2:
            return Sign.one()

        if left_length == 1:
            first_factor = self.expand_mp_input(inputs[0])

            if first_factor is None:
                first_factor = inputs[0]

            return -self.sg(first_factor)

        tail_coeff = self.mp_split_coefficient_for_inputs(
            inputs[1:],
            left_length - 1
        )
        prefix_coeff = self.mp_split_coefficient_for_inputs(
            inputs[:left_length],
            1
        )

        return simplify_coeff(tail_coeff * prefix_coeff)

    def mp_split_coefficient(self, left_factor, right_length):
        """
        Legacy local split coefficient kept for older helper calls.

        Higher Massey expansion should use mp_split_coefficient_for_inputs,
        because arity >= 4 signs depend on the split position in the whole
        word.
        """
        if right_length == 1:
            return Sign.one()

        return -self.sg(left_factor)

    def normalize_mp_input(self, x):
        """
        If x is a length-one MasseyProduct Q.mp(a), replace it by a.
        If x is a coefficient-one monomial Element, replace it by its Path.
        Otherwise leave it unchanged.
        """
        if isinstance(x, MasseyProduct) and len(x.inputs) == 1:
            return x.inputs[0]

        if self.is_zero_mp_input(x):
            return 0

        if isinstance(x, Element) and len(x.terms) == 1:
            path, coeff = next(iter(x.terms.items()))

            if coeff == 1:
                return path

        return x

    def normalize_mp_inputs(self, inputs):
        """
        Normalize a tuple/list of Massey product inputs.
        """
        return tuple(self.normalize_mp_input(x) for x in inputs)

    def has_resolved_proper_subblock(self, block):
        """
        Return True if the given block contains a smaller consecutive
        subblock whose Massey product is already resolved.

        Example:
            block = (x, y, z)

        If (y, z) is resolved, then block is not minimal.
        """
        block = self.normalize_mp_inputs(block)
        n = len(block)

        for length in range(2, n):
            for start in range(0, n - length + 1):
                subblock = block[start:start + length]
                if self.mp_block_has_primitive(subblock):
                    return True

        return False


    def is_minimal_resolved_block(self, block):
        """
        Return True if block is resolved and contains no smaller resolved
        consecutive subblock.
        """
        block = self.normalize_mp_inputs(block)
        key = self.mp_key(*block)

        if not self.mp_block_has_primitive(block):
            return False

        return not self.has_resolved_proper_subblock(block)
        

    def find_primitives_mp(self, M, record=True, minimal_only=True):
        """
        Find primitive candidates for a MasseyProduct or MPProduct.
        """
        if isinstance(M, MPProduct):
            return self.find_primitives_mp_product(
                M,
                record=record,
                minimal_only=minimal_only
            )

        if isinstance(M, MasseyProduct):
            return self.find_primitives_mp_product(
                M.as_product(),
                record=record,
                minimal_only=minimal_only
            )

        print("Warning: find_primitives_mp expects a MasseyProduct or MPProduct.")
        print("Got:", M)
        return []

    def path_to_mp_product(self, p):
        """
        Convert an ordinary path/monomial into an MPProduct of length-one
        Massey products.

        Example:
            x1x2x3
        becomes:
            Q.mp(x1) * Q.mp(x2) * Q.mp(x3)
        """
        if isinstance(p, Arrow):
            return self.mp(p).as_product()

        if isinstance(p, Element):
            if len(p.terms) != 1:
                print("Warning: cannot convert a sum to an MPProduct.")
                return None

            p = next(iter(p.terms.keys()))

        if not isinstance(p, Path):
            print("Warning: path_to_mp_product expects an Arrow, Path, or monomial Element.")
            print("Got:", p)
            return None

        factors = []

        for arrow_name in p.arrows:
            arrow = self.arrows[arrow_name]
            factors.append(self.mp(arrow))

        return MPProduct(self, factors)

    def find_primitives_mp(self, P, record=True, minimal_only=True):
        """
        Core primitive finder.

        Accepts:
            MasseyProduct
            MPProduct
            MPElement
            ordinary Arrow/Path/monomial Element

        Convention:
            mp(a)        is flattened to a
            mp(a, b)     is flattened to a*b
            mp(a,b,c)    is treated as one irreducible factor
            mp(... n>=3) is treated as one irreducible factor

        Ordinary paths are converted to products of length-one MPs.
        """

        if not (
            isinstance(P, MasseyProduct)
            or isinstance(P, MPProduct)
            or isinstance(P, MPElement)
            or isinstance(P, Arrow)
            or isinstance(P, Path)
            or isinstance(P, Element)
        ):
            print("Warning: find_primitives_mp expects a MasseyProduct, MPProduct, MPElement, or monomial path.")
            print("Got:", P)
            return []

        if isinstance(P, MPElement):
            key = self.mp_expression_key(P)
            primitive = self.resolved_mp_expressions.get(key, None)

            if primitive is not None:
                return [{
                    "left": (),
                    "primitive": primitive,
                    "right": (),
                    "block": (P,),
                    "sign_source": (),
                }]

            if len(P.terms) == 1:
                product, coeff = next(iter(P.terms.items()))
                candidates = self.find_primitives_mp_product(
                    product,
                    record=record,
                    minimal_only=minimal_only
                )

                for candidate in candidates:
                    candidate["coefficient"] = coeff

                return candidates

            return []

        # First normalize the expression using the m_2-only rule.
        # This turns mp(x1,x2)*mp(x3,x4) into x1*x2*x3*x4,
        # but keeps mp(y1,y2,y3) as one atomic factor.
        factors = self.flatten_m2_only(P)

        if not factors:
            return []

        P = self.multiply_mp_factors(factors)

        # After normalization, P may have become an ordinary Arrow/Path/Element.
        # Convert ordinary paths to MPProduct of length-one MPs.
        if isinstance(P, Arrow) or isinstance(P, Path) or isinstance(P, Element):
            P = self.path_to_mp_product(P)

            if P is None:
                return []

        # A single higher Massey product should be converted to an MPProduct
        # containing that one irreducible factor.
        elif isinstance(P, MasseyProduct):
            P = P.as_product()

        elif isinstance(P, MPProduct):
            pass

        else:
            print("Warning: normalization produced an unsupported expression.")
            print("Got:", P)
            return []

        return self.find_primitives_mp_product(
            P,
            record=record,
            minimal_only=minimal_only
        )

    def show_mp_primitive_candidates(self):
        """
        Display recorded primitive candidates for Massey products.
        """
        if not self.mp_primitive_candidates:
            print("No Massey primitive candidates recorded.")
            return

        for key, candidates in self.mp_primitive_candidates.items():
            print(f"Primitive candidates for Q.mp{key}:")
            for c in candidates:
                print(f"    {c}")

    def format_mp_input_product(self, inputs):
        """
        Format a tuple/list of MP inputs as a product.

        Examples:
            () -> "1"
            (x, y) -> "xy"
            (Q.mp(x,y), z) -> "Q.mp(x,y)z"
        """
        inputs = tuple(inputs)

        if len(inputs) == 0:
            return "1"

        pieces = []

        for x in inputs:
            if isinstance(x, MasseyProduct) and len(x.inputs) == 1:
                pieces.append(str(x.inputs[0]))
            else:
                pieces.append(str(x))

        return "".join(pieces)


    def format_mp_primitive_candidate(self, candidate):
        """
        Format one symbolic primitive candidate found by find_primitives_mp.

        Candidate has the form:
            {
                "left": ...,
                "primitive": ...,
                "right": ...,
                "block": ...,
                "sign_source": ...,
            }

        It represents:
            (-1)^g(left) * left * primitive * right.
        """
        if candidate.get("kind") == "bridge_replacement":
            return str(candidate.get("primitive_element", candidate.get("primitive")))

        left = tuple(candidate["left"])
        primitive = candidate["primitive"]
        right = tuple(candidate["right"])

        left_str = self.format_mp_input_product(left)
        right_str = self.format_mp_input_product(right)

        if len(left) == 0:
            sign_str = ""
        else:
            sign_str = f"(-1)^g({left_str})*"

        if len(left) == 0 and len(right) == 0:
            body = f"{primitive}"
        elif len(left) == 0:
            body = f"{primitive}{right_str}"
        elif len(right) == 0:
            body = f"{left_str}{primitive}"
        else:
            body = f"{left_str}{primitive}{right_str}"

        return sign_str + body

    def show_mp_primitives(self, M, record=True):
        """
        Exhibit known primitive candidates for a MasseyProduct M.

        It shows:
            1. the direct recorded primitive, if M itself is resolved;
            2. primitive candidates coming from resolved consecutive subblocks.
        """
        if M is None:
            print("Warning: cannot show primitives of None.")
            return []

        if not isinstance(M, MasseyProduct):
            print("Warning: show_mp_primitives expects a MasseyProduct.")
            print("Got:", M)
            return []

        key = self.mp_key(*M.inputs)

        print(f"Primitive candidates for {M}:")

        found_any = False

        # Direct primitive, e.g. v[Q.mp(...)].
        if key in self.resolved_massey_products:
            primitive = self.resolved_massey_products[key]
            print("  Direct primitive:")
            print(f"    {primitive}")
            found_any = True

        # Symbolic candidates from resolved subblocks.
        candidates = self.find_primitives_mp(M, record=record)

        if candidates:
            print("  Candidates from resolved subblocks:")

            for candidate in candidates:
                candidate_str = self.format_mp_primitive_candidate(candidate)

                if candidate.get("kind") == "bridge_replacement":
                    replacements = candidate.get("bridge_replacements", [])
                    path_pieces = []

                    if replacements:
                        path_pieces.append(self.format_mp_input_product(replacements[0]["from"]))

                        for replacement in replacements:
                            path_pieces.append(self.format_mp_input_product(replacement["to"]))

                    path_str = " -> ".join(path_pieces) if path_pieces else "bridge replacement"
                    source_str = self.format_mp_input_product(candidate.get("source_product_factors", ()))
                    replacement_block_str = self.format_mp_input_product(candidate.get("replacement_block", ()))
                    primitive_block_str = self.format_mp_input_product(candidate.get("primitive_block", candidate.get("block", ())))

                    print(f"    using bridge replacement {path_str}:")
                    print(f"      replacement block {replacement_block_str} inside {source_str}")
                    print(f"      primitive block {primitive_block_str} inside {source_str}")
                    print(f"      {candidate_str}")
                    continue

                block = candidate["block"]
                block_str = self.format_mp_input_product(block)

                print(f"    using resolved block {block_str}:")
                print(f"      {candidate_str}")

            found_any = True

        if not found_any:
            print("  No primitive candidates found.")

        return candidates

    def as_mp_product(self, P):
        """
        Convert a MasseyProduct or MPProduct into an MPProduct.
        """
        if isinstance(P, MPProduct):
            return P

        if isinstance(P, MasseyProduct):
            return P.as_product()

        print("Warning: expected a MasseyProduct or MPProduct.")
        print("Got:", P)
        return None

    def mp_product_block_key(self, block):
        """
        Convert a block of MPProduct factors into the corresponding
        Massey product key.

        Example:
            (Q.mp(x1), Q.mp(x2)) -> (x1, x2)

            (Q.mp(x1,x2), Q.mp(x3)) -> (Q.mp(x1,x2), x3)
        """
        inputs = []

        for factor in block:
            if isinstance(factor, MasseyProduct) and len(factor.inputs) == 1:
                inputs.append(factor.inputs[0])
            else:
                inputs.append(factor)

        return self.mp_key(*inputs)

    def canonical_mp_factors(self, value):
        """
        Convert a product-like value into MPProduct factors after applying
        the m_2-only flattening convention.

        Ordinary arrows/paths become length-one Massey products, while
        higher Massey products remain atomic factors.
        """

        if isinstance(value, (MasseyProduct, MPProduct, Arrow, Path, Element)):
            raw_factors = self.flatten_m2_only(value)
        else:
            raw_factors = []

            for factor in tuple(value):
                raw_factors.extend(self.flatten_m2_only(factor))

        factors = []

        for factor in raw_factors:
            if isinstance(factor, MPProduct):
                factors.extend(self.canonical_mp_factors(factor.factors))
            elif isinstance(factor, MasseyProduct):
                factors.append(factor)
            else:
                factors.append(self.mp(factor))

        return tuple(factors)

    def mp_factors_equal(self, left, right):
        """
        Compare two MP factor lists using the same stable keys as MPProduct.
        """

        left = tuple(left)
        right = tuple(right)

        if len(left) != len(right):
            return False

        return all(
            self.mp_factor_key(a) == self.mp_factor_key(b)
            for a, b in zip(left, right)
        )

    def mp_factors_to_element(self, factors):
        """
        Expand a nonempty tuple of MP factors into an actual DG element.
        """

        factors = tuple(factors)

        if not factors:
            return None

        return self.expand_mp_product(MPProduct(self, factors))

    def multiply_element_on_right_by_mp_factors(self, element, factors):
        """
        Return element * product(factors), with the empty product omitted.
        """

        result = to_element(element)
        factors = tuple(factors)

        if not factors:
            return result

        right = self.mp_factors_to_element(factors)

        if right is None:
            return None

        return result * right

    def multiply_element_on_left_by_mp_factors(self, factors, element, leibniz_sign=False):
        """
        Return product(factors) * element, with optional Koszul sign.

        The optional sign is the one needed in d(sg(h) * h * b) = h*d(b)
        when h is a cycle.
        """

        result = to_element(element)
        factors = tuple(factors)

        if not factors:
            return result

        left = self.mp_factors_to_element(factors)

        if left is None:
            return None

        result = left * result

        if leibniz_sign:
            result = self.sg(left) * result

        return result

    def primitive_candidate_to_element(self, candidate):
        """
        Convert a symbolic primitive candidate into an actual Element.
        """

        if "primitive_element" in candidate:
            return candidate["primitive_element"]

        primitive = to_element(candidate["primitive"])
        left_factors = tuple(candidate.get("left", ()))
        right_factors = tuple(candidate.get("right", ()))

        if left_factors:
            left = self.mp_factors_to_element(left_factors)

            if left is None:
                return None

            primitive = self.sg(left) * (left * primitive)

        if right_factors:
            right = self.mp_factors_to_element(right_factors)

            if right is None:
                return None

            primitive = primitive * right

        coeff = candidate.get("coefficient", 1)

        if coeff != 1:
            primitive = coeff * primitive

        return primitive

    def primitive_candidate_crosses_boundary(self, candidate, boundary):
        """
        Return True if candidate's resolved block crosses the given split.
        """

        start = len(tuple(candidate.get("left", ())))
        end = start + len(tuple(candidate.get("block", ())))

        return start < boundary < end

    def bridge_replacement_primitive_candidates(self, P, record=True, minimal_only=True):
        """
        Detect primitives created by replacing across bridge relations.

        Bridge replacements are searched as non-looping paths in the whole
        product-state graph generated by bridge terms. Thus replacements may
        occur at different, overlapping, or newly-created positions along one
        chain.
        """

        if isinstance(P, MasseyProduct):
            P = P.as_product()

        if not isinstance(P, MPProduct):
            return []

        self.ensure_bridge_history()

        factors = self.canonical_mp_factors(P)

        if len(factors) < 2:
            return []

        target_element = self.mp_factors_to_element(factors)

        if target_element is None:
            return []

        def factors_key(some_factors):
            return tuple(self.mp_factor_key(factor) for factor in tuple(some_factors))

        edges = self.replacement_edge_records()

        if not edges:
            return []

        candidates = []

        def bridge_part(edge, left_factors, right_factors):
            part = to_element(edge["bridge_cell"])

            if left_factors:
                left = self.mp_factors_to_element(left_factors)

                if left is None:
                    return None

                part = self.sg(left) * (left * part)

            if right_factors:
                right = self.mp_factors_to_element(right_factors)

                if right is None:
                    return None

                part = part * right

            return part

        def extend_bridge_sum(accumulated_bridge, accumulated_coeff, edge, left_factors, right_factors):
            part = bridge_part(edge, left_factors, right_factors)

            if part is None:
                return None, None

            bridge_scalar = simplify_coeff(accumulated_coeff * edge["bridge_coeff"])
            next_bridge = accumulated_bridge + bridge_scalar * part
            next_coeff = simplify_coeff(accumulated_coeff * edge["source_coeff"])

            return next_bridge, next_coeff

        def replacement_side_for_span(start, stop, state_len):
            if start == 0:
                return "right"

            if stop == state_len:
                return "left"

            return "middle"

        def path_profile_contexts(path_records, primitive_span):
            contexts = []

            for prefix_len in range(0, len(path_records) + 1):
                if prefix_len == 0:
                    context = "original"
                    context_factors = factors
                    context_key = ("original", self.mp_factor_tuple_key(context_factors))
                else:
                    context = "bridge_prefix"
                    context_factors = tuple(path_records[prefix_len - 1]["source_product_after"])
                    prefix_key = tuple(
                        self.bridge_edge_key(edge)
                        for edge in path_records[:prefix_len]
                    )
                    context_key = (
                        "bridge_prefix",
                        self.mp_factor_tuple_key(context_factors),
                        prefix_key
                    )

                touched = set()
                context_origins = list(range(len(context_factors)))

                for record in path_records[prefix_len:]:
                    start, stop = record["occurrence_span_before"]
                    touched.update(
                        origin
                        for origin in context_origins[start:stop]
                        if origin is not None
                    )
                    context_origins = (
                        context_origins[:start]
                        + [None] * len(record["replacement_factors"])
                        + context_origins[stop:]
                    )

                if primitive_span is not None:
                    for i in range(primitive_span[0], primitive_span[1]):
                        if 0 <= i < len(context_origins):
                            touched.add(context_origins[i])

                span = self.span_from_touched_positions(touched)

                if span is None:
                    continue

                contexts.append({
                    "context": context,
                    "context_factors": context_factors,
                    "context_key": context_key,
                    "span": span,
                    "block": tuple(context_factors[span[0]:span[1]]),
                    "prefix_length": prefix_len,
                })

            return contexts

        def search_replacement_state(current_factors, current_origins, accumulated_bridge, accumulated_coeff, path_records):
            source_product = MPProduct(self, current_factors)
            source_candidates = self.find_primitives_mp_product(
                source_product,
                record=False,
                minimal_only=minimal_only,
                use_bridge_replacements=False
            )

            for source_candidate in source_candidates:
                source_primitive = self.primitive_candidate_to_element(source_candidate)

                if source_primitive is None:
                    continue

                primitive_element = accumulated_bridge + accumulated_coeff * source_primitive

                if not (self.d(primitive_element) - target_element).is_zero():
                    continue

                source_block = tuple(source_candidate.get("block", ()))
                source_left = tuple(source_candidate.get("left", ()))
                source_right = tuple(source_candidate.get("right", ()))
                source_block_start = len(source_left)
                source_block_end = source_block_start + len(source_block)
                primitive_span = (source_block_start, source_block_end)
                profile_contexts = path_profile_contexts(path_records, primitive_span)

                if not profile_contexts:
                    continue

                bridge_replacements = [
                    {
                        "from": record["target_factors"],
                        "to": record["replacement_factors"],
                        "bridge": record["bridge_cell"],
                        "bridge_expression": record["bridge_expression"],
                        "occurrence_span": record["occurrence_span_before"],
                        "source_before": record["source_product_before"],
                        "source_after": record["source_product_after"],
                    }
                    for record in path_records
                ]
                first_record = path_records[0]
                last_record = path_records[-1]
                replacement_record = {
                    "original_product_factors": factors,
                    "original_replaced_block": first_record["target_factors"],
                    "original_replaced_span": first_record["occurrence_span_before"],
                    "first_step_source_product_factors": first_record["source_product_after"],
                    "first_step_replacement_span": first_record["replacement_span_after"],
                    "source_product_factors": current_factors,
                    "replacement_block": last_record["replacement_factors"],
                    "replacement_span": last_record["replacement_span_after"],
                    "primitive_block": source_block,
                    "primitive_block_span": primitive_span,
                    "primitive_left": source_left,
                    "primitive_right": source_right,
                    "bridge_replacements": bridge_replacements,
                    "profile_contexts": profile_contexts,
                }

                candidates.append({
                    "kind": "bridge_replacement",
                    "side": replacement_side_for_span(
                        first_record["occurrence_span_before"][0],
                        first_record["occurrence_span_before"][1],
                        len(factors)
                    ),
                    "left": (),
                    "primitive": primitive_element,
                    "primitive_element": primitive_element,
                    "right": (),
                    "block": source_block,
                    "original_product_factors": factors,
                    "original_replaced_block": first_record["target_factors"],
                    "original_replaced_span": first_record["occurrence_span_before"],
                    "first_step_source_product_factors": first_record["source_product_after"],
                    "first_step_replacement_span": first_record["replacement_span_after"],
                    "source_product_factors": current_factors,
                    "replacement_block": last_record["replacement_factors"],
                    "replacement_span": last_record["replacement_span_after"],
                    "primitive_block": source_block,
                    "primitive_block_span": primitive_span,
                    "primitive_left": source_left,
                    "primitive_right": source_right,
                    "bridge_replacements": bridge_replacements,
                    "bridge_left_factors": tuple(factors[:first_record["occurrence_span_before"][0]]),
                    "bridge_right_factors": tuple(factors[first_record["occurrence_span_before"][1]:]),
                    "replacement_record": replacement_record,
                    "sign_source": (),
                    "bridge": first_record["bridge_cell"],
                    "bridge_expression": first_record["bridge_expression"],
                    "bridge_path": list(path_records),
                    "target_term": first_record["target_product"],
                    "replacement_term": last_record["replacement_product"],
                    "replacement_factors": current_factors,
                    "source_primitive": source_candidate,
                    "profile_contexts": profile_contexts,
                })

        max_depth = max(1, len(edges) * len(factors))
        queue = [(
            factors,
            list(range(len(factors))),
            self.zero(),
            1,
            [],
            {factors_key(factors)}
        )]

        while queue:
            current_factors, current_origins, accumulated_bridge, accumulated_coeff, path_records, seen_state_keys = queue.pop(0)

            if path_records:
                search_replacement_state(
                    current_factors,
                    current_origins,
                    accumulated_bridge,
                    accumulated_coeff,
                    path_records
                )

            if len(path_records) >= max_depth:
                continue

            state_len = len(current_factors)

            for edge in edges:
                target_len = len(edge["target_factors"])

                if state_len < target_len:
                    continue

                for start in range(0, state_len - target_len + 1):
                    stop = start + target_len

                    if not self.mp_factors_equal(current_factors[start:stop], edge["target_factors"]):
                        continue

                    next_factors = (
                        current_factors[:start]
                        + edge["replacement_factors"]
                        + current_factors[stop:]
                    )
                    next_key = factors_key(next_factors)

                    if next_key in seen_state_keys:
                        continue

                    next_bridge, next_coeff = extend_bridge_sum(
                        accumulated_bridge,
                        accumulated_coeff,
                        edge,
                        current_factors[:start],
                        current_factors[stop:]
                    )

                    if next_bridge is None:
                        continue

                    next_origins = (
                        current_origins[:start]
                        + [None] * len(edge["replacement_factors"])
                        + current_origins[stop:]
                    )
                    path_record = dict(edge)
                    path_record.update({
                        "from": edge["target_factors"],
                        "to": edge["replacement_factors"],
                        "occurrence_span_before": (start, stop),
                        "replacement_span_after": (start, start + len(edge["replacement_factors"])),
                        "source_product_before": current_factors,
                        "source_product_after": next_factors,
                        "origins_before": tuple(current_origins),
                        "origins_after": tuple(next_origins),
                    })
                    queue.append((
                        next_factors,
                        next_origins,
                        next_bridge,
                        next_coeff,
                        path_records + [path_record],
                        seen_state_keys | {next_key}
                    ))

        out = []
        seen = set()

        for candidate in candidates:
            primitive = candidate.get("primitive_element", candidate.get("primitive"))
            primitive_key = primitive.key() if hasattr(primitive, "key") else repr(primitive)
            path_key = tuple(
                (
                    record.get("occurrence_span_before"),
                    record.get("target_key"),
                    record.get("replacement_key"),
                )
                for record in candidate.get("bridge_path", [])
            )
            key = (
                primitive_key,
                path_key,
                candidate.get("primitive_block_span", None),
            )

            if key not in seen:
                seen.add(key)
                out.append(candidate)

        return out

    def find_primitives_mp_product(self, P, record=True, minimal_only=True, use_bridge_replacements=True):
        """
        Find primitive candidates in a product of Massey products.

        Example:
            P = Q.mp(x1) * Q.mp(x2) * Q.mp(x3)

        If Q.mp(x1,x2) is resolved, this finds the candidate

            v[Q.mp(x1,x2)] * Q.mp(x3)

        symbolically, without expanding.
        """
        if isinstance(P, MasseyProduct):
            P = P.as_product()

        if not isinstance(P, MPProduct):
            print("Warning: expected a MasseyProduct or MPProduct.")
            print("Got:", P)
            return []

        factors = P.factors
        n = len(factors)

        candidates = []

        for i in range(n):
            for j in range(i + 1, n + 1):
                block = factors[i:j]

                if len(block) == 1:
                    factor = block[0]

                    if not isinstance(factor, MasseyProduct):
                        continue

                    if len(factor.inputs) == 1:
                        continue

                    block_key = self.mp_key(*factor.inputs)
                else:
                    block_key = self.mp_product_block_key(block)

                primitive = self.mp_block_primitive(
                    *block_key,
                    allow_bridge_replacements=False
                )

                if primitive is None:
                    continue

                if minimal_only and len(block) > 1:
                    if self.has_resolved_proper_mp_product_subblock(
                        block,
                        allow_bridge_replacements=use_bridge_replacements
                    ):
                        continue

                candidate = {
                    "left": factors[:i],
                    "primitive": primitive,
                    "right": factors[j:],
                    "block": block,
                    "sign_source": factors[:i],
                }

                candidates.append(candidate)

        if use_bridge_replacements:
            candidates.extend(
                self.bridge_replacement_primitive_candidates(
                    P,
                    record=False,
                    minimal_only=minimal_only
                )
            )

        if record:
            key = tuple(factors)
            self.mp_primitive_candidates[key] = candidates

        return candidates

    def has_resolved_proper_mp_product_subblock(self, block, allow_bridge_replacements=True):
        """
        Return True if a block of MPProduct factors contains a smaller
        resolved consecutive subblock.
        """
        block = tuple(block)
        n = len(block)

        for i in range(n):
            for j in range(i + 1, n + 1):
                if i == 0 and j == n:
                    continue

                subblock = block[i:j]

                if len(subblock) == 1:
                    factor = subblock[0]

                    if not isinstance(factor, MasseyProduct):
                        continue

                    if len(factor.inputs) == 1:
                        continue

                    subkey = self.mp_key(*factor.inputs)
                else:
                    subkey = self.mp_product_block_key(subblock)

                if self.mp_block_has_primitive(
                    subkey,
                    allow_bridge_replacements=allow_bridge_replacements
                ):
                    return True

        return False

    def has_resolved_proper_factor_subblock(self, factors):
        """
        Return True if a product block of MP factors contains a smaller
        resolved consecutive subblock.
        """
        factors = tuple(factors)
        n = len(factors)

        # A length-one factor that is itself resolved counts as a smaller
        # resolved subblock.
        for f in factors:
            if isinstance(f, MasseyProduct):
                if len(f.inputs) == 1:
                    continue

                if self.mp_block_has_primitive(f.inputs):
                    return True

        # Longer proper consecutive factor subblocks.
        for length in range(2, n):
            for start in range(0, n - length + 1):
                subblock = factors[start:start + length]
                subkey = self.mp_product_block_key(subblock)

                if self.mp_block_has_primitive(subkey):
                    return True

        return False

    def flatten_m2_only(self, P):
        """
        Flatten an MP expression/product into a list of factors.

        Rules:
            mp(a)        -> a
            mp(a, b)     -> a, b
            mp(a,b,c)    -> atomic factor
            mp(... n>=3) -> atomic factor

        This ensures that m_2 is treated as ordinary multiplication,
        while m_3 and higher are treated as irreducible blocks.
        """

        # Product of MP factors
        if isinstance(P, MPProduct):
            out = []
            for factor in P.factors:
                out.extend(self.flatten_m2_only(factor))
            return out

        # Single Massey product
        if isinstance(P, MasseyProduct):
            if len(P.inputs) == 1:
                return self.flatten_m2_only(P.inputs[0])

            if len(P.inputs) == 2:
                out = []
                out.extend(self.flatten_m2_only(P.inputs[0]))
                out.extend(self.flatten_m2_only(P.inputs[1]))
                return out

            # Higher Massey products are irreducible
            return [P]

        # Ordinary path: split into actual arrows/cells if possible
        if isinstance(P, Path):
            return [
                self.arrows[name] if name in self.arrows else name
                for name in P.arrows
            ]

        # Ordinary element with one monomial term. Coefficients do not
        # change the MP factor pattern, so 2*x*y is flattened like x*y.
        if isinstance(P, Element):
            if len(P.terms) != 1:
                return [P]

            path, coeff = next(iter(P.terms.items()))

            if is_zero_coeff(coeff):
                return []

            return self.flatten_m2_only(path)

        # Fallback: atomic factor
        return [P]


    def multiply_mp_factors(self, factors):
        """
        Multiply a list of factors back into an expression.

        Higher MP factors remain atomic.
        """

        if not factors:
            return None

        if any(isinstance(f, (MasseyProduct, MPProduct)) for f in factors):
            product_factors = []

            for f in factors:
                if isinstance(f, MPProduct):
                    product_factors.extend(f.factors)
                elif isinstance(f, MasseyProduct):
                    product_factors.append(f)
                else:
                    product_factors.append(self.mp(f))

            return MPProduct(self, product_factors)

        result = factors[0]

        for f in factors[1:]:
            result = result * f

        return result


    def consecutive_mp_subblocks(self, factors, min_len=2):
        """
        Return all consecutive subblocks of a flattened MP factor list.

        Returns triples:
            (left, block, right)

        where:
            factors = left + block + right
        """

        blocks = []
        n = len(factors)

        for i in range(n):
            for j in range(i + min_len, n + 1):
                left = factors[:i]
                block = factors[i:j]
                right = factors[j:]
                blocks.append((left, block, right))

        return blocks

    def find_mp_primitives(self, P=None, record=True, minimal_only=True):
        """
        User-facing primitive finder.

        Usage 1:
            Q.find_mp_primitives()

        Then type:
            x1*x2*x3

        Usage 2:
            Q.find_mp_primitives("x1*x2*x3")

        Usage 3:
            Q.find_mp_primitives(x1*x2*x3)

        Usage 4:
            Q.find_mp_primitives(Q.mp(x1)*Q.mp(x2)*Q.mp(x3))
        """
        if P is not None:
            if isinstance(P, str):
                try:
                    P = eval(P, self.mp_eval_environment())
                except Exception as e:
                    print("Could not understand this MP expression.")
                    print("Python error:", e)
                    return []

            return self.find_primitives_mp(
                P,
                record=record,
                minimal_only=minimal_only
            )

        while True:
            expr = input("Type MP product to search primitives, or type ok to finish: ")

            if expr == "ok":
                break

            try:
                P = eval(expr, self.mp_eval_environment())
            except Exception as e:
                print("Could not understand this MP expression.")
                print("Python error:", e)
                continue

            candidates = self.find_primitives_mp(
                P,
                record=record,
                minimal_only=minimal_only
            )

            if not candidates:
                print("No primitive candidates found.")
            else:
                print(f"Primitive candidates for {P}:")
                for c in candidates:
                    print(c)

    def mp_from_product(self, obj):
        """
        Convert a MasseyProduct or MPProduct into a MasseyProduct.

        Examples:
            Q.mp(x1, x2) stays Q.mp(x1, x2)

            Q.mp(x1) * Q.mp(x2) becomes Q.mp(x1, x2)

            Q.mp(x1,x2) * Q.mp(x3) becomes Q.mp(Q.mp(x1,x2), x3)
        """
        if isinstance(obj, MasseyProduct):
            return obj

        if isinstance(obj, MPProduct):
            inputs = []

            for factor in obj.factors:
                if isinstance(factor, MasseyProduct) and len(factor.inputs) == 1:
                    inputs.append(factor.inputs[0])
                else:
                    inputs.append(factor)

            return self.mp(*inputs)

        return None


    def mp_dependencies_of_key(self, key):
        """
        Return the immediate MP dependencies of key.

        Example:
            key = (x1, x2, x3)

        depends on:
            (x1, x2), (x2, x3)

        because Q.mp(x1,x2,x3) exists only if those lower MPs are resolved.
        """
        key = self.normalize_mp_inputs(key)

        deps = set()

        for subkey in self.required_lower_mps(*key):
            deps.add(self.normalize_mp_inputs(subkey))

        for x in key:
            if isinstance(x, MasseyProduct):
                deps.add(self.mp_key(*x.inputs))

        return deps


    def mp_depends_on_key(self, key, target_key, seen=None):
        """
        Return True if Q.mp(key) depends recursively on Q.mp(target_key).
        """
        key = self.normalize_mp_inputs(key)
        target_key = self.normalize_mp_inputs(target_key)

        if seen is None:
            seen = set()

        if key in seen:
            return False

        seen.add(key)

        deps = self.mp_dependencies_of_key(key)

        if target_key in deps:
            return True

        for dep in deps:
            if self.mp_depends_on_key(dep, target_key, seen=seen):
                return True

        return False


    def dependent_mps(self, key):
        """
        Return all existing/resolved MP keys depending on key.

        This includes key itself if it exists or is resolved.

        We look in both:
            self.massey_products
            self.resolved_massey_products

        because deleting v[Q.mp(1,2)] should also delete the existence of
        Q.mp(1,2,3), not only its primitive.
        """
        key = self.normalize_mp_inputs(key)

        all_keys = set(self.massey_products.keys()) | set(self.resolved_massey_products.keys())

        dependents = []

        for other_key in all_keys:
            other_key = self.normalize_mp_inputs(other_key)

            if other_key == key or self.mp_depends_on_key(other_key, key):
                dependents.append(other_key)

        dependents.sort(key=len, reverse=True)

        return dependents

    def cleanup_invalid_generated_mps(self):
        """
        Remove generated Massey products which are no longer definable.

        This does not delete genuine resolving cells. It only cleans the
        auxiliary list self.generated_massey_products after rollback.
        """

        if not hasattr(self, "generated_massey_products"):
            return []

        removed = []
        kept = []

        for M in self.generated_massey_products:
            if not isinstance(M, MasseyProduct):
                kept.append(M)
                continue

            inputs = self.normalize_mp_inputs(M.inputs)

            try:
                still_defined = self.can_define_mp(*inputs)
            except Exception:
                still_defined = False

            if still_defined:
                kept.append(M)
            else:
                removed.append(M)

        self.generated_massey_products = kept

        return removed


    def undo_latest_mp_cell(self, verbose=True):
        """
        Undo the latest MP resolving cell.

        This is stack-like deletion: only the most recently added resolving
        cell can be removed.

        This deletes the resolving cell/primitive record, not the underlying
        Massey product expression.

        Example:
            If the resolving cells were added in order:

                v[Q.mp(x,y)]
                v[Q.mp(y,z)]
                v[Q.mp(z,x)]

            then Q.undo_latest_mp_cell() removes only:

                v[Q.mp(z,x)]
        """

        self.ensure_mp_cell_history()

        if not self.mp_cell_history:
            if verbose:
                print("No MP resolving cells to undo.")
            return None

        key = self.mp_cell_history.pop()

        primitive = None

        if hasattr(self, "resolved_massey_products"):
            primitive = self.resolved_massey_products.pop(key, None)
        removed_attachment_records = self.forget_mp_cell_attachment_record(key)

        if primitive is not None:
            expr = primitive.index if hasattr(primitive, "index") else None
            removed_attachment_records.extend(
                self.remove_cell_family_record("v", expr, cell=primitive)
            )
            removed_attachment_records.extend(
                self.remove_attached_cell_arrow(primitive)
            )

        self.forget_attachment_history_entry("mp_cell", key, primitive)

        if verbose:
            print("Undid latest resolving cell:")
            print("   ", primitive)
            print("for:")
            print("   ", self.mp(*key))

        if hasattr(self, "cleanup_invalid_generated_mps"):
            removed_stale_mps = self.cleanup_invalid_generated_mps()
        else:
            removed_stale_mps = []

        if verbose and removed_stale_mps:
            print("Removed stale generated MP records:")
            for M in removed_stale_mps:
                print("   ", M)

        return {
            "undone": (key, primitive),
            "removed_stale_mps": removed_stale_mps,
            "remaining_history": list(self.mp_cell_history),
        }

    def ensure_bridge_history(self):
        """
        Make sure bridge bookkeeping exists.
        """

        if not hasattr(self, "bridge_history"):
            self.bridge_history = []

        if not hasattr(self, "bridge_expressions"):
            self.bridge_expressions = {}

        if not hasattr(self, "bridge_cells"):
            self.bridge_cells = {}

    def remove_attached_cell_arrow(self, cell):
        """
        Remove an attached cell arrow from arrows and differential records.
        """

        removed = []

        if not isinstance(cell, Arrow):
            return removed

        if cell in self.differential:
            removed.append(("differential", self.differential.pop(cell)))

        name = cell.name if hasattr(cell, "name") else None

        if name in self.arrows and self.arrows[name] is cell:
            removed.append(("arrows", self.arrows.pop(name)))

        return removed

    def remove_cell_family_record(self, family_name, expression=None, key=None, cell=None):
        """
        Remove a cell from a CellFamily lookup table.
        """

        removed = []
        family = getattr(self, family_name, None)

        if family is None or not hasattr(family, "cells"):
            return removed

        candidate_keys = []

        if key is not None:
            candidate_keys.append(key)

        if expression is not None:
            candidate_keys.append(family.key(expression))
            candidate_keys.append(repr(expression))

        for candidate_key in candidate_keys:
            if candidate_key in family.cells:
                value = family.cells[candidate_key]

                if cell is None or value is cell:
                    removed.append((family_name, family.cells.pop(candidate_key)))

        if cell is not None:
            for candidate_key, value in list(family.cells.items()):
                if value is cell:
                    removed.append((family_name, family.cells.pop(candidate_key)))

        return removed

    def forget_bridge_cell_attachment_record(self, key):
        """
        Remove lookup records for a bridge cell.
        """

        self.ensure_bridge_history()
        removed = []

        expr = self.bridge_expressions.pop(key, None)
        cell = self.bridge_cells.pop(key, None)

        if expr is not None:
            removed.append(("bridge_expressions", expr))

        if cell is not None:
            removed.append(("bridge_cells", cell))

        if hasattr(self, "v") and hasattr(self.v, "cells"):
            if key in self.v.cells:
                removed.append(("v", self.v.cells.pop(key)))

            if expr is not None:
                expr_key = self.v.key(expr)

                if expr_key in self.v.cells:
                    removed.append(("v", self.v.cells.pop(expr_key)))

            if cell is not None:
                removed.extend(self.remove_cell_family_record("v", expr, cell=cell))

        if cell is not None:
            removed.extend(self.remove_attached_cell_arrow(cell))

        self.forget_attachment_history_entry("bridge", key, cell)

        return removed

    def undo_latest_bridge_cell(self, verbose=True):
        """
        Undo the most recently attached bridge cell v[g1 - g2].
        """

        self.ensure_bridge_history()

        if not self.bridge_history:
            if verbose:
                print("No bridge cells to undo.")
            return None

        key = self.bridge_history.pop()
        expr = self.bridge_expressions.get(key, None)
        primitive = self.resolved_mp_expressions.pop(key, None)
        removed_attachment_records = self.forget_bridge_cell_attachment_record(key)

        if verbose:
            print("Undid latest bridge cell:")
            print("   ", primitive)
            print("for:")
            print("   ", expr if expr is not None else key)

        return {
            "undone": (key, primitive),
            "expression": expr,
            "removed_attachment_records": removed_attachment_records,
            "remaining_bridge_history": list(self.bridge_history),
        }

    def undo_latest_bridge(self, verbose=True):
        """
        Alias for undo_latest_bridge_cell.
        """

        return self.undo_latest_bridge_cell(verbose=verbose)

    def show_bridge_history(self):
        """
        Print bridge cells in the order they were added.
        """

        self.ensure_bridge_history()

        if not self.bridge_history:
            print("No bridge cells in history.")
            return []

        for i, key in enumerate(self.bridge_history):
            expr = self.bridge_expressions.get(key, key)
            primitive = self.resolved_mp_expressions.get(key, None)
            print(f"{i}: {primitive} bridges {expr}")

        return list(self.bridge_history)

    def undo_mp_expression_cell_by_key(self, key, entry=None, verbose=True):
        """
        Undo a non-bridge general MP-expression cell, such as v[P].
        """

        if entry is None:
            entry = {}

        expr = entry.get("expression", None)
        primitive = self.resolved_mp_expressions.pop(key, None)

        if primitive is None:
            primitive = entry.get("cell", None)

        if expr is None and primitive is not None and hasattr(primitive, "index"):
            expr = primitive.index

        removed_attachment_records = []

        if primitive is not None:
            removed_attachment_records.extend(
                self.remove_cell_family_record("v", expr, key=key, cell=primitive)
            )
            removed_attachment_records.extend(
                self.remove_attached_cell_arrow(primitive)
            )

        self.forget_attachment_history_entry("mp_expression_cell", key, primitive)

        if verbose:
            print("Undid latest MP-expression cell:")
            print("   ", primitive)
            print("for:")
            print("   ", expr if expr is not None else key)

        return {
            "undone": (key, primitive),
            "expression": expr,
            "removed_attachment_records": removed_attachment_records,
            "remaining_attachment_history": list(getattr(self, "attachment_history", [])),
        }

    def undo_ordinary_cell_from_entry(self, entry, verbose=True):
        """
        Undo an ordinary u[f] cell from a universal-history entry.
        """

        key = entry.get("key", None)
        expr = entry.get("expression", None)
        cell = entry.get("cell", None)
        family = entry.get("family", "u")

        removed_attachment_records = []
        removed_attachment_records.extend(
            self.remove_cell_family_record(family, expr, key=key, cell=cell)
        )

        if hasattr(self, "cells") and cell is not None:
            for cell_key, value in list(self.cells.items()):
                if value is cell:
                    removed_attachment_records.append(("cells", self.cells.pop(cell_key)))

        if cell is not None:
            removed_attachment_records.extend(self.remove_attached_cell_arrow(cell))

        self.forget_attachment_history_entry("ordinary_cell", key, cell)

        if verbose:
            print("Undid latest ordinary cell:")
            print("   ", cell)
            print("for:")
            print("   ", expr if expr is not None else key)

        return {
            "undone": (key, cell),
            "expression": expr,
            "removed_attachment_records": removed_attachment_records,
            "remaining_attachment_history": list(getattr(self, "attachment_history", [])),
        }

    def undo_latest_attachment(self, verbose=True):
        """
        Undo the most recently attached cell of any tracked kind.

        This covers ordinary cells, resolved MP cells, general MP-expression
        cells, and bridge cells.
        """

        self.ensure_attachment_history()

        if not self.attachment_history:
            if verbose:
                print("No attached cells to undo.")
            return None

        entry = self.attachment_history.pop()
        kind = entry.get("kind")
        key = entry.get("key")

        if kind == "bridge":
            return self.undo_latest_bridge_cell(verbose=verbose)

        if kind == "mp_cell":
            return self.undo_latest_mp_cell(verbose=verbose)

        if kind == "mp_expression_cell":
            return self.undo_mp_expression_cell_by_key(
                key,
                entry=entry,
                verbose=verbose
            )

        if kind == "ordinary_cell":
            return self.undo_ordinary_cell_from_entry(
                entry,
                verbose=verbose
            )

        if verbose:
            print("Unknown attachment kind:", kind)

        return {
            "undone": None,
            "entry": entry,
            "remaining_attachment_history": list(self.attachment_history),
        }

    def undo_latest_cell(self, verbose=True):
        """
        Alias for undo_latest_attachment.
        """

        return self.undo_latest_attachment(verbose=verbose)

    def show_mp_cell_history(self):
        """
        Print MP resolving cells in the order they were added.
        """

        self.ensure_mp_cell_history()

        if not self.mp_cell_history:
            print("No MP resolving cells in history.")
            return []

        for i, key in enumerate(self.mp_cell_history):
            primitive = None

            if hasattr(self, "resolved_massey_products"):
                primitive = self.resolved_massey_products.get(key, None)

            print(f"{i}: {primitive} resolves {self.mp(*key)}")

        return list(self.mp_cell_history)

    "Now we define generating function"

    def mp_extension_candidates(self):
        """
        Return possible outside factors q used to extend Massey products.

        Important:
            Do NOT include primitives v[...].
            Do NOT include raw tuple keys like (x,x).
            Do NOT include resolved_massey_products values.

        Otherwise generate_massey_products will try meaningless things like:
            mp(v[Q.mp(x,x)], x, x)
            mp((x,x), x, x)
        """

        candidates = []

        # Arrows
        if hasattr(self, "arrows"):
            if isinstance(self.arrows, dict):
                candidates.extend(self.arrows.values())
            else:
                candidates.extend(self.arrows)

        # Ordinary attached cells, if you want cells to be possible inputs
        if hasattr(self, "cells"):
            if isinstance(self.cells, dict):
                candidates.extend(self.cells.values())
            else:
                candidates.extend(self.cells)

        # Do not feed newly generated Massey products back in as outside
        # factors here. The current generation step is only the direct
        # 3-fold MPProduct detection; higher generated-input cases need a
        # separate defining-system rule.

        # Remove duplicates while preserving order
        out = []
        seen = set()

        for x in candidates:
            # Suppress tuple keys and obvious primitive/cell wrappers if needed
            if isinstance(x, tuple):
                continue

            k = repr(x)

            if k not in seen:
                seen.add(k)
                out.append(x)

        return out
        
    def try_generate_mp(self, *inputs, record=True, verbose=False):
        """
        Try to generate/register the Massey product mp(inputs).

        Returns:
            M       if mp(inputs) can be defined;
            None    otherwise.
        """

        inputs = self.normalize_mp_inputs(inputs)

        if not self.can_define_mp(*inputs):
            if verbose:
                print("Cannot define MP:")
                print("inputs =", inputs)
            return None

        try:
            M = self.mp(*inputs)
        except Exception as e:
            if verbose:
                print("Could not generate MP:")
                print("inputs =", inputs)
                print("Python error:", e)
            return None

        if record:
            if not hasattr(self, "generated_massey_products"):
                self.generated_massey_products = []

            key = self.mp_key(*inputs)

            old_keys = {
                self.mp_key(*self.normalize_mp_inputs(x.inputs))
                for x in self.generated_massey_products
                if isinstance(x, MasseyProduct)
            }

            if key not in old_keys:
                self.generated_massey_products.append(M)

        return M

    def primitive_uses_nontrivial_block(self, primitive_data, outside_factor):
        """
        Decide whether a primitive candidate genuinely involves the new product,
        rather than merely coming from the outside factor '?' alone.

        Expected primitive_data shape:
            {
                "left": ...,
                "primitive": ...,
                "right": ...,
                "block": ...,
                ...
            }

        We reject the case where block == (outside_factor,).
        """

        if not isinstance(primitive_data, dict):
            return True

        block = primitive_data.get("block", None)

        if block is None:
            return True

        block = tuple(block)

        if len(block) == 1 and repr(block[0]) == repr(outside_factor):
            return False

        return True

    def massey_product_input_product(self, M):
        """
        The monomial/product of inputs that produces a Massey product M.

        For Q.mp(x1,x2,x3), this is Q.mp(x1)*Q.mp(x2)*Q.mp(x3).
        This is the monomial whose primitive blocks control whether M has
        an obvious primitive.
        """

        if not isinstance(M, MasseyProduct):
            return None

        factors = self.canonical_mp_factors(self.normalize_mp_inputs(M.inputs))

        if not factors:
            return None

        return MPProduct(self, factors)

    def mp_factor_tuple_key(self, factors):
        """
        Stable key for a tuple of MP factors.
        """

        return tuple(self.mp_factor_key(factor) for factor in tuple(factors))

    def spans_are_disjoint(self, first, second):
        """
        Return True if two half-open spans do not intersect.
        """

        return first[1] <= second[0] or second[1] <= first[0]

    def span_from_touched_positions(self, positions):
        """
        Convert a set of touched positions to its maximal containing span.
        """

        positions = [p for p in positions if p is not None]

        if not positions:
            return None

        return (min(positions), max(positions) + 1)

    def bridge_edge_key(self, edge):
        """
        Stable-enough key for one bridge edge in a replacement path.
        """

        return (
            edge.get("replacement_type", "bridge"),
            edge.get("bridge_key", None),
            edge.get("ainf_inputs", None),
            id(edge.get("bridge_cell", None)),
            edge.get("target_key"),
            edge.get("replacement_key"),
        )

    def bridge_candidate_profile_after_prefix(self, candidate, prefix_len):
        """
        Profile a bridge primitive after reducing by a bridge-prefix path.

        The profile records the context monomial and the maximal block in
        that context touched by the remaining replacements and final
        primitive-producing block.
        """

        path_edges = list(candidate.get("bridge_path", []))

        if not path_edges or prefix_len < 1 or prefix_len > len(path_edges):
            return None

        left_factors = tuple(candidate.get("bridge_left_factors", ()))
        right_factors = tuple(candidate.get("bridge_right_factors", ()))
        prefix_block = tuple(path_edges[prefix_len - 1]["replacement_factors"])
        context_factors = left_factors + prefix_block + right_factors
        prefix_start = len(left_factors)

        if prefix_len == len(path_edges):
            span = candidate.get("primitive_block_span", None)
        else:
            touched = set()
            current_origins = list(range(prefix_start, prefix_start + len(prefix_block)))

            for edge in path_edges[prefix_len:]:
                touched.update(origin for origin in current_origins if origin is not None)
                current_origins = [None] * len(edge["replacement_factors"])

            final_origins = (
                list(range(0, len(left_factors)))
                + current_origins
                + list(range(prefix_start + len(prefix_block), len(context_factors)))
            )
            primitive_span = candidate.get("primitive_block_span", None)

            if primitive_span is not None:
                for i in range(primitive_span[0], primitive_span[1]):
                    if 0 <= i < len(final_origins):
                        touched.add(final_origins[i])

            span = self.span_from_touched_positions(touched)

        if span is None:
            return None

        prefix_key = tuple(self.bridge_edge_key(edge) for edge in path_edges[:prefix_len])

        return {
            "context": "bridge_prefix",
            "context_factors": context_factors,
            "context_key": ("bridge_prefix", self.mp_factor_tuple_key(context_factors), prefix_key),
            "span": span,
            "block": tuple(context_factors[span[0]:span[1]]),
            "candidate": candidate,
            "prefix_length": prefix_len,
        }

    def bridge_candidate_original_profile(self, candidate, ambient_factors):
        """
        Profile a bridge primitive in the original monomial.
        """

        path_edges = list(candidate.get("bridge_path", []))

        if not path_edges:
            return None

        left_factors = tuple(candidate.get("bridge_left_factors", ()))
        right_factors = tuple(candidate.get("bridge_right_factors", ()))
        original_block = tuple(path_edges[0]["target_factors"])
        block_start = len(left_factors)
        context_factors = tuple(ambient_factors)
        touched = set()
        current_origins = list(range(block_start, block_start + len(original_block)))

        for edge in path_edges:
            touched.update(origin for origin in current_origins if origin is not None)
            current_origins = [None] * len(edge["replacement_factors"])

        final_origins = (
            list(range(0, len(left_factors)))
            + current_origins
            + list(range(block_start + len(original_block), len(context_factors)))
        )
        primitive_span = candidate.get("primitive_block_span", None)

        if primitive_span is not None:
            for i in range(primitive_span[0], primitive_span[1]):
                if 0 <= i < len(final_origins):
                    touched.add(final_origins[i])

        span = self.span_from_touched_positions(touched)

        if span is None:
            return None

        return {
            "context": "original",
            "context_factors": context_factors,
            "context_key": ("original", self.mp_factor_tuple_key(context_factors)),
            "span": span,
            "block": tuple(context_factors[span[0]:span[1]]),
            "candidate": candidate,
            "prefix_length": 0,
        }

    def primitive_candidate_profiles_for_obvious_test(self, candidate, ambient_factors):
        """
        Return possible block-intersection profiles for a primitive candidate.
        """

        ambient_factors = tuple(ambient_factors)

        if candidate.get("kind") == "bridge_replacement":
            if candidate.get("profile_contexts"):
                return [
                    {
                        **profile,
                        "candidate": candidate,
                    }
                    for profile in candidate["profile_contexts"]
                ]

            profiles = []
            original_profile = self.bridge_candidate_original_profile(candidate, ambient_factors)

            if original_profile is not None:
                profiles.append(original_profile)

            path_edges = list(candidate.get("bridge_path", []))

            for prefix_len in range(1, len(path_edges) + 1):
                profile = self.bridge_candidate_profile_after_prefix(candidate, prefix_len)

                if profile is not None:
                    profiles.append(profile)

            return profiles

        block = tuple(candidate.get("block", ()))
        start = len(tuple(candidate.get("left", ())))
        span = (start, start + len(block))

        return [{
            "context": "original",
            "context_factors": ambient_factors,
            "context_key": ("original", self.mp_factor_tuple_key(ambient_factors)),
            "span": span,
            "block": block,
            "candidate": candidate,
            "prefix_length": 0,
        }]

    def ensure_bridge_interval_history(self):
        """
        Ensure bridge-interval bookkeeping exists on older notebook objects.
        """

        if not hasattr(self, "bridge_interval_records"):
            self.bridge_interval_records = []

        if not hasattr(self, "bridge_interval_equalities"):
            self.bridge_interval_equalities = []

    def primitive_candidate_interval_key(self, candidate):
        """
        Stable key for the primitive data used in a bridge interval.
        """

        if candidate is None:
            return None

        primitive = candidate.get("primitive_element", candidate.get("primitive", None))

        if hasattr(primitive, "key"):
            primitive_key = primitive.key()
        else:
            primitive_key = repr(primitive)

        path_key = tuple(
            (
                record.get("occurrence_span_before", None),
                self.bridge_edge_key(record),
            )
            for record in candidate.get("bridge_path", [])
        )
        source_key = None

        if candidate.get("kind") == "bridge_replacement":
            source_key = self.primitive_candidate_interval_key(
                candidate.get("source_primitive", None)
            )

        return (
            candidate.get("kind", "mp_cell"),
            primitive_key,
            self.mp_factor_tuple_key(candidate.get("left", ())),
            self.mp_factor_tuple_key(candidate.get("block", ())),
            self.mp_factor_tuple_key(candidate.get("right", ())),
            path_key,
            source_key,
        )

    def bridge_interval_profile_is_connected(self, direct_profile, bridge_profile):
        """
        Test whether a bridge chain is connected to the first MP-cell block.

        The direct profile marks the initial touched region. Each bridge must
        overlap that current touched region, and its replacement block then
        becomes touched. The final MP-cell primitive must also overlap the
        touched region. This counts overlaps through factors introduced by
        earlier bridges, not only through original factors.
        """

        if direct_profile.get("context_key") != bridge_profile.get("context_key"):
            return False

        bridge_candidate = bridge_profile.get("candidate", None)

        if bridge_candidate is None or bridge_candidate.get("kind") != "bridge_replacement":
            return False

        path_records = list(bridge_candidate.get("bridge_path", []))
        prefix_len = bridge_profile.get("prefix_length", 0) or 0

        if prefix_len < 0 or prefix_len > len(path_records):
            return False

        context_factors = tuple(bridge_profile.get("context_factors", ()))
        touched = [False] * len(context_factors)
        start, stop = direct_profile.get("span", (None, None))

        if start is None or stop is None:
            return False

        if start < 0 or stop > len(touched) or start >= stop:
            return False

        for i in range(start, stop):
            touched[i] = True

        for record in path_records[prefix_len:]:
            span = record.get("occurrence_span_before", None)

            if span is None:
                return False

            rstart, rstop = span

            if rstart < 0 or rstop > len(touched) or rstart >= rstop:
                return False

            if not any(touched[rstart:rstop]):
                return False

            replacement_len = len(tuple(record.get("replacement_factors", ())))
            touched = touched[:rstart] + ([True] * replacement_len) + touched[rstop:]

        primitive_span = bridge_candidate.get("primitive_block_span", None)

        if primitive_span is None:
            return False

        pstart, pstop = primitive_span

        if pstart < 0 or pstop > len(touched) or pstart >= pstop:
            return False

        return any(touched[pstart:pstop])

    def bridge_interval_data_for_profiles(self, first_candidate, second_candidate, first_profile, second_profile):
        """
        Return interval metadata, or None when a direct/bridge pair is non-Massey.

        Pairs with zero or two bridge primitives are not bridge intervals here,
        so they return an empty dict and keep the existing behavior.
        """

        first_is_bridge = first_candidate.get("kind") == "bridge_replacement"
        second_is_bridge = second_candidate.get("kind") == "bridge_replacement"

        if not first_is_bridge and not second_is_bridge:
            return {}

        first_span = first_profile.get("span", None)
        second_span = second_profile.get("span", None)

        if first_span is None or second_span is None:
            return None

        if self.spans_are_disjoint(first_span, second_span):
            return None

        if first_is_bridge and second_is_bridge:
            first_connected = self.bridge_interval_profile_is_connected(
                second_profile,
                first_profile
            )
            second_connected = self.bridge_interval_profile_is_connected(
                first_profile,
                second_profile
            )

            if not first_connected and not second_connected:
                return None

            if first_span[0] <= second_span[0]:
                orientation = "left_to_right"
            else:
                orientation = "right_to_left"

            return {
                "kind": "bridge_interval",
                "orientation": orientation,
                "first_candidate": first_candidate,
                "second_candidate": second_candidate,
                "first_profile": first_profile,
                "second_profile": second_profile,
                "bridge_candidates": (first_candidate, second_candidate),
                "bridge_touched_blocks": (
                    tuple(first_profile.get("block", ())),
                    tuple(second_profile.get("block", ()))
                ),
                "bridge_path": (
                    list(first_candidate.get("bridge_path", []))
                    + list(second_candidate.get("bridge_path", []))
                ),
                "final_mp_cell_candidates": (
                    first_candidate.get("source_primitive", None),
                    second_candidate.get("source_primitive", None)
                ),
            }

        if first_is_bridge:
            bridge_candidate = first_candidate
            bridge_profile = first_profile
            direct_candidate = second_candidate
            direct_profile = second_profile
        else:
            bridge_candidate = second_candidate
            bridge_profile = second_profile
            direct_candidate = first_candidate
            direct_profile = first_profile

        if not self.bridge_interval_profile_is_connected(direct_profile, bridge_profile):
            return None

        direct_span = direct_profile["span"]
        bridge_span = bridge_profile["span"]

        if direct_span[0] <= bridge_span[0]:
            orientation = "left_to_right"
        else:
            orientation = "right_to_left"

        return {
            "kind": "bridge_interval",
            "orientation": orientation,
            "direct_candidate": direct_candidate,
            "bridge_candidate": bridge_candidate,
            "direct_profile": direct_profile,
            "bridge_profile": bridge_profile,
            "first_mp_cell_block": tuple(direct_profile.get("block", ())),
            "bridge_touched_block": tuple(bridge_profile.get("block", ())),
            "bridge_path": list(bridge_candidate.get("bridge_path", [])),
            "final_mp_cell_candidate": bridge_candidate.get("source_primitive", None),
        }

    def element_key_up_to_sign(self, element):
        """
        Key for comparing cycle representatives up to an overall sign.
        """

        if element is None:
            return None

        try:
            positive_key = element.key()
            negative_key = (-element).key()
        except Exception:
            positive_key = repr(element)
            negative_key = repr(-element)

        return tuple(sorted((positive_key, negative_key), key=repr))

    def bridge_interval_representative_data(self, record_data, interval_data):
        """
        Build the cycle F*tail - prefix*G for a bridge interval.

        The two primitive candidates are already full-product primitives. The
        left one corresponds to F*f_{m+1}...f_n and the right one to the
        Koszul-signed f_1...f_{s-1}*G. Their difference is the associated
        Massey-product representative.
        """

        candidates = tuple(record_data.get("candidates", ()))
        profiles = tuple(record_data.get("profiles", ()))

        if len(candidates) != 2 or len(profiles) != 2:
            return None

        first_span = profiles[0].get("span", None)
        second_span = profiles[1].get("span", None)

        if first_span is None or second_span is None:
            return None

        if first_span[0] <= second_span[0]:
            left_candidate = candidates[0]
            right_candidate = candidates[1]
            left_profile = profiles[0]
            right_profile = profiles[1]
        else:
            left_candidate = candidates[1]
            right_candidate = candidates[0]
            left_profile = profiles[1]
            right_profile = profiles[0]

        left_primitive = self.primitive_candidate_to_element(left_candidate)
        right_primitive = self.primitive_candidate_to_element(right_candidate)

        if left_primitive is None or right_primitive is None:
            return None

        representative = left_primitive - right_primitive
        differential = self.d(representative)

        return {
            "representative": representative,
            "massey_cycle": representative,
            "left_primitive": left_primitive,
            "right_primitive": right_primitive,
            "formula": "left_primitive - right_primitive",
            "left_candidate": left_candidate,
            "right_candidate": right_candidate,
            "left_profile": left_profile,
            "right_profile": right_profile,
            "left_span": left_profile.get("span", None),
            "right_span": right_profile.get("span", None),
            "left_block": tuple(left_profile.get("block", ())),
            "right_block": tuple(right_profile.get("block", ())),
            "differential": differential,
            "is_cycle": differential.is_zero(),
            "key_up_to_sign": self.element_key_up_to_sign(representative),
        }

    def remember_bridge_interval_generation(self, expression, local_massey_product, record_data, interval_data):
        """
        Remember bridge intervals and note alternate names for the same interval.
        """

        if not interval_data:
            return

        self.ensure_bridge_interval_history()
        expression_key = self.generation_item_key(expression)
        local_key = self.generation_item_key(local_massey_product)
        representative_data = record_data.get("bridge_interval_representative", None)
        representative_key = None

        if representative_data is not None:
            representative_key = representative_data.get("key_up_to_sign", None)

        direct_candidate = interval_data.get("direct_candidate", None)

        if direct_candidate is None:
            direct_candidate = interval_data.get("first_candidate", None)

        final_candidate = interval_data.get("final_mp_cell_candidate", None)

        if final_candidate is None:
            final_candidates = tuple(interval_data.get("final_mp_cell_candidates", ()))
            final_candidate = final_candidates[-1] if final_candidates else interval_data.get("second_candidate", None)

        direct_key = self.primitive_candidate_interval_key(direct_candidate)
        final_key = self.primitive_candidate_interval_key(final_candidate)
        endpoint_keys = tuple(sorted((direct_key, final_key), key=repr))
        bridge_path_key = tuple(sorted(
            (
                self.bridge_edge_key(record)
                for record in interval_data.get("bridge_path", [])
            ),
            key=repr
        ))
        if representative_key is not None:
            group_key = ("bridge_interval_representative", representative_key)
        else:
            group_key = ("bridge_interval", endpoint_keys, bridge_path_key)

        for old_record in self.bridge_interval_records:
            if old_record.get("group_key") != group_key:
                continue

            if old_record.get("expression_key") == expression_key:
                return

            equality_key = tuple(sorted(
                (old_record.get("expression_key"), expression_key),
                key=repr
            ))

            if not any(eq.get("key") == equality_key for eq in self.bridge_interval_equalities):
                self.bridge_interval_equalities.append({
                    "key": equality_key,
                    "left": old_record.get("expression"),
                    "right": expression,
                    "left_local_massey_product": old_record.get("local_massey_product"),
                    "right_local_massey_product": local_massey_product,
                    "left_representative": old_record.get("representative", None),
                    "right_representative": (
                        representative_data.get("representative", None)
                        if representative_data is not None
                        else None
                    ),
                    "reason": "same bridge interval with different presentation",
                    "group_key": group_key,
                })

        self.bridge_interval_records.append({
            **interval_data,
            "expression": expression,
            "expression_key": expression_key,
            "local_massey_product": local_massey_product,
            "local_massey_product_key": local_key,
            "representative_data": representative_data,
            "representative": (
                representative_data.get("representative", None)
                if representative_data is not None
                else None
            ),
            "massey_cycle": (
                representative_data.get("massey_cycle", None)
                if representative_data is not None
                else None
            ),
            "representative_key_up_to_sign": representative_key,
            "generation_record": record_data,
            "group_key": group_key,
        })

    def mp_input_from_factor_block(self, factors):
        """
        Convert a nonempty factor block into one Massey-product input.

        A single length-one Q.mp(x) becomes x. A longer monomial block is
        returned as its underlying Path, e.g. x2*x3 becomes the path x2x3.
        """

        factors = tuple(factors)

        if not factors:
            return None

        if len(factors) == 1:
            factor = factors[0]

            if isinstance(factor, MasseyProduct) and len(factor.inputs) == 1:
                return factor.inputs[0]

            return factor

        element = self.mp_factors_to_element(factors)

        if element is None or len(element.terms) != 1:
            return None

        path, coeff = next(iter(element.terms.items()))

        if coeff != 1:
            return None

        return path

    def induced_massey_products_from_factor_product(self, factors, record=True, minimal_only=True):
        """
        Generate named 3-fold MPs from primitive-block pairs in a product.

        This is used when the ambient monomial/product itself is not a
        defined higher Massey product. Overlapping blocks produce
        m3(f1, f2, f3); disjoint blocks produce m3(f1, 0, f3).
        """

        try:
            ambient_factors = self.canonical_mp_factors(factors)
        except Exception:
            return []

        if not ambient_factors:
            return []

        product = MPProduct(self, ambient_factors)
        candidates = self.find_primitives_mp_product(
            product,
            record=record,
            minimal_only=minimal_only
        )
        profile_groups = [
            self.primitive_candidate_profiles_for_obvious_test(candidate, ambient_factors)
            for candidate in candidates
        ]
        induced_massey_products = []
        induced_keys = set()

        for i in range(len(candidates)):
            for j in range(i + 1, len(candidates)):
                for first_profile in profile_groups[i]:
                    for second_profile in profile_groups[j]:
                        if first_profile["context_key"] != second_profile["context_key"]:
                            continue

                        interval_data = self.bridge_interval_data_for_profiles(
                            candidates[i],
                            candidates[j],
                            first_profile,
                            second_profile
                        )

                        if interval_data is None:
                            continue

                        if self.spans_are_disjoint(first_profile["span"], second_profile["span"]):
                            induced_inputs = self.zero_m3_inputs_from_disjoint_profiles(
                                first_profile,
                                second_profile
                            )

                            if induced_inputs is None:
                                continue

                            induced_M = self.try_generate_mp(
                                *induced_inputs,
                                record=record,
                                verbose=False
                            )

                            if induced_M is None:
                                continue

                            induced_key = self.mp_key(*induced_M.inputs)

                            if induced_key not in induced_keys:
                                induced_keys.add(induced_key)
                                induced_massey_products.append(induced_M)

                            continue

                        induced_inputs = self.induced_m3_inputs_from_profiles(
                            first_profile,
                            second_profile
                        )

                        if induced_inputs is None:
                            continue

                        induced_M = self.try_generate_mp(
                            *induced_inputs,
                            record=record,
                            verbose=False
                        )

                        if induced_M is None:
                            continue

                        induced_key = self.mp_key(*induced_M.inputs)

                        if induced_key not in induced_keys:
                            induced_keys.add(induced_key)
                            induced_massey_products.append(induced_M)

        return induced_massey_products

    def induced_m3_inputs_from_profiles(self, first_profile, second_profile):
        """
        If two maximal primitive blocks overlap, return m3 input blocks.

        If A = f1*f2 and B = f2*f3, this returns (f1, f2, f3).
        """

        if first_profile["context_key"] != second_profile["context_key"]:
            return None

        first_span = first_profile["span"]
        second_span = second_profile["span"]

        if self.spans_are_disjoint(first_span, second_span):
            return None

        if first_span[0] <= second_span[0]:
            left_span = first_span
            right_span = second_span
        else:
            left_span = second_span
            right_span = first_span

        # Proper overlap only. Containment does not determine f1,f2,f3.
        if not (left_span[0] < right_span[0] < left_span[1] < right_span[1]):
            return None

        context_factors = tuple(first_profile["context_factors"])
        first_block = context_factors[left_span[0]:right_span[0]]
        middle_block = context_factors[right_span[0]:left_span[1]]
        third_block = context_factors[left_span[1]:right_span[1]]

        inputs = (
            self.mp_input_from_factor_block(first_block),
            self.mp_input_from_factor_block(middle_block),
            self.mp_input_from_factor_block(third_block),
        )

        if any(x is None for x in inputs):
            return None

        return inputs

    def zero_m3_inputs_from_disjoint_profiles(self, first_profile, second_profile):
        """
        If two maximal primitive blocks are disjoint, return (f1, 0, f3).

        Here f1 is the left maximal primitive block and f3 is the right one.
        The middle input records that their intersection is empty.
        """

        if first_profile["context_key"] != second_profile["context_key"]:
            return None

        first_span = first_profile["span"]
        second_span = second_profile["span"]

        if not self.spans_are_disjoint(first_span, second_span):
            return None

        if first_span[0] <= second_span[0]:
            left_span = first_span
            right_span = second_span
        else:
            left_span = second_span
            right_span = first_span

        context_factors = tuple(first_profile["context_factors"])
        first_block = context_factors[left_span[0]:left_span[1]]
        third_block = context_factors[right_span[0]:right_span[1]]

        inputs = (
            self.mp_input_from_factor_block(first_block),
            0,
            self.mp_input_from_factor_block(third_block),
        )

        if inputs[0] is None or inputs[2] is None:
            return None

        return inputs

    def mp_expression_factor_for_output(self, factor):
        """
        Use x instead of Q.mp(x) when displaying outside factors.
        """

        if isinstance(factor, MasseyProduct) and len(factor.inputs) == 1:
            return factor.inputs[0]

        return factor

    def mp_product_expression_with_local_massey(self, context_factors, span, local_massey_product):
        """
        Replace one context subblock by a local Massey product.

        Example:
            x1*x2*x3*x4 with local Q.mp(x1,x2,x3)
            gives Q.mp(x1,x2,x3)*x4.
        """

        context_factors = tuple(context_factors)
        start, stop = span
        output_factors = [
            self.mp_expression_factor_for_output(factor)
            for factor in context_factors[:start]
        ]
        output_factors.append(local_massey_product)
        output_factors.extend(
            self.mp_expression_factor_for_output(factor)
            for factor in context_factors[stop:]
        )

        if len(output_factors) == 1:
            return output_factors[0]

        return MPProduct(self, output_factors)

    def generated_massey_products_from_mp_product(self, P, record=True, minimal_only=True):
        """
        Generate all named 3-fold Massey products detected inside an MPProduct.

        The returned expressions keep irrelevant outside factors. Thus if
        x1*x2*x3 is the active subblock in x1*x2*x3*x4, the name recorded is
        Q.mp(x1,x2,x3)*x4.
        """

        if isinstance(P, MasseyProduct):
            P = P.as_product()

        try:
            ambient_factors = self.canonical_mp_factors(P)
        except Exception:
            return {
                "have_primitives": [],
                "no_obvious_primitives": [],
                "records": [],
                "candidates": [],
            }

        if not ambient_factors:
            return {
                "have_primitives": [],
                "no_obvious_primitives": [],
                "records": [],
                "candidates": [],
            }

        product = MPProduct(self, ambient_factors)
        candidates = self.find_primitives_mp_product(
            product,
            record=record,
            minimal_only=minimal_only
        )
        profile_groups = [
            self.primitive_candidate_profiles_for_obvious_test(candidate, ambient_factors)
            for candidate in candidates
        ]
        have_primitives = []
        no_obvious_primitives = []
        records = []
        seen = set()

        def remember(kind, expression, record_data):
            key = (kind, self.mp_expression_key(expression))

            if key in seen:
                return

            seen.add(key)
            records.append(record_data)

            if kind == "have_primitives":
                have_primitives.append(expression)
            else:
                no_obvious_primitives.append(expression)

        for i in range(len(candidates)):
            for j in range(i + 1, len(candidates)):
                for first_profile in profile_groups[i]:
                    for second_profile in profile_groups[j]:
                        if first_profile["context_key"] != second_profile["context_key"]:
                            continue

                        interval_data = self.bridge_interval_data_for_profiles(
                            candidates[i],
                            candidates[j],
                            first_profile,
                            second_profile
                        )

                        if interval_data is None:
                            continue

                        first_span = first_profile["span"]
                        second_span = second_profile["span"]
                        cover_span = (
                            min(first_span[0], second_span[0]),
                            max(first_span[1], second_span[1])
                        )

                        if self.spans_are_disjoint(first_span, second_span):
                            local_inputs = self.zero_m3_inputs_from_disjoint_profiles(
                                first_profile,
                                second_profile
                            )
                            kind = "have_primitives"
                            has_obvious = True
                        else:
                            local_inputs = self.induced_m3_inputs_from_profiles(
                                first_profile,
                                second_profile
                            )
                            kind = "no_obvious_primitives"
                            has_obvious = False

                        if local_inputs is None:
                            continue

                        local_M = self.try_generate_mp(
                            *local_inputs,
                            record=record,
                            verbose=False
                        )

                        if local_M is None:
                            continue

                        context_factors = tuple(first_profile["context_factors"])
                        expression = self.mp_product_expression_with_local_massey(
                            context_factors,
                            cover_span,
                            local_M
                        )
                        record_data = {
                            "expression": expression,
                            "local_massey_product": local_M,
                            "has_obvious_primitive": has_obvious,
                            "candidates": (candidates[i], candidates[j]),
                            "profiles": (first_profile, second_profile),
                            "context_factors": context_factors,
                            "cover_span": cover_span,
                            "ambient_product": product,
                        }

                        if interval_data:
                            record_data["bridge_interval"] = interval_data
                            representative_data = self.bridge_interval_representative_data(
                                record_data,
                                interval_data
                            )

                            if representative_data is not None:
                                record_data["bridge_interval_representative"] = representative_data

                        remember(kind, expression, record_data)

                        if record and interval_data:
                            self.remember_bridge_interval_generation(
                                expression,
                                local_M,
                                record_data,
                                interval_data
                            )

        return {
            "have_primitives": have_primitives,
            "no_obvious_primitives": no_obvious_primitives,
            "records": records,
            "candidates": candidates,
        }

    def classify_massey_product_primitives(self, M, record=True, minimal_only=True):
        """
        Classify whether M has an obvious primitive by block disjointness.
        """
        if isinstance(M, MasseyProduct) and self.has_zero_mp_input(M.inputs):
            return {
                "has_obvious_primitive": True,
                "candidates": [],
                "profiles": [],
                "obvious_pair": None,
                "obvious_split": None,
                "obvious_factorization": None,
                "ambient_product": None,
                "obvious_massey_product": M,
                "obvious_massey_products": [M],
                "induced_massey_products": [],
            }


        product = self.massey_product_input_product(M)

        if product is None:
            return {
                "has_obvious_primitive": False,
                "candidates": [],
                "profiles": [],
                "obvious_pair": None,
                "obvious_split": None,
                "obvious_factorization": None,
                "obvious_massey_product": None,
                "obvious_massey_products": [],
                "induced_massey_products": [],
            }

        ambient_factors = tuple(product.factors)
        candidates = self.find_primitives_mp_product(
            product,
            record=record,
            minimal_only=minimal_only
        )

        profile_groups = [
            self.primitive_candidate_profiles_for_obvious_test(candidate, ambient_factors)
            for candidate in candidates
        ]
        induced_massey_products = []
        induced_keys = set()

        for i in range(len(candidates)):
            for j in range(i + 1, len(candidates)):
                for first_profile in profile_groups[i]:
                    for second_profile in profile_groups[j]:
                        if first_profile["context_key"] != second_profile["context_key"]:
                            continue

                        interval_data = self.bridge_interval_data_for_profiles(
                            candidates[i],
                            candidates[j],
                            first_profile,
                            second_profile
                        )

                        if interval_data is None:
                            continue

                        if not self.spans_are_disjoint(first_profile["span"], second_profile["span"]):
                            induced_inputs = self.induced_m3_inputs_from_profiles(
                                first_profile,
                                second_profile
                            )

                            if induced_inputs is not None:
                                induced_M = self.try_generate_mp(
                                    *induced_inputs,
                                    record=record,
                                    verbose=False
                                )

                                if induced_M is not None:
                                    induced_key = self.mp_key(*induced_M.inputs)

                                    if induced_key not in induced_keys:
                                        induced_keys.add(induced_key)
                                        induced_massey_products.append(induced_M)

                            continue

                        if first_profile["span"][0] <= second_profile["span"][0]:
                            left_profile = first_profile
                            right_profile = second_profile
                        else:
                            left_profile = second_profile
                            right_profile = first_profile

                        split_index = left_profile["span"][1]
                        context_factors = tuple(left_profile["context_factors"])
                        obvious_massey_product = None
                        zero_inputs = self.zero_m3_inputs_from_disjoint_profiles(
                            left_profile,
                            right_profile
                        )

                        if zero_inputs is not None:
                            obvious_massey_product = self.try_generate_mp(
                                *zero_inputs,
                                record=record,
                                verbose=False
                            )

                        return {
                            "has_obvious_primitive": True,
                            "candidates": candidates,
                            "profiles": profile_groups,
                            "obvious_pair": (
                                candidates[i],
                                candidates[j],
                            ),
                            "obvious_profiles": (
                                first_profile,
                                second_profile,
                            ),
                            "ordered_obvious_profiles": (
                                left_profile,
                                right_profile,
                            ),
                            "obvious_split": split_index,
                            "obvious_factorization": (
                                context_factors[:split_index],
                                context_factors[split_index:],
                            ),
                            "ambient_product": product,
                            "obvious_massey_product": obvious_massey_product,
                            "obvious_massey_products": (
                                [obvious_massey_product]
                                if obvious_massey_product is not None
                                else []
                            ),
                            "induced_massey_products": induced_massey_products,
                        }

        return {
            "has_obvious_primitive": False,
            "candidates": candidates,
            "profiles": profile_groups,
            "obvious_pair": None,
            "obvious_split": None,
            "obvious_factorization": None,
            "ambient_product": product,
            "obvious_massey_product": None,
            "obvious_massey_products": [],
            "induced_massey_products": induced_massey_products,
        }

    def dedupe_massey_products(self, items):
        """
        Remove duplicate MasseyProduct objects while preserving order.
        """

        out = []
        seen = set()

        for M in items:
            key = repr(M)

            if key not in seen:
                seen.add(key)
                out.append(M)

        return out

    def flatten_m2_only_terms(self, P):
        """
        Return coefficient/factor-list pairs for an MP expression.

        Coefficients are kept for bookkeeping, but generation uses only the
        factor list. Thus 2*x*y and x*y generate the same MP patterns, while
        2*x*y - 3*y*z is processed term by term.
        """

        if isinstance(P, MPElement):
            out = []

            for product, coeff in P.terms.items():
                if is_zero_coeff(coeff):
                    continue

                factors = self.flatten_m2_only(product)

                if factors:
                    out.append((coeff, factors))

            return out

        if isinstance(P, Element):
            out = []

            for path, coeff in P.terms.items():
                if is_zero_coeff(coeff):
                    continue

                factors = self.flatten_m2_only(path)

                if factors:
                    out.append((coeff, factors))

            return out

        factors = self.flatten_m2_only(P)

        if not factors:
            return []

        return [(1, factors)]

    def generation_cells(self, cells=None):
        """
        Return actual attached cell objects used by generate_massey_products.
        """

        raw = []

        if cells is not None:
            if isinstance(cells, dict):
                raw.extend(cells.values())
            else:
                raw.extend(cells)
        else:
            if hasattr(self, "cells"):
                if isinstance(self.cells, dict):
                    raw.extend(self.cells.values())
                else:
                    raw.extend(self.cells)

            if hasattr(self, "attachment_history"):
                for entry in self.attachment_history:
                    cell = entry.get("cell", None)

                    if cell is not None:
                        raw.append(cell)

            if hasattr(self, "resolved_massey_products"):
                raw.extend(self.resolved_massey_products.values())

            if hasattr(self, "resolved_mp_expressions"):
                raw.extend(self.resolved_mp_expressions.values())

        out = []
        seen = set()

        for cell in raw:
            if not isinstance(cell, Arrow):
                continue

            key = id(cell)

            if key not in seen:
                seen.add(key)
                out.append(cell)

        return out

    def generation_item_key(self, item):
        """
        Stable key for a generated item, allowing product-valued names.
        """

        if isinstance(item, MasseyProduct):
            return ("MasseyProduct", self.mp_key(*item.inputs))

        if isinstance(item, (MPProduct, MPElement)):
            return self.mp_expression_key(item)

        if hasattr(item, "key"):
            return item.key()

        return repr(item)

    def remember_generated_mp_product_result(self, result, source=None):
        """
        Remember the have/no-obvious classification of product-level output.
        """

        if not hasattr(self, "generated_mp_expression_data"):
            self.generated_mp_expression_data = {}

        for kind in ("have_primitives", "no_obvious_primitives"):
            for expression in result.get(kind, []):
                key = self.generation_item_key(expression)
                self.generated_mp_expression_data[key] = {
                    "classification": kind,
                    "source": source,
                    "product_result": result,
                }

    def direct_mp_block_primitive(self, *inputs):
        """
        Return a primitive recorded directly by a cell, without replacements.

        This is used to build A_inf replacement primitives without recursing
        back into the replacement graph being generated.
        """

        inputs = self.normalize_mp_inputs(inputs)

        if len(inputs) == 0:
            return None

        if self.has_zero_mp_input(inputs):
            return self.zero()

        if len(inputs) == 1:
            return self.expand_mp_input(inputs[0])

        key = tuple(inputs)

        try:
            if key in self.resolved_massey_products:
                return self.resolved_massey_products[key]
        except TypeError:
            pass

        try:
            factors = self.canonical_mp_factors(inputs)
            expr_key = self.mp_expression_key(MPProduct(self, factors))
        except Exception:
            factors = None
            expr_key = None

        if expr_key is not None:
            primitive = getattr(self, "resolved_mp_expressions", {}).get(expr_key, None)

            if primitive is not None:
                return primitive

            target_product = MPProduct(self, factors)
            target_key = self.mp_product_key(target_product)

            for expression_key, primitive in getattr(self, "resolved_mp_expressions", {}).items():
                expression = None

                for entry in getattr(self, "attachment_history", []):
                    if entry.get("key", None) == expression_key:
                        expression = entry.get("expression", None)
                        break

                if not isinstance(expression, MPElement) or len(expression.terms) != 1:
                    continue

                product, coeff = next(iter(expression.terms.items()))

                if is_zero_coeff(coeff):
                    continue

                if self.mp_product_key(product) != target_key:
                    continue

                try:
                    inverse_coeff = simplify_coeff(1 / coeff)
                except Exception:
                    continue

                return inverse_coeff * primitive

        if factors is None:
            return None

        element = self.mp_factors_to_element(factors)

        if element is not None and len(element.terms) == 1:
            path, coeff = next(iter(element.terms.items()))

            if coeff == 1:
                path_key = tuple(path.arrows)
                primitive = getattr(self, "cells", {}).get(path_key, None)

                if primitive is not None:
                    return primitive

        return None

    def can_define_mp_direct(self, *inputs):
        """
        Definability test using only directly recorded primitives.
        """

        inputs = self.normalize_mp_inputs(inputs)
        n = len(inputs)

        if n == 0:
            return False

        if self.has_zero_mp_input(inputs):
            return True

        if n == 1:
            return True

        if n == 2:
            return self.are_composable(inputs[0], inputs[1])

        for subkey in self.required_lower_mps(*inputs):
            if self.direct_mp_block_primitive(*subkey) is None:
                return False

        return True

    def get_or_create_mp_direct(self, *inputs):
        """
        Return/register Q.mp(inputs) after direct definability has been checked.
        """

        inputs = self.normalize_mp_inputs(inputs)

        if not self.can_define_mp_direct(*inputs):
            return None

        key = tuple(inputs)

        if key in self.massey_products:
            return self.massey_products[key]

        M = MasseyProduct(self, key)
        self.massey_products[key] = M
        return M

    def ainf_H_block(self, block):
        """
        H-block for A_inf replacement primitives, using direct primitives.
        """

        block = tuple(self.normalize_mp_input(x) for x in tuple(block))

        if len(block) == 0:
            return None

        if len(block) == 1:
            return self.expand_mp_input(block[0])

        primitive = self.direct_mp_block_primitive(*block)

        if primitive is None:
            return None

        return to_element(primitive)

    def ainf_middle_primitive_for_inputs(self, inputs):
        """
        Primitive for the outer A_inf replacement relation.

        For inputs f1,...,fN this is the sum over the middle splits
        H(f1...fi)*H(f_{i+1}...fN), with the same signs as expand_mp.
        """

        inputs = tuple(self.normalize_mp_input(x) for x in tuple(inputs))
        n = len(inputs)

        if n < 4:
            return self.zero()

        primitive = self.zero()

        for left_length in range(2, n - 1):
            left_inputs = inputs[:left_length]
            right_inputs = inputs[left_length:]
            left_factor = self.ainf_H_block(left_inputs)
            right_factor = self.ainf_H_block(right_inputs)

            if left_factor is None or right_factor is None:
                return None

            coeff = -self.mp_split_coefficient_for_inputs(inputs, left_length)
            primitive = primitive + coeff * (left_factor * right_factor)

        return primitive

    def coeff_inverse_if_unit(self, coeff):
        """
        Invert scalar or one-term Koszul-sign coefficients when possible.
        """

        coeff = simplify_coeff(coeff)

        if is_zero_coeff(coeff):
            return None

        if isinstance(coeff, Sign):
            if len(coeff.terms) != 1:
                return None

            symbols, scalar = next(iter(coeff.terms.items()))

            if simplify(scalar) == 0:
                return None

            return Sign({symbols: simplify(1 / scalar)})

        try:
            return simplify_coeff(1 / coeff)
        except Exception:
            return None

    def coeff_quotient_if_possible(self, numerator, denominator):
        """
        Return numerator / denominator for coefficients when recognizable.
        """

        inverse = self.coeff_inverse_if_unit(denominator)

        if inverse is None:
            return None

        try:
            return simplify_coeff(numerator * inverse)
        except Exception:
            return None

    def element_scalar_multiple(self, numerator, denominator):
        """
        If numerator = c * denominator, return c; otherwise return None.
        """

        numerator = to_element(numerator)
        denominator = to_element(denominator)

        if not denominator.terms:
            return None

        for path, coeff in numerator.terms.items():
            if path not in denominator.terms and not is_zero_coeff(coeff):
                return None

        scalar = None

        for path, denominator_coeff in denominator.terms.items():
            numerator_coeff = numerator.terms.get(path, 0)
            quotient = self.coeff_quotient_if_possible(
                numerator_coeff,
                denominator_coeff
            )

            if quotient is None:
                return None

            if scalar is None:
                scalar = quotient
                continue

            if not is_zero_coeff(simplify_coeff(quotient - scalar)):
                return None

        if scalar is None:
            return None

        if (numerator - scalar * denominator).is_zero():
            return simplify_coeff(scalar)

        return None

    def ainf_edge_relation_data(self, target_factors, replacement_factors, primitive):
        """
        Orient an A_inf replacement by checking d(primitive) = target + c*replacement.
        """

        target_element = self.mp_factors_to_element(target_factors)
        replacement_element = self.mp_factors_to_element(replacement_factors)

        if target_element is None or replacement_element is None or primitive is None:
            return None

        for primitive_candidate in (primitive, -primitive):
            differential = self.d(primitive_candidate)
            residual = differential - target_element
            replacement_coeff = self.element_scalar_multiple(
                residual,
                replacement_element
            )

            if replacement_coeff is not None:
                return {
                    "primitive": primitive_candidate,
                    "bridge_coeff": simplify_coeff(1),
                    "source_coeff": simplify_coeff(-replacement_coeff),
                    "relation_replacement_coeff": replacement_coeff,
                }

        return None

    def ainf_replacement_edge_records(self):
        """
        Return oriented A_inf replacement edges from registered higher MPs.
        """

        if getattr(self, "_building_ainf_replacement_edges", False):
            return []

        self._building_ainf_replacement_edges = True

        try:
            return self._ainf_replacement_edge_records_unchecked()
        finally:
            self._building_ainf_replacement_edges = False

    def _ainf_replacement_edge_records_unchecked(self):
        """
        Internal A_inf replacement edge enumerator.
        """

        edges = []
        seen = set()
        mp_items = list(getattr(self, "massey_products", {}).items())
        candidates = list(self.mp_extension_candidates())

        def add_oriented_edges(inputs, target_factors, replacement_factors, primitive):
            for oriented_target, oriented_replacement in (
                (target_factors, replacement_factors),
                (replacement_factors, target_factors),
            ):
                relation = self.ainf_edge_relation_data(
                    oriented_target,
                    oriented_replacement,
                    primitive
                )

                if relation is None:
                    continue

                edge_key = (
                    "ainf",
                    self.mp_factor_tuple_key(oriented_target),
                    self.mp_factor_tuple_key(oriented_replacement),
                    self.element_key_up_to_sign(relation["primitive"]),
                )

                if edge_key in seen:
                    continue

                seen.add(edge_key)
                edges.append({
                    "replacement_type": "ainf",
                    "target_product": MPProduct(self, oriented_target),
                    "replacement_product": MPProduct(self, oriented_replacement),
                    "target_factors": tuple(oriented_target),
                    "replacement_factors": tuple(oriented_replacement),
                    "target_key": self.mp_factor_tuple_key(oriented_target),
                    "replacement_key": self.mp_factor_tuple_key(oriented_replacement),
                    "bridge_coeff": relation["bridge_coeff"],
                    "source_coeff": relation["source_coeff"],
                    "bridge_cell": relation["primitive"],
                    "bridge_expression": None,
                    "ainf_inputs": tuple(inputs),
                    "ainf_primitive": relation["primitive"],
                    "relation_replacement_coeff": relation["relation_replacement_coeff"],
                })

        for key, left_mp in mp_items:
            inputs = tuple(self.normalize_mp_inputs(key))

            if len(inputs) < 3 or not isinstance(left_mp, MasseyProduct):
                continue

            for q in candidates:
                total_inputs = inputs + (q,)

                q_mp = self.get_or_create_mp_direct(q)
                first_mp = self.get_or_create_mp_direct(inputs[0])
                right_mp = self.get_or_create_mp_direct(*total_inputs[1:])

                if q_mp is not None and first_mp is not None and right_mp is not None:
                    primitive = self.ainf_middle_primitive_for_inputs(total_inputs)

                    if primitive is not None:
                        add_oriented_edges(
                            total_inputs,
                            (left_mp, q_mp),
                            (first_mp, right_mp),
                            primitive
                        )

                total_inputs = (q,) + inputs

                q_mp = self.get_or_create_mp_direct(q)
                last_mp = self.get_or_create_mp_direct(inputs[-1])
                left_extended_mp = self.get_or_create_mp_direct(*total_inputs[:-1])

                if q_mp is not None and last_mp is not None and left_extended_mp is not None:
                    primitive = self.ainf_middle_primitive_for_inputs(total_inputs)

                    if primitive is not None:
                        add_oriented_edges(
                            total_inputs,
                            (q_mp, left_mp),
                            (left_extended_mp, last_mp),
                            primitive
                        )

        return edges

    def bridge_replacement_edge_records(self, only_bridge_cell=None, excluded_bridge_cells=None):
        """
        Return oriented bridge replacement records from all bridge cells.
        """

        self.ensure_bridge_history()
        edges = []
        excluded_bridge_cells = set(excluded_bridge_cells or [])

        for bridge_key in list(getattr(self, "bridge_history", [])):
            bridge_expr = getattr(self, "bridge_expressions", {}).get(bridge_key, None)
            bridge_cell = getattr(self, "bridge_cells", {}).get(bridge_key, None)

            if bridge_cell is None:
                bridge_cell = getattr(self, "resolved_mp_expressions", {}).get(bridge_key, None)

            if bridge_expr is None or bridge_cell is None:
                continue

            if only_bridge_cell is not None and bridge_cell is not only_bridge_cell:
                continue

            if id(bridge_cell) in excluded_bridge_cells or bridge_cell in excluded_bridge_cells:
                continue

            if not isinstance(bridge_expr, MPElement):
                continue

            bridge_terms = list(bridge_expr.terms.items())

            if len(bridge_terms) < 2:
                continue

            for target_product, target_coeff in bridge_terms:
                target_factors = self.canonical_mp_factors(target_product)

                if not target_factors:
                    continue

                for replacement_product, replacement_coeff in bridge_terms:
                    if replacement_product == target_product:
                        continue

                    replacement_factors = self.canonical_mp_factors(replacement_product)

                    if not replacement_factors:
                        continue

                    try:
                        if target_coeff == 1:
                            bridge_coeff = simplify_coeff(1)
                            source_coeff = simplify_coeff(-replacement_coeff)
                        elif target_coeff == -1:
                            bridge_coeff = simplify_coeff(-1)
                            source_coeff = simplify_coeff(replacement_coeff)
                        else:
                            bridge_coeff = simplify_coeff(1 / target_coeff)
                            source_coeff = simplify_coeff(-replacement_coeff / target_coeff)
                    except Exception:
                        continue

                    edges.append({
                        "replacement_type": "bridge",
                        "target_product": target_product,
                        "replacement_product": replacement_product,
                        "target_factors": target_factors,
                        "replacement_factors": replacement_factors,
                        "target_key": self.mp_factor_tuple_key(target_factors),
                        "replacement_key": self.mp_factor_tuple_key(replacement_factors),
                        "bridge_coeff": bridge_coeff,
                        "source_coeff": source_coeff,
                        "bridge_cell": bridge_cell,
                        "bridge_expression": bridge_expr,
                        "bridge_key": bridge_key,
                    })

        return edges

    def replacement_edge_context_primitive(self, edge, left_factors=(), right_factors=(), include_edge_coeff=True):
        """
        Put a replacement-edge primitive inside left/right MP-factor context.
        """

        primitive = to_element(edge["bridge_cell"])
        left_factors = tuple(left_factors)
        right_factors = tuple(right_factors)

        if left_factors:
            left = self.mp_factors_to_element(left_factors)

            if left is None:
                return None

            primitive = self.sg(left) * (left * primitive)

        if right_factors:
            right = self.mp_factors_to_element(right_factors)

            if right is None:
                return None

            primitive = primitive * right

        if include_edge_coeff:
            primitive = edge.get("bridge_coeff", 1) * primitive

        return primitive

    def compatible_ainf_bridge_replacement_edge_records(self, bridge_edges=None, ainf_edges=None):
        """
        Add compatibility edges for an A_inf replacement followed by a bridge.

        If A -> B is an A_inf replacement, A -> C is a bridge replacement
        in the A-side monomial, and C -> D is another A_inf replacement,
        then add the induced replacement B -> D. This lets searches see
        chains such as
            m3(x1,x2,x3)*x4 ~ x1*m3(x2,x3,x4)
            x4 ~ y4
            m3(x1,x2,x3)*y4 ~ x1*m3(x2,x3,y4).
        """

        if bridge_edges is None:
            bridge_edges = self.bridge_replacement_edge_records()

        if ainf_edges is None:
            ainf_edges = self.ainf_replacement_edge_records()

        bridge_edges = [
            edge for edge in bridge_edges
            if edge.get("replacement_type", "bridge") == "bridge"
        ]
        ainf_edges = [
            edge for edge in ainf_edges
            if edge.get("replacement_type") == "ainf"
        ]

        ainf_by_target = {}

        for edge in ainf_edges:
            ainf_by_target.setdefault(edge["target_key"], []).append(edge)

        edges = []
        seen = set()

        for first_edge in ainf_edges:
            first_source_coeff = first_edge.get("source_coeff", None)
            first_source_inverse = self.coeff_inverse_if_unit(first_source_coeff)

            if first_source_inverse is None:
                continue

            first_target = tuple(first_edge["target_factors"])
            first_replacement = tuple(first_edge["replacement_factors"])
            first_primitive = self.replacement_edge_context_primitive(first_edge)

            if first_primitive is None:
                continue

            for bridge_edge in bridge_edges:
                bridge_target = tuple(bridge_edge["target_factors"])
                bridge_target_len = len(bridge_target)

                if bridge_target_len == 0 or bridge_target_len > len(first_target):
                    continue

                for start in range(0, len(first_target) - bridge_target_len + 1):
                    stop = start + bridge_target_len

                    if not self.mp_factors_equal(first_target[start:stop], bridge_target):
                        continue

                    bridged_target = (
                        first_target[:start]
                        + tuple(bridge_edge["replacement_factors"])
                        + first_target[stop:]
                    )
                    bridged_target_key = self.mp_factor_tuple_key(bridged_target)
                    bridge_primitive = self.replacement_edge_context_primitive(
                        bridge_edge,
                        first_target[:start],
                        first_target[stop:]
                    )

                    if bridge_primitive is None:
                        continue

                    bridge_source_coeff = bridge_edge.get("source_coeff", None)

                    if bridge_source_coeff is None:
                        continue

                    for second_edge in ainf_by_target.get(bridged_target_key, []):
                        second_replacement = tuple(second_edge["replacement_factors"])
                        second_primitive = self.replacement_edge_context_primitive(second_edge)

                        if second_primitive is None:
                            continue

                        middle_coeff = simplify_coeff(bridge_source_coeff * first_source_inverse)
                        primitive = (
                            (-first_source_inverse) * first_primitive
                            + first_source_inverse * bridge_primitive
                            + middle_coeff * second_primitive
                        )
                        relation = self.ainf_edge_relation_data(
                            first_replacement,
                            second_replacement,
                            primitive
                        )

                        if relation is None:
                            continue

                        edge_key = (
                            "ainf_bridge_compatibility",
                            self.mp_factor_tuple_key(first_replacement),
                            self.mp_factor_tuple_key(second_replacement),
                            self.bridge_edge_key(first_edge),
                            self.bridge_edge_key(bridge_edge),
                            self.bridge_edge_key(second_edge),
                            self.element_key_up_to_sign(relation["primitive"]),
                        )

                        if edge_key in seen:
                            continue

                        seen.add(edge_key)
                        edges.append({
                            "replacement_type": "ainf_bridge_compatibility",
                            "target_product": MPProduct(self, first_replacement),
                            "replacement_product": MPProduct(self, second_replacement),
                            "target_factors": first_replacement,
                            "replacement_factors": second_replacement,
                            "target_key": self.mp_factor_tuple_key(first_replacement),
                            "replacement_key": self.mp_factor_tuple_key(second_replacement),
                            "bridge_coeff": relation["bridge_coeff"],
                            "source_coeff": relation["source_coeff"],
                            "bridge_cell": relation["primitive"],
                            "bridge_expression": None,
                            "compatibility_edges": (
                                self.bridge_edge_key(first_edge),
                                self.bridge_edge_key(bridge_edge),
                                self.bridge_edge_key(second_edge),
                            ),
                            "compatibility_first_edge": first_edge,
                            "compatibility_bridge_edge": bridge_edge,
                            "compatibility_second_edge": second_edge,
                            "compatibility_bridged_target": bridged_target,
                            "relation_replacement_coeff": relation["relation_replacement_coeff"],
                        })

        return edges

    def replacement_edge_records(self, only_bridge_cell=None, excluded_bridge_cells=None, include_ainf=True):
        """
        Return all replacement edges used by primitive/interval searches.
        """

        bridge_edges = self.bridge_replacement_edge_records(
            only_bridge_cell=only_bridge_cell,
            excluded_bridge_cells=excluded_bridge_cells
        )
        edges = list(bridge_edges)

        if include_ainf and only_bridge_cell is None:
            ainf_edges = self.ainf_replacement_edge_records()
            edges.extend(ainf_edges)
            edges.extend(self.compatible_ainf_bridge_replacement_edge_records(
                bridge_edges=bridge_edges,
                ainf_edges=ainf_edges
            ))

        return edges

    def mp_cell_factor_records(self):
        """
        Return resolved non-bridge MP cells as coefficient-stripped products.
        """

        raw = []

        if hasattr(self, "attachment_history"):
            for entry in self.attachment_history:
                cell = entry.get("cell", None)
                expression = entry.get("expression", None)

                if cell is not None and expression is not None:
                    raw.append((cell, expression, entry.get("kind", None)))

        if hasattr(self, "v") and hasattr(self.v, "cells"):
            for cell in self.v.cells.values():
                expression = getattr(cell, "index", None)

                if expression is not None:
                    raw.append((cell, expression, getattr(cell, "cell_kind", None)))

        records = []
        seen = set()

        for cell, expression, kind in raw:
            if not isinstance(cell, Arrow):
                continue

            if isinstance(expression, MPElement) and len(expression.terms) >= 2:
                continue

            try:
                terms = self.flatten_m2_only_terms(expression)
            except Exception:
                continue

            for coeff, factors in terms:
                try:
                    canonical_factors = self.canonical_mp_factors(factors)
                except Exception:
                    continue

                if len(canonical_factors) < 2:
                    continue

                record_key = (id(cell), self.mp_factor_tuple_key(canonical_factors))

                if record_key in seen:
                    continue

                seen.add(record_key)
                records.append({
                    "cell": cell,
                    "expression": expression,
                    "kind": kind,
                    "coefficient": coeff,
                    "factors": canonical_factors,
                })

        return records

    def effective_right_overlaps(self, current_factors, target_factors):
        """
        Effective right overlaps: suffix(current) = prefix(target), append nonempty.
        """

        current_factors = tuple(current_factors)
        target_factors = tuple(target_factors)
        overlaps = []
        max_overlap = min(len(current_factors), len(target_factors) - 1)

        for overlap in range(1, max_overlap + 1):
            if self.mp_factors_equal(current_factors[-overlap:], target_factors[:overlap]):
                overlaps.append({
                    "overlap": overlap,
                    "concatenated_factors": target_factors[overlap:],
                })

        return overlaps

    def effective_left_overlaps(self, current_factors, target_factors):
        """
        Effective left overlaps: suffix(target) = prefix(current), prepend nonempty.
        """

        current_factors = tuple(current_factors)
        target_factors = tuple(target_factors)
        overlaps = []
        max_overlap = min(len(current_factors), len(target_factors) - 1)

        for overlap in range(1, max_overlap + 1):
            if self.mp_factors_equal(target_factors[-overlap:], current_factors[:overlap]):
                overlaps.append({
                    "overlap": overlap,
                    "concatenated_factors": target_factors[:len(target_factors) - overlap],
                })

        return overlaps

    def minimal_effective_actions(self, current_factors, records, target_factors_getter):
        """
        Keep only shortest left/right effective actions from one state.

        This prevents a bridge/cell that already acts after appending x3
        from also being counted after appending x3*x4.
        """

        right_actions = []
        left_actions = []

        for record in records:
            target_factors = tuple(target_factors_getter(record))

            for overlap_data in self.effective_right_overlaps(current_factors, target_factors):
                right_actions.append({
                    "record": record,
                    "target_factors": target_factors,
                    **overlap_data,
                })

            for overlap_data in self.effective_left_overlaps(current_factors, target_factors):
                left_actions.append({
                    "record": record,
                    "target_factors": target_factors,
                    **overlap_data,
                })

        def keep_minimal(actions):
            if not actions:
                return []

            min_length = min(len(action["concatenated_factors"]) for action in actions)
            return [
                action for action in actions
                if len(action["concatenated_factors"]) == min_length
            ]

        return keep_minimal(right_actions), keep_minimal(left_actions)

    def susceptible_mp_products_from_factors(self, factors=None, verbose=False, initial_states=None, excluded_bridge_cells=None):
        """
        Find ambient products reached by effective bridge expansions/terminations.

        A state keeps both the current bridge-replaced product and the ambient
        unreplaced product obtained by the actual concatenations. Terminations
        are reported as ambient products, because those are the products whose
        primitive search should see the whole bridge chain.
        """

        if factors is None:
            base_factors = ()
        else:
            try:
                base_factors = self.canonical_mp_factors(factors)
            except Exception:
                return []

        if not base_factors and not initial_states:
            return []

        bridge_edges = self.replacement_edge_records(
            excluded_bridge_cells=excluded_bridge_cells
        )
        mp_cells = self.mp_cell_factor_records()
        products = []
        seen_products = set()
        warned_repeats = set()

        def remember_product(product_factors, termination_record, path_records, state_base_factors):
            product_factors = tuple(product_factors)
            product = MPProduct(self, product_factors)
            key = self.mp_product_key(product)

            if key in seen_products:
                return

            seen_products.add(key)
            products.append({
                "product": product,
                "product_factors": product_factors,
                "termination": termination_record,
                "bridge_expansion_path": list(path_records),
                "base_factors": tuple(state_base_factors),
            })

        if initial_states is None:
            queue = [(
                base_factors,
                base_factors,
                [],
                frozenset(),
                base_factors
            )]
        else:
            queue = []

            for state in initial_states:
                queue.append((
                    tuple(state.get("current_factors", ())),
                    tuple(state.get("ambient_factors", ())),
                    list(state.get("path_records", ())),
                    frozenset(state.get("used_actions", frozenset())),
                    tuple(state.get("base_factors", base_factors)),
                ))

        seen_states = {
            (
                self.mp_factor_tuple_key(current_factors),
                self.mp_factor_tuple_key(ambient_factors),
                used_actions,
            )
            for current_factors, ambient_factors, path_records, used_actions, state_base_factors in queue
        }

        while queue:
            current_factors, ambient_factors, path_records, used_actions, state_base_factors = queue.pop(0)

            zero_right_termination = False
            zero_left_termination = False

            if path_records:
                for cell_record in mp_cells:
                    target_factors = tuple(cell_record["factors"])

                    if len(target_factors) <= len(current_factors):
                        if self.mp_factors_equal(current_factors[-len(target_factors):], target_factors):
                            zero_right_termination = True
                            remember_product(tuple(ambient_factors), {
                                "side": "right",
                                "cell_record": cell_record,
                                "overlap": len(target_factors),
                                "concatenated_factors": (),
                                "current_factors": tuple(current_factors),
                                "zero_extra_concatenation": True,
                            }, path_records, state_base_factors)

                        if self.mp_factors_equal(current_factors[:len(target_factors)], target_factors):
                            zero_left_termination = True
                            remember_product(tuple(ambient_factors), {
                                "side": "left",
                                "cell_record": cell_record,
                                "overlap": len(target_factors),
                                "concatenated_factors": (),
                                "current_factors": tuple(current_factors),
                                "zero_extra_concatenation": True,
                            }, path_records, state_base_factors)

            right_terminations, left_terminations = self.minimal_effective_actions(
                current_factors,
                mp_cells,
                lambda record: record["factors"]
            )

            if zero_right_termination:
                right_terminations = []

            if zero_left_termination:
                left_terminations = []

            for action in right_terminations:
                cell_record = action["record"]
                concatenated = tuple(action["concatenated_factors"])
                product_factors = tuple(ambient_factors) + concatenated
                remember_product(product_factors, {
                    "side": "right",
                    "cell_record": cell_record,
                    "overlap": action["overlap"],
                    "concatenated_factors": concatenated,
                    "current_factors": tuple(current_factors),
                }, path_records, state_base_factors)

            for action in left_terminations:
                cell_record = action["record"]
                concatenated = tuple(action["concatenated_factors"])
                product_factors = concatenated + tuple(ambient_factors)
                remember_product(product_factors, {
                    "side": "left",
                    "cell_record": cell_record,
                    "overlap": action["overlap"],
                    "concatenated_factors": concatenated,
                    "current_factors": tuple(current_factors),
                }, path_records, state_base_factors)

            right_expansions, left_expansions = self.minimal_effective_actions(
                current_factors,
                bridge_edges,
                lambda record: record["target_factors"]
            )

            for action in right_expansions:
                    edge = action["record"]
                    overlap = action["overlap"]
                    concatenated = tuple(action["concatenated_factors"])
                    temporary_length = len(current_factors) + len(concatenated)
                    occurrence_start = len(current_factors) - overlap
                    occurrence_stop = occurrence_start + len(edge["target_factors"])
                    position_from_right = temporary_length - occurrence_stop
                    action_key = (
                        "right",
                        self.mp_factor_tuple_key(concatenated),
                        self.bridge_edge_key(edge),
                        position_from_right,
                    )

                    if action_key in used_actions:
                        if verbose and action_key not in warned_repeats:
                            warned_repeats.add(action_key)
                            print("Warning: skipped repeated right bridge expansion; bridge replacement chain may be infinite.")
                        continue

                    keep_prefix = tuple(current_factors[:len(current_factors) - overlap])
                    next_current = keep_prefix + tuple(edge["replacement_factors"])
                    next_ambient = tuple(ambient_factors) + concatenated
                    next_actions = used_actions | {action_key}
                    state_key = (
                        self.mp_factor_tuple_key(next_current),
                        self.mp_factor_tuple_key(next_ambient),
                        next_actions,
                    )

                    if state_key in seen_states:
                        continue

                    seen_states.add(state_key)
                    path_record = {
                        "side": "right",
                        "edge": edge,
                        "overlap": overlap,
                        "replacement_position_from_right": position_from_right,
                        "replacement_span_before": (occurrence_start, occurrence_stop),
                        "concatenated_factors": concatenated,
                        "source_before": tuple(current_factors),
                        "source_after": next_current,
                        "ambient_before": tuple(ambient_factors),
                        "ambient_after": next_ambient,
                    }
                    queue.append((
                        next_current,
                        next_ambient,
                        path_records + [path_record],
                        next_actions,
                        state_base_factors,
                    ))

            for action in left_expansions:
                    edge = action["record"]
                    overlap = action["overlap"]
                    concatenated = tuple(action["concatenated_factors"])
                    occurrence_start = 0
                    occurrence_stop = len(edge["target_factors"])
                    position_from_left = occurrence_start
                    action_key = (
                        "left",
                        self.mp_factor_tuple_key(concatenated),
                        self.bridge_edge_key(edge),
                        position_from_left,
                    )

                    if action_key in used_actions:
                        if verbose and action_key not in warned_repeats:
                            warned_repeats.add(action_key)
                            print("Warning: skipped repeated left bridge expansion; bridge replacement chain may be infinite.")
                        continue

                    next_current = tuple(edge["replacement_factors"]) + tuple(current_factors[overlap:])
                    next_ambient = concatenated + tuple(ambient_factors)
                    next_actions = used_actions | {action_key}
                    state_key = (
                        self.mp_factor_tuple_key(next_current),
                        self.mp_factor_tuple_key(next_ambient),
                        next_actions,
                    )

                    if state_key in seen_states:
                        continue

                    seen_states.add(state_key)
                    path_record = {
                        "side": "left",
                        "edge": edge,
                        "overlap": overlap,
                        "replacement_position_from_left": position_from_left,
                        "replacement_span_before": (occurrence_start, occurrence_stop),
                        "concatenated_factors": concatenated,
                        "source_before": tuple(current_factors),
                        "source_after": next_current,
                        "ambient_before": tuple(ambient_factors),
                        "ambient_after": next_ambient,
                    }
                    queue.append((
                        next_current,
                        next_ambient,
                        path_records + [path_record],
                        next_actions,
                        state_base_factors,
                    ))

        return products

    def bridge_seed_initial_states_from_cell(self, cell):
        """
        Seed susceptible searches from a newly attached bridge cell.

        If the bridge has sides A and B, each oriented side can overlap an
        existing MP-cell product on the right or on the left. The seed state
        is the minimal ambient product after that concatenation, with the new
        bridge replacement already performed once.
        """

        bridge_edges = self.bridge_replacement_edge_records(only_bridge_cell=cell)

        if not bridge_edges:
            return []

        mp_cells = self.mp_cell_factor_records()
        states = []
        seen = set()

        def remember_state(side, edge, cell_record, overlap, concatenated):
            base_factors = tuple(cell_record["factors"])
            concatenated = tuple(concatenated)

            if side == "right":
                temporary_length = len(base_factors) + len(concatenated)
                occurrence_start = len(base_factors) - overlap
                occurrence_stop = occurrence_start + len(edge["target_factors"])
                position_from_right = temporary_length - occurrence_stop
                action_key = (
                    "right",
                    self.mp_factor_tuple_key(concatenated),
                    self.bridge_edge_key(edge),
                    position_from_right,
                )
                keep_prefix = tuple(base_factors[:len(base_factors) - overlap])
                current_after = keep_prefix + tuple(edge["replacement_factors"])
                ambient_after = base_factors + concatenated
                replacement_position_key = {
                    "replacement_position_from_right": position_from_right,
                }
            else:
                occurrence_start = 0
                occurrence_stop = len(edge["target_factors"])
                position_from_left = occurrence_start
                action_key = (
                    "left",
                    self.mp_factor_tuple_key(concatenated),
                    self.bridge_edge_key(edge),
                    position_from_left,
                )
                current_after = tuple(edge["replacement_factors"]) + tuple(base_factors[overlap:])
                ambient_after = concatenated + base_factors
                replacement_position_key = {
                    "replacement_position_from_left": position_from_left,
                }

            state_key = (
                side,
                id(cell_record["cell"]),
                self.bridge_edge_key(edge),
                self.mp_factor_tuple_key(base_factors),
                self.mp_factor_tuple_key(ambient_after),
                action_key,
            )

            if state_key in seen:
                return

            seen.add(state_key)
            path_record = {
                "side": side,
                "edge": edge,
                "overlap": overlap,
                "replacement_span_before": (occurrence_start, occurrence_stop),
                "concatenated_factors": concatenated,
                "source_before": base_factors,
                "source_after": current_after,
                "ambient_before": base_factors,
                "ambient_after": ambient_after,
                "initial_bridge_seed": True,
                "base_cell_record": cell_record,
                **replacement_position_key,
            }
            states.append({
                "current_factors": current_after,
                "ambient_factors": ambient_after,
                "path_records": [path_record],
                "used_actions": frozenset({action_key}),
                "base_factors": base_factors,
                "seed_bridge_cell": cell,
                "seed_edge": edge,
                "base_cell_record": cell_record,
            })

        for edge in bridge_edges:
            target_factors = tuple(edge["target_factors"])

            for cell_record in mp_cells:
                base_factors = tuple(cell_record["factors"])

                for action in self.effective_right_overlaps(base_factors, target_factors):
                    remember_state(
                        "right",
                        edge,
                        cell_record,
                        action["overlap"],
                        action["concatenated_factors"]
                    )

                for action in self.effective_left_overlaps(base_factors, target_factors):
                    remember_state(
                        "left",
                        edge,
                        cell_record,
                        action["overlap"],
                        action["concatenated_factors"]
                    )

        return states

    def generate_massey_products_from_bridge_cell(self, cell, record=True, verbose=True):
        """
        Generate MP names whose first bridge expansion is the given bridge.
        """

        seed_states = self.bridge_seed_initial_states_from_cell(cell)

        if not seed_states:
            return []

        generated = []
        seen_seed_products = set()

        for seed_state in seed_states:
            product = MPProduct(self, tuple(seed_state["ambient_factors"]))
            product_key = self.mp_product_key(product)

            if product_key in seen_seed_products:
                continue

            seen_seed_products.add(product_key)
            result = self.generated_massey_products_from_mp_product(
                product,
                record=record,
                minimal_only=True
            )
            self.remember_generated_mp_product_result(
                result,
                source={
                    "kind": "bridge_seed_product",
                    "product": product,
                    "seed_state": seed_state,
                }
            )
            items = result.get("have_primitives", []) + result.get("no_obvious_primitives", [])

            if verbose and items:
                print("Generated from bridge seed product", product, ":", items)

            generated.extend(items)

        susceptible_products = self.susceptible_mp_products_from_factors(
            None,
            verbose=verbose,
            initial_states=seed_states,
            excluded_bridge_cells={cell, id(cell)}
        )

        for product_record in susceptible_products:
            result = self.generated_massey_products_from_mp_product(
                product_record["product"],
                record=record,
                minimal_only=True
            )
            self.remember_generated_mp_product_result(result, source=product_record)
            items = result.get("have_primitives", []) + result.get("no_obvious_primitives", [])

            if verbose and items:
                print("Generated from bridge-seeded susceptible product", product_record["product"], ":", items)

            generated.extend(items)

        return self.dedupe_massey_products(generated)

    def generate_massey_products_from_susceptible_products(self, factors, record=True, verbose=True):
        """
        Generate product-level MP names from effective bridge-chain products.
        """

        generated = []

        try:
            base_product = MPProduct(self, self.canonical_mp_factors(factors))
        except Exception:
            base_product = None

        if base_product is not None:
            base_result = self.generated_massey_products_from_mp_product(
                base_product,
                record=record,
                minimal_only=True
            )
            self.remember_generated_mp_product_result(
                base_result,
                source={
                    "kind": "base_product",
                    "product": base_product,
                }
            )
            generated.extend(
                base_result.get("have_primitives", [])
                + base_result.get("no_obvious_primitives", [])
            )

        susceptible_products = self.susceptible_mp_products_from_factors(
            factors,
            verbose=verbose
        )

        for product_record in susceptible_products:
            result = self.generated_massey_products_from_mp_product(
                product_record["product"],
                record=record,
                minimal_only=True
            )
            self.remember_generated_mp_product_result(result, source=product_record)
            items = result.get("have_primitives", []) + result.get("no_obvious_primitives", [])

            if verbose and items:
                print("Generated from susceptible product", product_record["product"], ":", items)

            generated.extend(items)

        return self.dedupe_massey_products(generated)

    def generate_massey_products_from_factors(self, factors, record=True, verbose=True, coefficient=1):
        """
        Generate MP candidates from one coefficient-stripped factor list.
        """

        generated = []
        candidates = self.mp_extension_candidates()

        if not factors:
            return []

        if len(factors) == 1 and isinstance(factors[0], MasseyProduct):
            M = factors[0]
            inputs = list(self.normalize_mp_inputs(M.inputs))

            if len(inputs) < 3:
                return []

            m = len(inputs)
            f1_to_fm_minus_1 = inputs[:m - 1]
            f2_to_fm = inputs[1:]

            for q in candidates:
                if self.can_define_mp(q, *f1_to_fm_minus_1):
                    left_test = self.mp(q, *f1_to_fm_minus_1)
                    left_primitives = self.find_mp_primitives(left_test, record=record)

                    if left_primitives:
                        new_M = self.try_generate_mp(q, *inputs, record=record, verbose=verbose)

                        if new_M is not None:
                            generated.append(new_M)

                            if verbose:
                                print("Generated left extension:", new_M)

                if self.can_define_mp(*f2_to_fm, q):
                    right_test = self.mp(*f2_to_fm, q)
                    right_primitives = self.find_mp_primitives(right_test, record=record)

                    if right_primitives:
                        new_M = self.try_generate_mp(*inputs, q, record=record, verbose=verbose)

                        if new_M is not None:
                            generated.append(new_M)

                            if verbose:
                                print("Generated right extension:", new_M)

            return self.dedupe_massey_products(generated)

        g = factors
        s = len(g)

        if s < 2:
            return []

        generated.extend(
            self.generate_massey_products_from_susceptible_products(
                g,
                record=record,
                verbose=verbose
            )
        )

        for q in candidates:
            for i in range(1, s + 1):
                test_factors = [q] + g[:i]
                test_expr = self.multiply_mp_factors(test_factors)
                primitive_candidates = self.find_mp_primitives(test_expr, record=record)
                primitive_candidates = [
                    p for p in primitive_candidates
                    if self.primitive_uses_nontrivial_block(p, q)
                ]

                if primitive_candidates:
                    new_M = self.try_generate_mp(q, *g, record=record, verbose=verbose)

                    if new_M is not None:
                        generated.append(new_M)

                        if verbose:
                            print("Generated left reducible extension:", new_M)
                    else:
                        induced_items = self.induced_massey_products_from_factor_product(
                            [q] + g,
                            record=record,
                            minimal_only=True
                        )
                        generated.extend(induced_items)

                        if verbose and induced_items:
                            print("Generated induced left 3-fold products:", induced_items)

                    break

            for i in range(0, s):
                test_factors = g[i:] + [q]
                test_expr = self.multiply_mp_factors(test_factors)
                primitive_candidates = self.find_mp_primitives(test_expr, record=record)
                primitive_candidates = [
                    p for p in primitive_candidates
                    if self.primitive_uses_nontrivial_block(p, q)
                ]

                if primitive_candidates:
                    new_M = self.try_generate_mp(*g, q, record=record, verbose=verbose)

                    if new_M is not None:
                        generated.append(new_M)

                        if verbose:
                            print("Generated right reducible extension:", new_M)
                    else:
                        induced_items = self.induced_massey_products_from_factor_product(
                            g + [q],
                            record=record,
                            minimal_only=True
                        )
                        generated.extend(induced_items)

                        if verbose and induced_items:
                            print("Generated induced right 3-fold products:", induced_items)

                    break

        return self.dedupe_massey_products(generated)

    def generate_massey_products_from_cell(self, cell, record=True, verbose=True):
        """
        Generate possible new Massey products from one attached cell.

        Principle:

        Case 1:
            d(cell) = mp(f1, ..., fm), an irreducible higher block.

            Search for primitives of:
                mp(?, f1, ..., f_{m-1})

            If such primitives exist, generate:
                mp(?, f1, ..., f_m)

            Similarly, search for primitives of:
                mp(f2, ..., f_m, ?)

            If such primitives exist, generate:
                mp(f1, ..., f_m, ?)

        Case 2:
            d(cell) = g1 * g2 * ... * gs, a reducible product.

            For each outside factor q, and for each prefix g1*...*gi,
            search for primitives of:
                q * g1 * ... * gi

            If such a primitive genuinely uses the product and not merely q,
            generate:
                mp(q, g1, ..., gs)

            Similarly on the right:
                gi * ... * gs * q

            and generate:
                mp(g1, ..., gs, q)

        Important convention:
            mp(a, b) is treated as ordinary multiplication.
            mp(a, b, c) and higher are treated as irreducible blocks.
        """

        dcell = self.d(cell)

        generated = []
        term_factor_lists = self.flatten_m2_only_terms(dcell)
        is_bridge_cell = (
            getattr(cell, "cell_kind", None) == "bridge"
            or any(
                bridge_cell is cell
                for bridge_cell in getattr(self, "bridge_cells", {}).values()
            )
        )

        if is_bridge_cell:
            generated.extend(
                self.generate_massey_products_from_bridge_cell(
                    cell,
                    record=record,
                    verbose=verbose
                )
            )

        if len(term_factor_lists) != 1:
            for coeff, factors in term_factor_lists:
                generated.extend(
                    self.generate_massey_products_from_factors(
                        factors,
                        record=record,
                        verbose=verbose,
                        coefficient=coeff
                    )
                )

            return self.dedupe_massey_products(generated)

        if not term_factor_lists:
            return []

        coeff, factors = term_factor_lists[0]

        candidates = self.mp_extension_candidates()

        if not factors:
            return []

        # ------------------------------------------------------------
        # Case 1:
        # d(cell) is one irreducible higher Massey product mp(f1,...,fm)
        # ------------------------------------------------------------
        if len(factors) == 1 and isinstance(factors[0], MasseyProduct):
            M = factors[0]
            inputs = list(self.normalize_mp_inputs(M.inputs))

            if len(inputs) < 3:
                return []

            m = len(inputs)

            f1_to_fm_minus_1 = inputs[:m - 1]
            f2_to_fm = inputs[1:]

            for q in candidates:
                # ----------------------------------------------------
                # Left extension:
                #
                # If mp(q, f1, ..., f_{m-1}) is defined
                # and has a primitive, then generate
                # mp(q, f1, ..., f_m).
                # ----------------------------------------------------
                if self.can_define_mp(q, *f1_to_fm_minus_1):
                    left_test = self.mp(q, *f1_to_fm_minus_1)

                    left_primitives = self.find_mp_primitives(
                        left_test,
                        record=record
                    )

                    if left_primitives:
                        new_M = self.try_generate_mp(
                            q,
                            *inputs,
                            record=record,
                            verbose=verbose
                        )

                        if new_M is not None:
                            generated.append(new_M)

                            if verbose:
                                print("Generated left extension:", new_M)

                # ----------------------------------------------------
                # Right extension:
                #
                # If mp(f2, ..., f_m, q) is defined
                # and has a primitive, then generate
                # mp(f1, ..., f_m, q).
                # ----------------------------------------------------
                if self.can_define_mp(*f2_to_fm, q):
                    right_test = self.mp(*f2_to_fm, q)

                    right_primitives = self.find_mp_primitives(
                        right_test,
                        record=record
                    )

                    if right_primitives:
                        new_M = self.try_generate_mp(
                            *inputs,
                            q,
                            record=record,
                            verbose=verbose
                        )

                        if new_M is not None:
                            generated.append(new_M)

                            if verbose:
                                print("Generated right extension:", new_M)

            # Remove duplicates
            out = []
            seen = set()

            for M in generated:
                key = repr(M)
                if key not in seen:
                    seen.add(key)
                    out.append(M)

            return out

        # ------------------------------------------------------------
        # Case 2:
        # d(cell) is reducible: g1 * g2 * ... * gs
        # ------------------------------------------------------------
        g = factors
        s = len(g)

        if s < 2:
            return []

        generated.extend(
            self.generate_massey_products_from_susceptible_products(
                g,
                record=record,
                verbose=verbose
            )
        )

        for q in candidates:
            # --------------------------------------------------------
            # Left reducible extension:
            #
            # Search q*g1*...*gi for all i.
            # If one has a nontrivial primitive, generate
            # mp(q, g1, ..., gs).
            # --------------------------------------------------------
            for i in range(1, s + 1):
                test_factors = [q] + g[:i]
                test_expr = self.multiply_mp_factors(test_factors)

                primitive_candidates = self.find_mp_primitives(
                    test_expr,
                    record=record
                )

                primitive_candidates = [
                    p for p in primitive_candidates
                    if self.primitive_uses_nontrivial_block(p, q)
                ]

                if primitive_candidates:
                    new_M = self.try_generate_mp(
                        q,
                        *g,
                        record=record,
                        verbose=verbose
                    )

                    if new_M is not None:
                        generated.append(new_M)

                        if verbose:
                            print("Generated left reducible extension:", new_M)
                    else:
                        induced_items = self.induced_massey_products_from_factor_product(
                            [q] + g,
                            record=record,
                            minimal_only=True
                        )
                        generated.extend(induced_items)

                        if verbose and induced_items:
                            print("Generated induced left 3-fold products:", induced_items)

                    break

            # --------------------------------------------------------
            # Right reducible extension:
            #
            # Search gi*...*gs*q for all i.
            # If one has a nontrivial primitive, generate
            # mp(g1, ..., gs, q).
            # --------------------------------------------------------
            for i in range(0, s):
                test_factors = g[i:] + [q]
                test_expr = self.multiply_mp_factors(test_factors)

                primitive_candidates = self.find_mp_primitives(
                    test_expr,
                    record=record
                )

                primitive_candidates = [
                    p for p in primitive_candidates
                    if self.primitive_uses_nontrivial_block(p, q)
                ]

                if primitive_candidates:
                    new_M = self.try_generate_mp(
                        *g,
                        q,
                        record=record,
                        verbose=verbose
                    )

                    if new_M is not None:
                        generated.append(new_M)

                        if verbose:
                            print("Generated right reducible extension:", new_M)
                    else:
                        induced_items = self.induced_massey_products_from_factor_product(
                            g + [q],
                            record=record,
                            minimal_only=True
                        )
                        generated.extend(induced_items)

                        if verbose and induced_items:
                            print("Generated induced right 3-fold products:", induced_items)

                    break

        # Remove duplicates
        out = []
        seen = set()

        for M in generated:
            key = repr(M)
            if key not in seen:
                seen.add(key)
                out.append(M)

        return out

    def generate_massey_products(
        self,
        cells=None,
        record=True,
        verbose=False,
        split_by_primitives=True
    ):
        """
        Generate Massey product candidates from the current resolved situation.

        Sources:
            1. Ordinary attached cells in self.cells, if they exist.
            2. Resolved Massey products in self.resolved_massey_products.

        By default this is quiet and returns two groups:

            {
                "have_primitives": [...],
                "no_obvious_primitives": [...]
            }

        Use split_by_primitives=False to return one flat list.
        """

        generated = []

        # ------------------------------------------------------------
        # Source 1: ordinary cells, if any
        # ------------------------------------------------------------
        cells = self.generation_cells(cells)

        for cell in cells:
            new_items = self.generate_massey_products_from_cell(
                cell,
                record=record,
                verbose=verbose
            )

            generated.extend(new_items)

        # ------------------------------------------------------------
        # Source 2: resolved Massey products
        # ------------------------------------------------------------
        if hasattr(self, "resolved_massey_products"):
            for key, primitive in list(self.resolved_massey_products.items()):
                new_items = self.generate_massey_products_from_resolved_mp_key(
                    key,
                    primitive=primitive,
                    record=record,
                    verbose=verbose
                )

                generated.extend(new_items)

        # Remove duplicates
        out = []
        seen = set()

        for M in generated:
            k = repr(M)

            if k not in seen:
                seen.add(k)
                out.append(M)

        if not split_by_primitives:
            return out

        have_primitives = []
        no_obvious_primitives = []

        obvious_primitive_data = {}
        have_keys = set()
        no_obvious_keys = set()

        def remember_generated_item(target, item):
            key = self.generation_item_key(item)

            if target == "have":
                if key not in have_keys:
                    have_keys.add(key)
                    have_primitives.append(item)
            else:
                if key not in no_obvious_keys:
                    no_obvious_keys.add(key)
                    no_obvious_primitives.append(item)

        for M in out:
            if not isinstance(M, MasseyProduct):
                item_key = self.generation_item_key(M)
                product_data = getattr(self, "generated_mp_expression_data", {}).get(item_key, {})
                obvious_primitive_data[item_key] = product_data

                if product_data.get("classification") == "have_primitives":
                    remember_generated_item("have", M)
                else:
                    remember_generated_item("no_obvious", M)

                continue

            primitive_data = self.classify_massey_product_primitives(
                M,
                record=record,
                minimal_only=True
            )
            obvious_primitive_data[self.mp_key(*M.inputs)] = primitive_data

            if primitive_data["has_obvious_primitive"]:
                obvious_M = primitive_data.get("obvious_massey_product", None)

                if obvious_M is not None:
                    obvious_key = self.mp_key(*obvious_M.inputs)
                    obvious_primitive_data[obvious_key] = {
                        **primitive_data,
                        "source_generated_massey_product": M,
                    }
                    remember_generated_item("have", obvious_M)
                else:
                    remember_generated_item("have", M)
            else:
                induced = primitive_data.get("induced_massey_products", [])

                if induced:
                    for induced_M in induced:
                        induced_key = self.mp_key(*induced_M.inputs)
                        obvious_primitive_data[induced_key] = {
                            **primitive_data,
                            "source_generated_massey_product": M,
                        }
                        remember_generated_item("no_obvious", induced_M)
                else:
                    remember_generated_item("no_obvious", M)

        result = {
            "have_primitives": have_primitives,
            "no_obvious_primitives": no_obvious_primitives,
            "primitive_data": obvious_primitive_data,
        }

        print("Have primitives:", have_primitives)
        print("No obvious primitives:", no_obvious_primitives)

        return result

    def generate_massey_products_from_resolved_mp_key(
        self,
        key,
        primitive=None,
        record=True,
        verbose=False
    ):
        """
        Generate new Massey products from one already resolved MP key.

        If key corresponds to mp(f1, ..., fm), then its primitive exists.
        Therefore we try to generate:

            mp(q, f1, ..., fm)

        and

            mp(f1, ..., fm, q)

        for all extension candidates q.

        Example:
            resolved mp(x,x)  ==>  generate mp(x,x,x).
        """

        generated = []

        if isinstance(key, tuple):
            inputs = list(key)
        else:
            if verbose:
                print("Cannot read resolved MP key:", key)
            return []

        if not inputs:
            return []

        candidates = self.mp_extension_candidates()

        for q in candidates:
            # Left extension: mp(q, f1, ..., fm)
            new_M = self.try_generate_mp(
                q,
                *inputs,
                record=record,
                verbose=verbose
            )

            if new_M is not None:
                generated.append(new_M)

                if verbose:
                    print("Generated from resolved MP, left extension:", new_M)

            # Right extension: mp(f1, ..., fm, q)
            new_M = self.try_generate_mp(
                *inputs,
                q,
                record=record,
                verbose=verbose
            )

            if new_M is not None:
                generated.append(new_M)

                if verbose:
                    print("Generated from resolved MP, right extension:", new_M)

        # Remove duplicates
        out = []
        seen = set()

        for M in generated:
            k = repr(M)

            if k not in seen:
                seen.add(k)
                out.append(M)

        return out

    def ensure_mp_cell_history(self):
        """
        Make sure the MP resolving-cell history exists.

        If history was not created when earlier MP cells were resolved,
        backfill it from resolved_massey_products using dictionary order.
        """

        if not hasattr(self, "mp_cell_history"):
            self.mp_cell_history = []

        if hasattr(self, "resolved_massey_products"):
            for key in self.resolved_massey_products.keys():
                if key not in self.mp_cell_history:
                    self.mp_cell_history.append(key)

    def rollback_mp_cells_to_key(self, key, verbose=True):
        """
        Roll back MP resolving cells to the stage before `key` was added.

        If history is:

            [k1, k2, k3]

        then rollback_mp_cells_to_key(k2) deletes k2 and k3.

        This deletes resolving cells/primitive records, not the underlying
        Massey product expressions themselves.
        """

        self.ensure_mp_cell_history()

        if key not in self.mp_cell_history:
            if verbose:
                print("This MP resolving cell is not in the history:")
                print("   ", key)
            return {
                "deleted": [],
                "remaining_history": list(self.mp_cell_history),
            }

        start = self.mp_cell_history.index(key)

        keys_to_delete = self.mp_cell_history[start:]

        deleted = []

        for k in reversed(keys_to_delete):
            primitive = None

            if hasattr(self, "resolved_massey_products"):
                primitive = self.resolved_massey_products.pop(k, None)
            self.forget_mp_cell_attachment_record(k)

            deleted.append((k, primitive))

            if verbose:
                if primitive is not None:
                    print("Deleted resolving cell:")
                    print("   ", primitive)
                    print("for:")
                    print("   ", self.mp(*k))
                else:
                    print("Removed stale history key:")
                    print("   ", k)

        self.mp_cell_history = self.mp_cell_history[:start]

        # Clean generated MP records which are no longer definable.
        if hasattr(self, "cleanup_invalid_generated_mps"):
            removed_stale_mps = self.cleanup_invalid_generated_mps()
        else:
            removed_stale_mps = []

        if verbose and removed_stale_mps:
            print("Removed stale generated MP records:")
            for M in removed_stale_mps:
                print("   ", M)

        return {
            "deleted": deleted,
            "removed_stale_mps": removed_stale_mps,
            "remaining_history": list(self.mp_cell_history),
        }

    def rollback_mp_cell(self, P=None, verbose=True):
        """
        Explicit rollback deletion.

        This deletes the requested resolving cell and all resolving cells
        added after it.

        Use this only when you intentionally want to undo to an earlier stage.
        """

        self.ensure_mp_cell_history()

        if P is not None:
            if isinstance(P, str):
                try:
                    P = eval(P, self.mp_eval_environment())
                except Exception as e:
                    print("Could not understand this expression.")
                    print("Python error:", e)
                    return None

            key = self.find_mp_history_key_from_target(P)

            if key is None:
                print("No resolving cell found for:")
                print("   ", P)
                print("Remember: rollback applies to resolving cells, not arbitrary Massey products.")
                return None

            return self.rollback_mp_cells_to_key(
                key,
                verbose=verbose
            )

        while True:
            expr = input("Type resolving cell to roll back to, or type ok to finish: ")

            if expr == "ok":
                break

            try:
                P = eval(expr, self.mp_eval_environment())
            except Exception as e:
                print("Could not understand this expression.")
                print("Python error:", e)
                continue

            key = self.find_mp_history_key_from_target(P)

            if key is None:
                print("No resolving cell found for:")
                print("   ", P)
                print("Remember: rollback applies to resolving cells, not arbitrary Massey products.")
                continue

            self.rollback_mp_cells_to_key(
                key,
                verbose=verbose
            )

    def find_mp_history_key_from_target(self, target):
        """
        Find the MP key corresponding to a deletion/rollback target.

        Accepted targets:
            Q.mp(...)
            Q.mp(x)*Q.mp(y)
            x*y
            v[Q.mp(...)]
            ordinary Arrow/Path/monomial Element

        This returns the key of the resolving cell to roll back.
        """

        self.ensure_mp_cell_history()

        history = getattr(self, "mp_cell_history", [])
        resolved = getattr(self, "resolved_massey_products", {})

        possible_keys = []

        # ------------------------------------------------------------
        # Helper: add possible key if valid
        # ------------------------------------------------------------
        def add_key_from_inputs(inputs):
            try:
                inputs = self.normalize_mp_inputs(tuple(inputs))
                key = self.mp_key(*inputs)
                possible_keys.append(key)
            except Exception:
                pass

        # ------------------------------------------------------------
        # Case 1: target is a MasseyProduct
        # ------------------------------------------------------------
        if isinstance(target, MasseyProduct):
            add_key_from_inputs(target.inputs)

        # ------------------------------------------------------------
        # Case 2: target is an MPProduct, e.g. Q.mp(x)*Q.mp(y)
        # ------------------------------------------------------------
        elif isinstance(target, MPProduct):
            # First try m_2-only flattening.
            try:
                factors = self.flatten_m2_only(target)
                add_key_from_inputs(factors)
            except Exception:
                pass

            # Also try manually extracting length-one MPs.
            try:
                raw_factors = list(target.factors)
                extracted = []

                for f in raw_factors:
                    if isinstance(f, MasseyProduct) and len(f.inputs) == 1:
                        extracted.append(f.inputs[0])
                    else:
                        extracted.append(f)

                add_key_from_inputs(extracted)
            except Exception:
                pass

        # ------------------------------------------------------------
        # Case 3: target is an ordinary Arrow/Path/Element
        # ------------------------------------------------------------
        elif isinstance(target, Arrow) or isinstance(target, Path) or isinstance(target, Element):
            try:
                factors = self.flatten_m2_only(target)
                add_key_from_inputs(factors)
            except Exception:
                pass

            try:
                P = self.path_to_mp_product(target)

                if P is not None:
                    factors = self.flatten_m2_only(P)
                    add_key_from_inputs(factors)
            except Exception:
                pass

        # ------------------------------------------------------------
        # Case 4: target might be the primitive itself, e.g. v[Q.mp(x,y)]
        # ------------------------------------------------------------
        for key, primitive in resolved.items():
            if target == primitive or repr(target) == repr(primitive):
                return key

        # ------------------------------------------------------------
        # Search by exact key in history first
        # ------------------------------------------------------------
        for key in possible_keys:
            if key in history:
                return key

        # ------------------------------------------------------------
        # Search by key in resolved records second
        # This handles the case where history was not recorded correctly.
        # ------------------------------------------------------------
        for key in possible_keys:
            if key in resolved:
                if key not in self.mp_cell_history:
                    self.mp_cell_history.append(key)
                return key

        # ------------------------------------------------------------
        # Final fallback: compare repr of keys.
        # Useful if keys contain equivalent but not identical objects.
        # ------------------------------------------------------------
        possible_reprs = {repr(k) for k in possible_keys}

        for key in history:
            if repr(key) in possible_reprs:
                return key

        for key in resolved:
            if repr(key) in possible_reprs:
                if key not in self.mp_cell_history:
                    self.mp_cell_history.append(key)
                return key

        return None


    def forget_mp_cell_attachment_record(self, key):
        """
        Remove bookkeeping records saying that the resolving cell for key
        has already been attached.

        This is used by undo_latest_mp_cell and rollback_mp_cell.
        It does not touch the formal Massey product itself.
        """

        removed = []

        possible_dict_names = [
            "mp_cells",
            "mp_cell_records",
            "attached_mp_cells",
            "attached_cells",
            "cell_by_mp_key",
            "cells_by_mp_key",
            "mp_cell_by_key",
            "cell_indices",
        ]

        for name in possible_dict_names:
            if hasattr(self, name):
                D = getattr(self, name)

                if isinstance(D, dict) and key in D:
                    removed.append((name, D.pop(key)))

        possible_set_names = [
            "attached_mp_cell_keys",
            "attached_cell_keys",
            "used_cell_indices",
            "used_mp_cell_indices",
        ]

        for name in possible_set_names:
            if hasattr(self, name):
                S = getattr(self, name)

                if isinstance(S, set) and key in S:
                    S.remove(key)
                    removed.append((name, key))

                if isinstance(S, list) and key in S:
                    while key in S:
                        S.remove(key)
                    removed.append((name, key))

        return removed



    
    def debug_mp_delete_target(self, target):
        """
        Debug why a deletion target is or is not found.
        """

        self.ensure_mp_cell_history()

        print("Target:", target)
        print("Target type:", type(target))

        print("\nHistory:")
        for key in self.mp_cell_history:
            print("   ", key, "=>", repr(key))

        print("\nResolved:")
        for key, primitive in getattr(self, "resolved_massey_products", {}).items():
            print("   ", key, "=>", primitive)

        print("\nDetected key:")
        print("   ", self.find_mp_history_key_from_target(target))

In [8]:
def rational_coeff(value):
    """
    Convert a scalar coefficient to an exact rational SymPy value.

    Formal Koszul signs stay as Sign objects. Plain numeric coefficients are
    normalized to SymPy Integers/Rationals so arithmetic never falls back to
    binary floats.
    """

    if isinstance(value, Sign):
        return value

    if isinstance(value, bool):
        return sp.Integer(int(value))

    if isinstance(value, int):
        return sp.Integer(value)

    if isinstance(value, float):
        return sp.Rational(str(value))

    if isinstance(value, str):
        text = value.strip()

        if not text:
            raise TypeError("Empty string is not a rational coefficient.")

        return sp.Rational(text.replace(" ", ""))

    try:
        if value.is_Rational:
            return sp.Rational(value)
    except Exception:
        pass

    try:
        if value.is_Float:
            return sp.Rational(str(value))
    except Exception:
        pass

    try:
        simplified = simplify(value)

        if simplified.is_Rational:
            return sp.Rational(simplified)

        if simplified.is_Float:
            return sp.Rational(str(simplified))
    except Exception:
        pass

    raise TypeError(f"Coefficient {value} is not rational.")


def coeff_key(c):
    """
    Canonical key for coefficients.
    """

    if isinstance(c, Sign):
        return (
            "Sign",
            tuple(
                sorted(
                    (tuple(sorted(k)), str(rational_coeff(v)))
                    for k, v in c.terms.items()
                )
            )
        )

    try:
        return ("Coeff", str(rational_coeff(c)))
    except Exception:
        return ("Coeff", str(c))


In [9]:
class VirtualHomotopy:
    """
    A formal placeholder h[f].

    This represents a homotopy value that has been requested,
    but whose actual primitive has not yet been assigned.
    """

    def __init__(self, Q, f):
        self.Q = Q
        self.f = to_element(f)

    def __repr__(self):
        return f"h[{self.f}]"

    def __str__(self):
        return f"h[{self.f}]"

In [10]:
class Sign:
    """
    Formal rational-linear combinations of Koszul sign monomials.

    A basic formal sign is written eps_x = (-1)^g(x).

    Internally, a term is stored as

        coefficient * product of eps_symbols

    where each eps_symbol squares to 1.

    For example:

        eps_x * eps_x = 1
        eps_x * eps_y = eps_x eps_y
        -eps_x * eps_x = -1
    """

    def __init__(self, terms=None):
        """
        terms should be a dictionary:

            frozenset(symbols) -> rational coefficient

        The empty frozenset represents the scalar 1.
        """

        if terms is None:
            terms = {}

        self.terms = {}

        for key, coeff in terms.items():
            key = frozenset(key)
            coeff = rational_coeff(coeff)

            if coeff != 0:
                self.terms[key] = rational_coeff(self.terms.get(key, 0) + coeff)

        self.terms = {
            key: coeff
            for key, coeff in self.terms.items()
            if rational_coeff(coeff) != 0
        }

    def key(self):
        return (
            "Sign",
            frozenset(
                (symbols, str(coeff))
                for symbols, coeff in self.terms.items()
            )
        )

    def __hash__(self):
        return hash(self.key())

    @staticmethod
    def one():
        return Sign({frozenset(): 1})

    @staticmethod
    def minus_one():
        return Sign({frozenset(): -1})

    @staticmethod
    def zero():
        return Sign({})

    @staticmethod
    def scalar(n):
        return Sign({frozenset(): rational_coeff(n)})

    @staticmethod
    def eps(symbol):
        """
        The formal sign (-1)^g(symbol).

        If symbol has a sign_name() method, use that for display.
        Otherwise use str(symbol).
        """

        if hasattr(symbol, "sign_name"):
            symbol = symbol.sign_name()
        else:
            symbol = str(symbol)

        return Sign({frozenset([symbol]): 1})

    def is_zero(self):
        return len(self.terms) == 0

    def __add__(self, other):
        other = to_sign(other)

        result = dict(self.terms)

        for key, coeff in other.terms.items():
            result[key] = result.get(key, 0) + coeff

        return Sign(result)

    def __radd__(self, other):
        return self + other

    def __neg__(self):
        return Sign({
            key: -coeff
            for key, coeff in self.terms.items()
        })

    def __sub__(self, other):
        return self + (-to_sign(other))

    def __rsub__(self, other):
        return to_sign(other) - self

    def __mul__(self, other):
        try:
            other = to_sign(other)
        except TypeError:
            return NotImplemented

        result = {}

        for key1, coeff1 in self.terms.items():
            for key2, coeff2 in other.terms.items():

                # eps_x^2 = 1, so multiplication is symmetric difference
                new_key = key1.symmetric_difference(key2)

                result[new_key] = result.get(new_key, 0) + coeff1 * coeff2

        return Sign(result)

    def __rmul__(self, other):
        try:
            other = to_sign(other)
        except TypeError:
            return NotImplemented

        return other * self

    def __eq__(self, other):
        try:
            other = to_sign(other)
        except TypeError:
            return False

        return self.terms == other.terms

    def __repr__(self):
        if self.is_zero():
            return "0"

        pieces = []

        for key, coeff in sorted(self.terms.items(), key=lambda item: sorted(item[0])):
            if key == frozenset():
                pieces.append(str(coeff))
                continue

            sign_part = "*".join(f"(-1)^g({s})" for s in sorted(key))

            if coeff == 1:
                pieces.append(sign_part)
            elif coeff == -1:
                pieces.append("-" + sign_part)
            else:
                pieces.append(f"{coeff}*{sign_part}")

        result = pieces[0]

        for piece in pieces[1:]:
            if piece.startswith("-"):
                result += " - " + piece[1:]
            else:
                result += " + " + piece

        return result


def to_sign(x):
    """
    Convert rational scalar coefficients into Sign objects.
    """

    if isinstance(x, Sign):
        return x

    try:
        return Sign.scalar(rational_coeff(x))
    except TypeError:
        pass

    raise TypeError(f"Cannot convert {x} to Sign.")


def simplify_coeff(c):
    """
    Simplify a coefficient.

    Sign coefficients keep their formal sign structure; ordinary scalar
    coefficients are exact rationals.
    """

    if isinstance(c, Sign):
        return c

    return rational_coeff(c)


def is_zero_coeff(c):
    """
    Test whether a coefficient is zero.
    """

    if isinstance(c, Sign):
        return c.is_zero()

    return rational_coeff(c) == 0


In [11]:
def is_scalar_coeff(c):
    """
    Scalars allowed as coefficients of Elements / VirtualElements.
    """

    if isinstance(c, Sign):
        return True

    try:
        rational_coeff(c)
        return True
    except TypeError:
        return False


In [12]:
class CellFamily:
    def __init__(self, quiver):
        self.quiver = quiver
        self.cells = {}

    def key(self, f):
        if hasattr(f, "key"):
            return f.key()

        return repr(f)

    def __getitem__(self, f):
        k = self.key(f)

        if k not in self.cells and repr(f) in self.cells:
            return self.cells[repr(f)]

        if k not in self.cells:
            print("No cell has been attached to this differential yet.")
            print("Differential:", f)
            return None

        return self.cells[k]

    def __setitem__(self, f, cell):
        if not isinstance(cell, Arrow):
            raise TypeError("Only Arrow objects can be stored as cells.")

        k = self.key(f)
        self.cells[k] = cell

    def __contains__(self, f):
        return self.key(f) in self.cells or repr(f) in self.cells

    def __repr__(self):
        return "CellFamily u"

In [13]:
class Element:
    def __init__(self, terms=None):
        if terms is None:
            terms = {}

        self.terms = {}

        for path, coeff in terms.items():
            coeff = simplify_coeff(coeff)

            if not is_zero_coeff(coeff):
                old_coeff = self.terms.get(path, 0)
                new_coeff = old_coeff + coeff
                new_coeff = simplify_coeff(new_coeff)

                if not is_zero_coeff(new_coeff):
                    self.terms[path] = new_coeff
                elif path in self.terms:
                    del self.terms[path]

    def __repr__(self):
        if not self.terms:
            return "0"

        pieces = []

        for path, coeff in self.terms.items():
            if coeff == 1:
                pieces.append(str(path))
            elif coeff == -1:
                pieces.append("-" + str(path))
            else:
                pieces.append(f"{coeff}*{path}")

        return " + ".join(pieces).replace("+ -", "- ")

    def key(self):
        return (
            "Element",
            frozenset(
                (
                    path.key() if hasattr(path, "key") else path,
                    coeff
                )
                for path, coeff in self.terms.items()
            )
        )

    def h(self):
        if not self.terms:
            return None

        heights = set()

        for path in self.terms:
            heights.add(path.h())

        if len(heights) == 1:
            return next(iter(heights))

        raise ValueError(f"Element is not homogeneous in height. Heights found: {heights}")

    def __add__(self, other):
        other = to_element(other)
        new_terms = dict(self.terms)

        for path, coeff in other.terms.items():
            new_terms[path] = new_terms.get(path, 0) + coeff

        return Element(new_terms)

    def __radd__(self, other):
        if other == 0:
            return self
        return to_element(other) + self

    def __neg__(self):
        return Element({
            path: -coeff
            for path, coeff in self.terms.items()
        })

    def __sub__(self, other):
        return self + (-to_element(other))

    def __rsub__(self, other):
        return to_element(other) - self

    def __rmul__(self, scalar):
        scalar = simplify_coeff(scalar)

        return Element({
            path: scalar * coeff
            for path, coeff in self.terms.items()
        })

    def __mul__(self, other):
        """
        Multiplication of elements, delegated to Q.l2.
        """
        other = to_element(other)

        if not self.terms:
            return Element({})

        some_path = next(iter(self.terms.keys()))
        Q = some_path.Q

        return Q.l2(self, other)

    def __eq__(self, other):
        other = to_element(other)
        return self.terms == other.terms
        
    def is_zero(self):
        return len(self.terms) == 0

    def g(self):
        if not self.terms:
            return None

        gradings = set()

        for path in self.terms:
            gradings.add(path.g())

        if len(gradings) == 1:
            return next(iter(gradings))

        raise ValueError(f"Element is not homogeneous in grading. Gradings found: {gradings}")

    def g(self):
        if not self.terms:
            return None

        gradings = set()

        for path in self.terms:
            gradings.add(path.g())

        if len(gradings) == 1:
            return next(iter(gradings))

        raise ValueError(f"Element is not homogeneous in grading. Gradings found: {gradings}")

    def sg(self):
        if not self.terms:
            return 1

        signs = set()

        for path in self.terms:
            signs.add(path.sg())

        if len(signs) == 1:
            return next(iter(signs))

        raise ValueError(f"Element is not homogeneous in sign. Signs found: {signs}")


In [14]:
class Path(Element):
    def __init__(self, Q, arrows, source=None, target=None):
        """
        A Path is also an Element.

        Q is the quiver.

        arrows is a list of arrow names.

        Examples:
            Path(Q, ["a"], "0", "1")
            Path(Q, ["a", "b"], "0", "2")
            Path(Q, [], "0", "0")     # idempotent e_0
        """
        self.Q = Q
        self.arrows = tuple(arrows)
        self.source = source
        self.target = target

        # A path is the element consisting of itself with coefficient 1
        super().__init__({self: 1})

    def key(self):
        return (
            "Path",
            self.source,
            tuple(
                arrow.name if hasattr(arrow, "name") else arrow
                for arrow in self.arrows
            ),
            self.target
        )

    def sign_name(self):
        return str(self)

    def h(self):
        total = 0

        for arrow_name in self.arrows:
            arrow = self.Q.arrows[arrow_name]
            total += arrow.h()

        return total

    def __repr__(self):
        if len(self.arrows) == 0:
            return f"e_{self.source}"

        pieces = []

        for arrow_name in self.arrows:
            arrow = self.Q.arrows[arrow_name]
            pieces.append(repr(arrow))

        return "".join(pieces)

    def __hash__(self):
        return hash((id(self.Q), self.arrows, self.source, self.target))

    def __eq__(self, other):
        return (
            isinstance(other, Path)
            and self.Q is other.Q
            and self.arrows == other.arrows
            and self.source == other.source
            and self.target == other.target
        )
    def koszul_sign(self):
        """
        Return the Koszul sign (-1)^g(self).

        If the path has known integer grading, return +1 or -1.
        If the grading is unknown, return a formal SymPy symbol.
        """

        degree = self.g()

        # Known integer grading
        if isinstance(degree, int):
            return 1 if degree % 2 == 0 else -1

        # Unknown grading: return formal sign
        return Symbol(f"(-1)^g({self})")
        
    def g(self):
        if len(self.arrows) == 0:
            return 0

        total = 0

        for arrow_name in self.arrows:
            arrow = self.Q.arrows[arrow_name]
            total = total + arrow.g()

        return total

In [15]:
class Arrow(Path):
    def __init__(self, Q, name, source, target, grading=None, height=0):
        """
        An arrow is a path of length one.

        Since your Path class expects:
            Path(Q, arrows, source, target)

        we call:
            super().__init__(Q, [name], source, target)
        """
        super().__init__(Q, [name], source, target)

        self.name = name
        self.grading = grading
        self.source = source
        self.target = target
        self.height = height

    def sign_name(self):
        return str(self)

    def h(self):
        return self.height

    def g(self):
        if self.grading is None:
            return Symbol(f"g({self})")
        return self.grading

    def assigngrading(self, grading):
        """
        Assign or overwrite the grading of this arrow.
        """
        self.grading = grading
        print(f"Assigned grading g({self.name}) = {grading}")
        
    def __repr__(self):
        if hasattr(self, "display_name"):
            return self.display_name
        return self.name

In [16]:
class AinfMonomial:
    def __init__(self, quiver, inputs):
        self.quiver = quiver
        self.inputs = tuple(inputs)

    def __repr__(self):
        inside = ", ".join(str(x) for x in self.inputs)
        return f"M({inside})"

    def __str__(self):
        return self.__repr__()

    def key(self):
        return (
            "AinfMonomial",
            tuple(self.input_key(x) for x in self.inputs)
        )

    def input_key(self, x):
        if hasattr(x, "key"):
            return x.key()

        return str(x)

    def __eq__(self, other):
        return (
            isinstance(other, AinfMonomial)
            and self.key() == other.key()
        )

    def __hash__(self):
        return hash(self.key())

    def g(self):
        total = 0

        for x in self.inputs:
            total += self.quiver.g(x)

        return total + 2 - len(self.inputs)

    def sg(self):
        result = Sign.scalar((-1) ** len(self.inputs))

        for x in self.inputs:
            result = result * self.quiver.sg(x)

        return result

    def expand_input(self, x):
        if isinstance(x, AinfMonomial):
            return x.expand()

        return x

    def lambda_block(self, block):
        block = tuple(block)

        if len(block) == 0:
            raise ValueError("Empty block is not allowed.")

        if len(block) == 1:
            return self.expand_input(block[0])

        if len(block) == 2:
            x, y = block
            x = self.expand_input(x)
            y = self.expand_input(y)

            return x * y

        return AinfMonomial(self.quiver, block)

    def H_block(self, block):
        block = tuple(block)

        if len(block) == 0:
            raise ValueError("Empty block is not allowed.")

        if len(block) == 1:
            return self.quiver.to_virtual_element(
                self.expand_input(block[0])
            )

        return self.quiver.homotopy_virtual(
            self.lambda_block(block)
        )

    def split_sign(self, left_length, total_length):
        return (-1) ** (total_length - 1 - left_length)

    def expand(self):
        inputs = tuple(
            self.expand_input(x)
            for x in self.inputs
        )

        n = len(inputs)

        if n == 0:
            raise ValueError("M needs at least one input.")

        if n == 1:
            return self.quiver.to_virtual_element(inputs[0])

        if n == 2:
            x, y = inputs
            return self.quiver.to_virtual_element(x * y)

        result = self.quiver.to_virtual_element(0)

        for left_length in range(n - 1, 0, -1):
            left_block = inputs[:left_length]
            right_block = inputs[left_length:]

            sign = self.split_sign(left_length, n)

            left_factor = self.H_block(left_block)
            right_factor = self.H_block(right_block)

            result = result + sign * left_factor * right_factor

        return result

    def compute(self):
        return self.expand()

In [17]:
class MasseyProduct:
    """
    Formal record of a Massey product.

    This object records that a Massey product with these inputs exists.
    The actual representing element is computed by .expand().
    """

    def __init__(self, quiver, inputs):
        self.Q = quiver
        self.inputs = tuple(inputs)

    def __repr__(self):
        inside = ",".join(str(x) for x in self.inputs)
        return f"Q.mp({inside})"

    def key(self):
        return (
            "MasseyProduct",
            tuple(
                x.key() if hasattr(x, "key") else str(x)
                for x in self.inputs
            )
        )


    def __eq__(self, other):
        return (
            isinstance(other, MasseyProduct)
            and self.Q is other.Q
            and self.key() == other.key()
        )


    def __hash__(self):
        return hash((id(self.Q), self.key()))

    @property
    def length(self):
        return len(self.inputs)

    @property
    def source(self):
        return self.inputs[0].source

    @property
    def target(self):
        return self.inputs[-1].target

    def is_circle(self):
        return self.source == self.target

    def expand(self):
        return self.Q.expand_mp(*self.inputs)


    def as_mp_input(self):
        """
        If this is a length-one Massey product Q.mp(x), use x as an input.

        This makes:
            Q.mp(x) * Q.mp(y)
        become:
            Q.mp(x, y)

        rather than:
            Q.mp(Q.mp(x), Q.mp(y)).
        """
        if len(self.inputs) == 1:
            return self.inputs[0]

        return self

    def as_product(self):
        """
        Regard this MasseyProduct as a one-factor MPProduct.
        """
        return MPProduct(self.Q, (self,))

    def as_mp_element(self):
        """
        Regard this MasseyProduct as a one-term MPElement.
        """

        return self.as_product().as_mp_element()


    def __mul__(self, other):
        """
        Product of Massey products.

        A single MasseyProduct is automatically regarded as a one-factor
        MPProduct.
        """
        return self.as_product() * other


    def __rmul__(self, other):
        """
        Product on the left.
        """
        if is_scalar_coeff(other):
            return MPElement(self.Q, {self.as_product(): other})

        return other * self.as_product()

    def __add__(self, other):
        return self.as_mp_element() + other

    def __radd__(self, other):
        if other == 0:
            return self.as_mp_element()

        return self.Q.to_mp_element(other) + self.as_mp_element()

    def __neg__(self):
        return -self.as_mp_element()

    def __sub__(self, other):
        return self.as_mp_element() - other

    def __rsub__(self, other):
        return self.Q.to_mp_element(other) - self.as_mp_element()

In [18]:
class MPProduct:
    """
    A product of MasseyProduct objects.

    This is used for bookkeeping products like

        Q.mp(x1) * Q.mp(x2, x3) * Q.mp(x4)

    without immediately expanding them into actual Elements.
    """

    def __init__(self, quiver, factors):
        self.Q = quiver
        self.factors = tuple(factors)

    def key(self):
        return self.Q.mp_product_key(self)

    def __eq__(self, other):
        return (
            isinstance(other, MPProduct)
            and self.Q is other.Q
            and self.key() == other.key()
        )

    def __hash__(self):
        return hash((id(self.Q), self.key()))

    def __repr__(self):
        if len(self.factors) == 0:
            return "1"

        return "*".join(str(f) for f in self.factors)

    def as_mp_element(self):
        return MPElement(self.Q, {self: 1})

    def __mul__(self, other):
        if is_scalar_coeff(other):
            return MPElement(self.Q, {self: other})

        if isinstance(other, MPElement):
            return self.as_mp_element() * other

        if isinstance(other, MPProduct):
            return MPProduct(self.Q, self.factors + other.factors)

        if isinstance(other, MasseyProduct):
            return MPProduct(self.Q, self.factors + (other,))

        # If other is an arrow/path, convert it to length-one MP.
        return MPProduct(self.Q, self.factors + (self.Q.mp(other),))

    def __rmul__(self, other):
        if is_scalar_coeff(other):
            return MPElement(self.Q, {self: other})

        if isinstance(other, MPElement):
            return other * self

        if isinstance(other, MPProduct):
            return MPProduct(self.Q, other.factors + self.factors)

        if isinstance(other, MasseyProduct):
            return MPProduct(self.Q, (other,) + self.factors)

        return MPProduct(self.Q, (self.Q.mp(other),) + self.factors)

    def __add__(self, other):
        return self.as_mp_element() + other

    def __radd__(self, other):
        if other == 0:
            return self.as_mp_element()

        return self.Q.to_mp_element(other) + self.as_mp_element()

    def __neg__(self):
        return -self.as_mp_element()

    def __sub__(self, other):
        return self.as_mp_element() - other

    def __rsub__(self, other):
        return self.Q.to_mp_element(other) - self.as_mp_element()

    @property
    def source(self):
        return self.factors[0].source

    @property
    def target(self):
        return self.factors[-1].target


class MPElement:
    """
    A finite linear combination of MPProduct objects.

    This is used for expressions such as

        2*(Q.mp(x)*Q.mp(y)) - 3*(Q.mp(z)*Q.mp(w)).
    """

    def __init__(self, quiver, terms=None):
        self.Q = quiver

        if terms is None:
            terms = {}

        self.terms = {}

        for product, coeff in terms.items():
            if isinstance(product, MasseyProduct):
                product = product.as_product()

            if not isinstance(product, MPProduct):
                raise TypeError("MPElement terms must be MPProduct keys.")

            coeff = simplify_coeff(coeff)

            if is_zero_coeff(coeff):
                continue

            old_coeff = self.terms.get(product, 0)
            new_coeff = simplify_coeff(old_coeff + coeff)

            if is_zero_coeff(new_coeff):
                if product in self.terms:
                    del self.terms[product]
            else:
                self.terms[product] = new_coeff

    def key(self):
        return (
            "MPElement",
            tuple(
                sorted(
                    (product.key(), coeff_key(coeff))
                    for product, coeff in self.terms.items()
                )
            )
        )

    def __eq__(self, other):
        try:
            other = self.Q.to_mp_element(other)
        except Exception:
            return False

        return self.key() == other.key()

    def __hash__(self):
        return hash((id(self.Q), self.key()))

    def __repr__(self):
        if not self.terms:
            return "0"

        pieces = []

        for product, coeff in self.terms.items():
            if coeff == 1:
                pieces.append(str(product))
            elif coeff == -1:
                pieces.append("-" + str(product))
            else:
                pieces.append(f"{coeff}*{product}")

        return " + ".join(pieces).replace("+ -", "- ")

    def __str__(self):
        return self.__repr__()

    def __add__(self, other):
        if other == 0:
            return self

        other = self.Q.to_mp_element(other)
        terms = dict(self.terms)

        for product, coeff in other.terms.items():
            terms[product] = terms.get(product, 0) + coeff

        return MPElement(self.Q, terms)

    def __radd__(self, other):
        return self + other

    def __neg__(self):
        return MPElement(
            self.Q,
            {
                product: -coeff
                for product, coeff in self.terms.items()
            }
        )

    def __sub__(self, other):
        return self + (-self.Q.to_mp_element(other))

    def __rsub__(self, other):
        return self.Q.to_mp_element(other) - self

    def __mul__(self, other):
        if is_scalar_coeff(other):
            other = simplify_coeff(other)

            return MPElement(
                self.Q,
                {
                    product: coeff * other
                    for product, coeff in self.terms.items()
                }
            )

        other = self.Q.to_mp_element(other)
        terms = {}

        for left_product, left_coeff in self.terms.items():
            for right_product, right_coeff in other.terms.items():
                product = MPProduct(
                    self.Q,
                    left_product.factors + right_product.factors
                )
                terms[product] = terms.get(product, 0) + left_coeff * right_coeff

        return MPElement(self.Q, terms)

    def __rmul__(self, other):
        if is_scalar_coeff(other):
            other = simplify_coeff(other)

            return MPElement(
                self.Q,
                {
                    product: other * coeff
                    for product, coeff in self.terms.items()
                }
            )

        other = self.Q.to_mp_element(other)
        return other * self


In [19]:
class VirtualElement:
    def __init__(self, quiver, terms=None):
        self.quiver = quiver

        if terms is None:
            terms = {}

        self.terms = {}

        for monomial, coeff in terms.items():
            if isinstance(monomial, VirtualAtom):
                monomial = monomial.as_monomial()

            if not isinstance(monomial, VirtualMonomial):
                raise TypeError(
                    "VirtualElement terms must have VirtualMonomial keys."
                )

            coeff = simplify_coeff(coeff)

            if not is_zero_coeff(coeff):
                old_coeff = self.terms.get(monomial, 0)
                new_coeff = simplify_coeff(old_coeff + coeff)

                if not is_zero_coeff(new_coeff):
                    self.terms[monomial] = new_coeff
                elif monomial in self.terms:
                    del self.terms[monomial]

    def __repr__(self):
        if not self.terms:
            return "0"

        pieces = []

        for monomial, coeff in self.terms.items():
            if coeff == 1:
                pieces.append(str(monomial))
            elif coeff == -1:
                pieces.append(f"-{monomial}")
            else:
                pieces.append(f"{coeff}*{monomial}")

        return " + ".join(pieces).replace("+ -", "- ")

    def __str__(self):
        return self.__repr__()

    def key(self):
        return (
            "VirtualElement",
            tuple(
                sorted(
                    (monomial.key(), coeff)
                    for monomial, coeff in self.terms.items()
                )
            )
        )

    def __eq__(self, other):
        return (
            isinstance(other, VirtualElement)
            and self.key() == other.key()
        )

    def __add__(self, other):
        other = self.quiver.to_virtual_element(other)

        terms = dict(self.terms)

        for monomial, coeff in other.terms.items():
            terms[monomial] = terms.get(monomial, 0) + coeff

        return VirtualElement(self.quiver, terms)

    def __radd__(self, other):
        return self + other

    def __neg__(self):
        return VirtualElement(
            self.quiver,
            {
                monomial: -coeff
                for monomial, coeff in self.terms.items()
            }
        )

    def __sub__(self, other):
        return self + (-self.quiver.to_virtual_element(other))

    def __rsub__(self, other):
        return self.quiver.to_virtual_element(other) + (-self)

    def __rmul__(self, other):
        if is_scalar_coeff(other):
            other = simplify_coeff(other)

            return VirtualElement(
                self.quiver,
                {
                    monomial: simplify_coeff(other * coeff)
                    for monomial, coeff in self.terms.items()
                }
            )

        other = self.quiver.to_virtual_element(other)
        return other * self

    def __mul__(self, other):
        if is_scalar_coeff(other):
            other = simplify_coeff(other)

            return VirtualElement(
                self.quiver,
                {
                    monomial: simplify_coeff(coeff * other)
                    for monomial, coeff in self.terms.items()
                }
            )

        other = self.quiver.to_virtual_element(other)

        result_terms = {}

        for m1, c1 in self.terms.items():
            for m2, c2 in other.terms.items():
                product = VirtualMonomial(
                    self.quiver,
                    tuple(m1.factors) + tuple(m2.factors)
                )

                result_terms[product] = result_terms.get(product, 0) + c1 * c2

        return VirtualElement(self.quiver, result_terms)

    def g(self):
        if not self.terms:
            return None

        gradings = set()

        for monomial in self.terms:
            gradings.add(monomial.g())

        if len(gradings) == 1:
            return next(iter(gradings))

        raise ValueError(f"VirtualElement is not homogeneous in grading. Gradings found: {gradings}")

    def sg(self):
        if not self.terms:
            return 1

        signs = set()

        for monomial in self.terms:
            signs.add(simplify_coeff(monomial.sg()))

        if len(signs) == 1:
            return next(iter(signs))

        raise ValueError(f"VirtualElement is not homogeneous in sign. Signs found: {signs}")


In [20]:
class VirtualMonomial:
    def __init__(self, quiver, factors):
        self.quiver = quiver
        self.factors = tuple(factors)

    def __repr__(self):
        if len(self.factors) == 0:
            return "1"

        return "*".join(str(factor) for factor in self.factors)

    def __str__(self):
        return self.__repr__()

    def factor_key(self, factor):
        if hasattr(factor, "key"):
            return factor.key()

        if hasattr(factor, "name"):
            return factor.name

        return str(factor)

    def key(self):
        return (
            "VirtualMonomial",
            tuple(self.factor_key(factor) for factor in self.factors)
        )

    def __eq__(self, other):
        return (
            isinstance(other, VirtualMonomial)
            and self.key() == other.key()
        )

    def __hash__(self):
        return hash(self.key())

    def __mul__(self, other):
        if isinstance(other, VirtualMonomial):
            return VirtualMonomial(
                self.quiver,
                self.factors + other.factors
            )

        if isinstance(other, (Arrow, Path)):
            return VirtualMonomial(
                self.quiver,
                self.factors + (other,)
            )

        return NotImplemented

    def __rmul__(self, other):
        if isinstance(other, (Arrow, Path)):
            return VirtualMonomial(
                self.quiver,
                (other,) + self.factors
            )

        return NotImplemented

    def g(self):
        total = 0

        for factor in self.factors:
            total += self.quiver.g(factor)

        return total

    def sg(self):
        sign = 1

        for factor in self.factors:
            sign *= self.quiver.sg(factor)

        return simplify_coeff(sign)

In [21]:
class VirtualAtom:
    def __init__(self, quiver, kind, data):
        self.quiver = quiver
        self.kind = kind
        self.data = data

    def __repr__(self):
        if self.kind == "h":
            return f"h[{self.data}]"

        return f"{self.kind}[{self.data}]"

    def __str__(self):
        return self.__repr__()

    def data_key(self, data):
        if isinstance(data, tuple):
            return tuple(self.data_key(x) for x in data)

        if hasattr(data, "key"):
            return data.key()

        if hasattr(data, "name"):
            return data.name

        return str(data)

    def key(self):
        return (
            "VirtualAtom",
            self.kind,
            self.data_key(self.data)
        )

    def __eq__(self, other):
        return (
            isinstance(other, VirtualAtom)
            and self.key() == other.key()
        )

    def __hash__(self):
        return hash(self.key())

    def as_monomial(self):
        return VirtualMonomial(self.quiver, (self,))

    def __mul__(self, other):
        return self.as_monomial() * other

    def __rmul__(self, other):
        return other * self.as_monomial()

    def g(self):
        if self.kind == "h":
            return self.quiver.g(self.data) - 1

        raise NotImplementedError(f"Grading not implemented for virtual atom kind {self.kind}.")

    def sg(self):
        if self.kind == "h":
            return -self.quiver.sg(self.data)

        raise NotImplementedError(f"Sign not implemented for virtual atom kind {self.kind}.")

In [22]:
def to_element(x):
    """
    Converts x to an Element.

    If x is already an Element, do nothing.
    """
    if isinstance(x, Element):
        return x

    if x == 0:
        return Element({})

    else:
        return Element({x: 1})

In [ ]:
from cyclic_extensions import install_cyclic_extensions

install_cyclic_extensions(Quiver, globals())

In [23]:
class HomotopyAccessor:
    def __init__(self, quiver):
        self.quiver = quiver

    def __getitem__(self, f):
        return self.quiver.homotopy_virtual(f)

In [24]:
def buildquiver():
    global h
    global L

    Q = Quiver()

    print("First, enter the vertices.")
    Q.add_vertices()

    print("Now, enter the arrows.")
    Q.add_arrows()

    h = HomotopyAccessor(Q)
    L = Q.L

    print("Quiver construction finished.")
    print("Created global homotopy accessor h. You can now use h[f].")
    print("Created global higher multiplication L. You can now use L(x1, ..., xn).")

    return Q

In [25]:
Q=buildquiver()

First, enter the vertices.


Enter a vertex name, or type ok to finish:  1


Created global variable e_1.


Enter a vertex name, or type ok to finish:  ok


Now, enter the arrows.


...

Quiver has only one vertex 1, source and target are automatically assigned.
Created global variable y.
Added arrow y: 1 -> 1, grading g(y).


Arrow name, or type ok to finish:  z


Quiver has only one vertex 1, source and target are automatically assigned.
Created global variable z.
Added arrow z: 1 -> 1, grading g(z).


Arrow name, or type ok to finish:  ok


Quiver construction finished.
Created global homotopy accessor h. You can now use h[f].
Created global higher multiplication L. You can now use L(x1, ..., xn).


In [26]:
Q.resolve_mp()

Start typing Massey product to be resolved, or type ok to finish:  x*y


Recorded primitive of Q.mp(x,y):
    h[Q.mp(x,y)] = v[Q.mp(x,y)]
Attached new cell v[Q.mp(x,y)]
Internal arrow name: cell_4
Source: 1
Target: 1
d(v[Q.mp(x,y)]) = xy
Resolved Q.mp(x,y) by v[Q.mp(x,y)].


Start typing Massey product to be resolved, or type ok to finish:  y*z


Recorded primitive of Q.mp(y,z):
    h[Q.mp(y,z)] = v[Q.mp(y,z)]
Attached new cell v[Q.mp(y,z)]
Internal arrow name: cell_5
Source: 1
Target: 1
d(v[Q.mp(y,z)]) = yz
Resolved Q.mp(y,z) by v[Q.mp(y,z)].


Start typing Massey product to be resolved, or type ok to finish:  z*x


Recorded primitive of Q.mp(z,x):
    h[Q.mp(z,x)] = v[Q.mp(z,x)]
Attached new cell v[Q.mp(z,x)]
Internal arrow name: cell_6
Source: 1
Target: 1
d(v[Q.mp(z,x)]) = zx
Resolved Q.mp(z,x) by v[Q.mp(z,x)].


Start typing Massey product to be resolved, or type ok to finish:  ok


In [27]:
Q.generate_massey_products()

Have primitives: []
No obvious primitives: [Q.mp(z,x,y), Q.mp(x,y,z), Q.mp(y,z,x)]


{'have_primitives': [],
 'no_obvious_primitives': [Q.mp(z,x,y), Q.mp(x,y,z), Q.mp(y,z,x)]}

In [28]:
Q.undo_latest_mp_cell()

Undid latest resolving cell:
    v[Q.mp(z,x)]
for:
    Q.mp(z,x)
Removed stale generated MP records:
    Q.mp(z,x,y)
    Q.mp(y,z,x)


{'undone': ((z, x), v[Q.mp(z,x)]),
 'removed_stale_mps': [Q.mp(z,x,y), Q.mp(y,z,x)],
 'remaining_history': [(x, y), (y, z)]}

In [29]:
Q.generate_massey_products()

Have primitives: []
No obvious primitives: [Q.mp(x,y,z)]


{'have_primitives': [], 'no_obvious_primitives': [Q.mp(x,y,z)]}

In [30]:
Q.delete_mp_cell()

AttributeError: 'Quiver' object has no attribute 'delete_mp_cell'

In [31]:
Q.generate_massey_products()

Have primitives: []
No obvious primitives: [Q.mp(x,y,z)]


{'have_primitives': [], 'no_obvious_primitives': [Q.mp(x,y,z)]}